In [2]:
# -*- coding: utf-8 -*-
"""
CAT: KmLow / VmaxHigh 分类模型（严格 split 版）
- 标签：log10 后中位数切分
- 分割：DOI 分组；缺 DOI -> 每行伪 DOI
- 12 模型比选；AUC_mean 主
- 严格 split：
  * MIN_TEST_SAMPLES = 6
  * MIN_TRAIN_CLASS_COUNT = 2
  * MIN_TEST_CLASS_COUNT  = 2
- 仍然尝试收集 40~50 个有效 split（若数据允许）
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.metrics import roc_auc_score, cohen_kappa_score, matthews_corrcoef, accuracy_score

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


# =========================
# 0) 路径与配置
# =========================
CAT_PATH = r"C:\Users\86158\Desktop\dataiscat.xlsx"
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\cat_cls_km_vmax_grouped_search_strictsplit")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_DOI = "data reference doi"
COL_KM = "Km/mM"
COL_VMAX = "Vmax/μM s-1"

# GroupShuffleSplit 配置（保持）
TEST_SIZE = 0.2
SPLIT_SEED = 42

# ===== 严格 split（只改这里）=====
MIN_TEST_SAMPLES = 6
MIN_TRAIN_CLASS_COUNT = 2
MIN_TEST_CLASS_COUNT = 2

# 目标：收集 40~50 个有效 split（如果能做到）
TARGET_VALID_SPLITS = 40
MAX_VALID_SPLITS = 50
MAX_TRIES = 10000


# =========================
# 1) 特征定义（不改）
# =========================
FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate "
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = [
    "shape", "surface treatment", "dispersion medium", "Substrate "
]


# =========================
# 2) 预处理（不改，保持 IC50 风格）
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    X = df_raw[FEATURE_COLS].copy()

    for c in NUMERIC_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in CATEGORICAL_COLS:
        X[c] = clean_text_series(X[c])

    return X


def make_preprocess():
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True))])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[("num", num_pipe, NUMERIC_COLS), ("cat", cat_pipe, CATEGORICAL_COLS)],
        remainder="drop"
    )


# =========================
# 3) 模型（不改）
# =========================
def get_models():
    models = {
        "LogReg": LogisticRegression(max_iter=5000, class_weight="balanced",
                                     solver="liblinear", random_state=42),
        "LinearSVC": LinearSVC(C=1.0, class_weight="balanced", random_state=42),
        "SVC_RBF": SVC(C=1.0, gamma="scale", class_weight="balanced",
                       probability=True, random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5, weights="distance"),
        "GaussianNB": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(max_depth=4, min_samples_leaf=2,
                                              class_weight="balanced", random_state=42),
        "RF": RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                     class_weight="balanced", random_state=42, n_jobs=-1),
        "ExtraTrees": ExtraTreesClassifier(n_estimators=800, min_samples_leaf=2,
                                           class_weight="balanced", random_state=42, n_jobs=-1),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                       max_depth=3, random_state=42),
        "HistGB": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                                 max_depth=4, random_state=42),
        "AdaBoost": AdaBoostClassifier(n_estimators=300, learning_rate=0.05, random_state=42),
    }
    if HAS_XGBOOST:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", random_state=42, n_jobs=-1, verbosity=0
        )
    return models


def needs_scaling(model_name):
    return model_name in {"LogReg", "LinearSVC", "SVC_RBF", "KNN", "GaussianNB"}


def build_model_pipe(model_name, model, preprocess):
    steps = [("preprocess", preprocess)]
    if needs_scaling(model_name):
        steps.append(("scaler", StandardScaler(with_mean=True)))
    steps.append(("model", model))
    return Pipeline(steps)


def score_to_prob(pipe, X):
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X)[:, 1]
    if hasattr(pipe, "decision_function"):
        z = np.asarray(pipe.decision_function(X), dtype=float)
        return 1.0 / (1.0 + np.exp(-z))
    return np.asarray(pipe.predict(X), dtype=float)


# =========================
# 4) 标签
# =========================
def make_label_median_log10(series: pd.Series, mode: str):
    v = pd.to_numeric(series, errors="coerce")
    mask = v.notna() & (v > 0)
    vals = v.loc[mask].astype(float).values
    logv = np.log10(vals)
    thr = float(np.median(logv))
    if mode == "low_good":
        y = (logv <= thr).astype(int)
    else:
        y = (logv >= thr).astype(int)
    return mask, y, thr


# =========================
# 5) 收集有效 split（严格判定）
# =========================
def collect_valid_group_splits(X, y, groups):
    valid_splits = []
    seen = set()
    tries = 0

    while tries < MAX_TRIES and len(valid_splits) < MAX_VALID_SPLITS:
        tries += 1
        splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SPLIT_SEED + tries)

        for tr, te in splitter.split(X, y, groups=groups):
            key = tuple(sorted(te.tolist()))
            if key in seen:
                continue
            seen.add(key)

            if len(te) < MIN_TEST_SAMPLES:
                continue

            ytr, yte = y[tr], y[te]

            if (np.sum(ytr == 0) < MIN_TRAIN_CLASS_COUNT) or (np.sum(ytr == 1) < MIN_TRAIN_CLASS_COUNT):
                continue
            if (np.sum(yte == 0) < MIN_TEST_CLASS_COUNT) or (np.sum(yte == 1) < MIN_TEST_CLASS_COUNT):
                continue

            valid_splits.append((tr, te))
            if len(valid_splits) >= MAX_VALID_SPLITS:
                break

    return valid_splits, tries


# =========================
# 6) 单任务训练
# =========================
def run_one_task(task_name, df_use, X, y, groups, preprocess, prefix):
    df_labeled = df_use.copy()
    df_labeled["y"] = y
    df_labeled.to_csv(OUT_DIR / f"{prefix}_labeled_data.csv", index=False, encoding="utf-8-sig")

    valid_splits, tries = collect_valid_group_splits(X, y, groups)

    print("\n" + "=" * 70)
    print(f"[Task] {task_name}")
    print("=" * 70)
    print(f"样本数: {len(y)} | DOI组数: {len(np.unique(groups))}")
    print(f"尝试分割次数: {tries} | 有效 split 数: {len(valid_splits)}")
    if len(valid_splits) < TARGET_VALID_SPLITS:
        print(f"[Warn] 未达到目标 {TARGET_VALID_SPLITS}，仅得到 {len(valid_splits)}")

    if len(valid_splits) == 0:
        raise RuntimeError(f"{task_name}: 一个有效 split 都没收集到。")

    models = get_models()
    rows = []

    for model_name, model in models.items():
        print(f"[Running] {task_name} | {model_name} ...")
        for split_id, (tr, te) in enumerate(valid_splits, start=1):
            Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()
            ytr, yte = y[tr], y[te]

            try:
                pipe = build_model_pipe(model_name, model, preprocess)
                pipe.fit(Xtr, ytr)

                pte = score_to_prob(pipe, Xte)
                pred = (pte >= 0.5).astype(int)

                rows.append({
                    "task": task_name, "model": model_name, "split": split_id,
                    "auc": float(roc_auc_score(yte, pte)),
                    "kappa": float(cohen_kappa_score(yte, pred)),
                    "mcc": float(matthews_corrcoef(yte, pred)),
                    "acc": float(accuracy_score(yte, pred)),
                    "n_train": int(len(tr)), "n_test": int(len(te)),
                    "pos_test": int(np.sum(yte == 1)), "neg_test": int(np.sum(yte == 0)),
                    "error": ""
                })
            except Exception as e:
                rows.append({
                    "task": task_name, "model": model_name, "split": split_id,
                    "auc": np.nan, "kappa": np.nan, "mcc": np.nan, "acc": np.nan,
                    "n_train": int(len(tr)), "n_test": int(len(te)),
                    "pos_test": int(np.sum(yte == 1)), "neg_test": int(np.sum(yte == 0)),
                    "error": repr(e)
                })

    res = pd.DataFrame(rows)
    res.to_csv(OUT_DIR / f"{prefix}_all_splits.csv", index=False, encoding="utf-8-sig")

    res_ok = res.dropna(subset=["auc", "kappa", "mcc", "acc"]).copy()
    if res_ok.empty:
        raise RuntimeError(f"{task_name}: 所有模型评估失败，请检查 {prefix}_all_splits.csv 的 error 列。")

    summary = (
        res_ok.groupby("model", as_index=False)
        .agg(
            auc_mean=("auc", "mean"),
            auc_median=("auc", "median"),
            auc_std=("auc", "std"),
            auc_max=("auc", "max"),
            kappa_mean=("kappa", "mean"),
            mcc_mean=("mcc", "mean"),
            acc_mean=("acc", "mean"),
            n_splits=("split", "count")
        )
        .sort_values(
            by=["auc_mean", "auc_median", "mcc_mean", "kappa_mean", "acc_mean", "auc_max"],
            ascending=[False, False, False, False, False, False]
        )
        .reset_index(drop=True)
    )
    summary.to_csv(OUT_DIR / f"{prefix}_summary.csv", index=False, encoding="utf-8-sig")

    best_model_name = str(summary.iloc[0]["model"])
    print("\n" + "=" * 60)
    print(f"[BEST] {task_name} = {best_model_name}")
    print(summary.head(10).to_string(index=False))
    print("=" * 60)

    best_model = get_models()[best_model_name]
    best_pipe = build_model_pipe(best_model_name, best_model, preprocess)
    best_pipe.fit(X, y)
    joblib.dump(best_pipe, OUT_DIR / f"{prefix}_best_model_refit_full.joblib")


# =========================
# 7) 主程序
# =========================
def main():
    df = pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME)

    missing = [c for c in FEATURE_COLS + [COL_DOI, COL_KM, COL_VMAX] if c not in df.columns]
    if missing:
        raise ValueError(f"CAT 表缺少列：{missing}")

    preprocess = make_preprocess()

    groups_all = df[COL_DOI].fillna("").astype(str).str.strip()
    miss = groups_all.eq("")
    groups_all.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df.index[miss]]
    groups_all = groups_all.values

    X_all = prepare_features(df)

    # KmLow
    m_km, y_km, thr_km = make_label_median_log10(df[COL_KM], mode="low_good")
    X_km = X_all.loc[m_km].copy()
    df_km = df.loc[m_km].copy()
    g_km = groups_all[m_km.values]

    print("\n" + "=" * 60)
    print("CAT KmLow (strict split)")
    print("=" * 60)
    print(f"n={len(y_km)} | median(log10(Km))={thr_km:.6f} | pos_rate={y_km.mean():.3f} | groups={len(np.unique(g_km))}")
    run_one_task("CAT_KmLow", df_km, X_km, y_km.astype(int), g_km, preprocess, "cat_cls_KmLow")

    # VmaxHigh
    m_v, y_v, thr_v = make_label_median_log10(df[COL_VMAX], mode="high_good")
    X_v = X_all.loc[m_v].copy()
    df_v = df.loc[m_v].copy()
    g_v = groups_all[m_v.values]

    print("\n" + "=" * 60)
    print("CAT VmaxHigh (strict split)")
    print("=" * 60)
    print(f"n={len(y_v)} | median(log10(Vmax))={thr_v:.6f} | pos_rate={y_v.mean():.3f} | groups={len(np.unique(g_v))}")
    run_one_task("CAT_VmaxHigh", df_v, X_v, y_v.astype(int), g_v, preprocess, "cat_cls_VmaxHigh")

    print("\nSaved to:", OUT_DIR)


if __name__ == "__main__":
    main()


CAT KmLow (strict split)
n=61 | median(log10(Km))=1.392697 | pos_rate=0.508 | groups=39

[Task] CAT_KmLow
样本数: 61 | DOI组数: 39
尝试分割次数: 50 | 有效 split 数: 50
[Running] CAT_KmLow | LogReg ...
[Running] CAT_KmLow | LinearSVC ...
[Running] CAT_KmLow | SVC_RBF ...
[Running] CAT_KmLow | KNN ...
[Running] CAT_KmLow | GaussianNB ...
[Running] CAT_KmLow | DecisionTree ...
[Running] CAT_KmLow | RF ...
[Running] CAT_KmLow | ExtraTrees ...
[Running] CAT_KmLow | GradientBoosting ...
[Running] CAT_KmLow | HistGB ...
[Running] CAT_KmLow | AdaBoost ...
[Running] CAT_KmLow | XGBoost ...

[BEST] CAT_KmLow = LinearSVC
           model  auc_mean  auc_median  auc_std  auc_max  kappa_mean  mcc_mean  acc_mean  n_splits
       LinearSVC  0.525269    0.545833 0.204610 1.000000    0.008247  0.010257  0.500817        50
      GaussianNB  0.523510    0.524405 0.207152 1.000000    0.003136  0.008631  0.531960        50
         SVC_RBF  0.511036    0.490741 0.221354 1.000000    0.031166  0.038836  0.504103        50

In [1]:
# -*- coding: utf-8 -*-
"""
CAT KmLow / VmaxHigh：标签方案 A+B 同时搜索（只改标签，不改特征）
A: 单阈值 (single_q)  q ∈ [0.30, 0.70] step=0.05
B: 两端切分 (extreme_q) 丢中间  q ∈ {0.30, 0.35, 0.40}  （可改）

严格 split + 12 模型不变。
输出：
- <task>_A_singleq_grid.csv
- <task>_B_extremeq_grid.csv
- <task>_overall_best.csv
- <task>_best_setting_model_summary.csv
- <task>_best_setting_labeled_data.csv
- <task>_best_setting_refit_full.joblib
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.metrics import roc_auc_score, cohen_kappa_score, matthews_corrcoef, accuracy_score

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


# =========================
# 0) 路径与配置
# =========================
CAT_PATH = r"C:\Users\86158\Desktop\dataiscat.xlsx"
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\cat_cls_threshold_tuning_AB")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_DOI = "data reference doi"
COL_KM = "Km/mM"
COL_VMAX = "Vmax/μM s-1"

# 严格 split（保持不变）
TEST_SIZE = 0.2
N_SPLITS = 50
SPLIT_SEED = 42

MIN_TEST_SAMPLES = 6
MIN_TRAIN_CLASS_COUNT = 2
MIN_TEST_CLASS_COUNT = 2

# 搜索参数
QS_SINGLE = np.round(np.arange(0.30, 0.70 + 1e-9, 0.05), 2).tolist()  # 0.30~0.70
QS_EXTREME = [0.30, 0.35, 0.40]  # 两端切分（丢中间）


# =========================
# 1) 特征定义（不改）
# =========================
FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate "
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = [
    "shape", "surface treatment", "dispersion medium", "Substrate "
]


# =========================
# 2) 预处理（不改）
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    X = df_raw[FEATURE_COLS].copy()

    for c in NUMERIC_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in CATEGORICAL_COLS:
        X[c] = clean_text_series(X[c])

    return X


def make_preprocess():
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True))])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[("num", num_pipe, NUMERIC_COLS), ("cat", cat_pipe, CATEGORICAL_COLS)],
        remainder="drop"
    )


# =========================
# 3) 模型（不改）
# =========================
def get_models():
    models = {
        "LogReg": LogisticRegression(max_iter=5000, class_weight="balanced",
                                     solver="liblinear", random_state=42),
        "LinearSVC": LinearSVC(C=1.0, class_weight="balanced", random_state=42),
        "SVC_RBF": SVC(C=1.0, gamma="scale", class_weight="balanced",
                       probability=True, random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=5, weights="distance"),
        "GaussianNB": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(max_depth=4, min_samples_leaf=2,
                                              class_weight="balanced", random_state=42),
        "RF": RandomForestClassifier(n_estimators=500, min_samples_leaf=2,
                                     class_weight="balanced", random_state=42, n_jobs=-1),
        "ExtraTrees": ExtraTreesClassifier(n_estimators=800, min_samples_leaf=2,
                                           class_weight="balanced", random_state=42, n_jobs=-1),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=200, learning_rate=0.05,
                                                       max_depth=3, random_state=42),
        "HistGB": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                                 max_depth=4, random_state=42),
        "AdaBoost": AdaBoostClassifier(n_estimators=300, learning_rate=0.05, random_state=42),
    }
    if HAS_XGBOOST:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=4,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
            objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", random_state=42, n_jobs=-1, verbosity=0
        )
    return models


def needs_scaling(model_name):
    return model_name in {"LogReg", "LinearSVC", "SVC_RBF", "KNN", "GaussianNB"}


def build_model_pipe(model_name, model, preprocess):
    steps = [("preprocess", preprocess)]
    if needs_scaling(model_name):
        steps.append(("scaler", StandardScaler(with_mean=True)))
    steps.append(("model", model))
    return Pipeline(steps)


def score_to_prob(pipe, X):
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X)[:, 1]
    if hasattr(pipe, "decision_function"):
        z = np.asarray(pipe.decision_function(X), dtype=float)
        return 1.0 / (1.0 + np.exp(-z))
    return np.asarray(pipe.predict(X), dtype=float)


# =========================
# 4) 标签（A/B）
# =========================
def build_labels_log10(values: pd.Series, task: str, scheme: str, q: float):
    v = pd.to_numeric(values, errors="coerce")
    m = v.notna() & (v > 0)
    vals = np.log10(v.loc[m].astype(float).values)

    if scheme == "single_q":
        if task == "KmLow":
            thr = float(np.quantile(vals, q))
            y = (vals <= thr).astype(int)
            keep = np.ones_like(y, dtype=bool)
            meta = {"scheme": scheme, "q": q, "thr": thr}
        else:  # VmaxHigh
            thr = float(np.quantile(vals, 1 - q))
            y = (vals >= thr).astype(int)
            keep = np.ones_like(y, dtype=bool)
            meta = {"scheme": scheme, "q": q, "thr": thr}

    elif scheme == "extreme_q":
        lo = float(np.quantile(vals, q))
        hi = float(np.quantile(vals, 1 - q))
        keep = (vals <= lo) | (vals >= hi)
        vals_k = vals[keep]
        if task == "KmLow":
            y = (vals_k <= lo).astype(int)
        else:
            y = (vals_k >= hi).astype(int)
        meta = {"scheme": scheme, "q": q, "lo": lo, "hi": hi, "dropped_mid": float(1 - 2*q)}
    else:
        raise ValueError(scheme)

    idx_all = np.where(m.values)[0]
    idx_keep = idx_all[keep]
    return idx_keep, y.astype(int), meta


# =========================
# 5) 评估
# =========================
def evaluate_one_setting(task_name, X_all, groups_all, idx_keep, y, preprocess):
    X = X_all.iloc[idx_keep].copy()
    groups = groups_all[idx_keep].copy()
    y = np.asarray(y, int)

    splitter = GroupShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=SPLIT_SEED)
    models = get_models()
    rows = []

    for model_name, model in models.items():
        for split_id, (tr, te) in enumerate(splitter.split(X, y, groups=groups), start=1):
            if len(te) < MIN_TEST_SAMPLES:
                continue

            ytr, yte = y[tr], y[te]
            if (np.sum(ytr == 0) < MIN_TRAIN_CLASS_COUNT) or (np.sum(ytr == 1) < MIN_TRAIN_CLASS_COUNT):
                continue
            if (np.sum(yte == 0) < MIN_TEST_CLASS_COUNT) or (np.sum(yte == 1) < MIN_TEST_CLASS_COUNT):
                continue

            try:
                pipe = build_model_pipe(model_name, model, preprocess)
                pipe.fit(X.iloc[tr], ytr)

                pte = score_to_prob(pipe, X.iloc[te])
                pred = (pte >= 0.5).astype(int)

                rows.append({
                    "model": model_name,
                    "split": split_id,
                    "auc": float(roc_auc_score(yte, pte)),
                    "kappa": float(cohen_kappa_score(yte, pred)),
                    "mcc": float(matthews_corrcoef(yte, pred)),
                    "acc": float(accuracy_score(yte, pred)),
                })
            except Exception:
                rows.append({
                    "model": model_name,
                    "split": split_id,
                    "auc": np.nan, "kappa": np.nan, "mcc": np.nan, "acc": np.nan
                })

    res = pd.DataFrame(rows)
    res_ok = res.dropna(subset=["auc", "kappa", "mcc", "acc"]).copy()
    if res_ok.empty:
        return pd.DataFrame(), pd.DataFrame()

    summ = (
        res_ok.groupby("model", as_index=False)
        .agg(
            auc_mean=("auc", "mean"),
            auc_median=("auc", "median"),
            auc_std=("auc", "std"),
            auc_max=("auc", "max"),
            kappa_mean=("kappa", "mean"),
            mcc_mean=("mcc", "mean"),
            acc_mean=("acc", "mean"),
            n_splits=("split", "count")
        )
        .sort_values(
            by=["auc_mean","auc_median","mcc_mean","kappa_mean","acc_mean","auc_max"],
            ascending=[False, False, False, False, False, False]
        )
        .reset_index(drop=True)
    )
    return res_ok, summ


# =========================
# 6) 搜索 A/B 并输出 overall best
# =========================
def run_task(task_name, df, X_all, groups_all, preprocess, col_target):
    results_A = []
    results_B = []

    # A：single_q
    for q in QS_SINGLE:
        idx_keep, y, meta = build_labels_log10(df[col_target], task_name, "single_q", q)
        if len(y) < 30 or len(np.unique(y)) < 2:
            continue
        _, summ = evaluate_one_setting(task_name, X_all, groups_all, idx_keep, y, preprocess)
        if summ.empty:
            continue
        best = summ.iloc[0].to_dict()
        results_A.append({
            **meta,
            "n_samples": int(len(y)),
            "n_groups": int(len(np.unique(groups_all[idx_keep]))),
            "best_model": best["model"],
            "best_auc_mean": float(best["auc_mean"]),
            "best_auc_median": float(best["auc_median"]),
            "best_n_splits": int(best["n_splits"]),
        })

    # B：extreme_q
    for q in QS_EXTREME:
        idx_keep, y, meta = build_labels_log10(df[col_target], task_name, "extreme_q", q)
        if len(y) < 30 or len(np.unique(y)) < 2:
            continue
        _, summ = evaluate_one_setting(task_name, X_all, groups_all, idx_keep, y, preprocess)
        if summ.empty:
            continue
        best = summ.iloc[0].to_dict()
        results_B.append({
            **meta,
            "n_samples": int(len(y)),
            "n_groups": int(len(np.unique(groups_all[idx_keep]))),
            "best_model": best["model"],
            "best_auc_mean": float(best["auc_mean"]),
            "best_auc_median": float(best["auc_median"]),
            "best_n_splits": int(best["n_splits"]),
        })

    dfA = pd.DataFrame(results_A).sort_values(
        by=["best_auc_mean","best_auc_median","best_n_splits"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    dfB = pd.DataFrame(results_B).sort_values(
        by=["best_auc_mean","best_auc_median","best_n_splits"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    dfA.to_csv(OUT_DIR / f"{task_name}_A_singleq_grid.csv", index=False, encoding="utf-8-sig")
    dfB.to_csv(OUT_DIR / f"{task_name}_B_extremeq_grid.csv", index=False, encoding="utf-8-sig")

    # overall best
    cand = []
    if not dfA.empty:
        cand.append(dfA.iloc[0].to_dict())
    if not dfB.empty:
        cand.append(dfB.iloc[0].to_dict())

    if not cand:
        print(f"[{task_name}] 没有可用方案")
        return

    overall = pd.DataFrame(cand).sort_values(
        by=["best_auc_mean","best_auc_median","best_n_splits"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    overall.to_csv(OUT_DIR / f"{task_name}_overall_best.csv", index=False, encoding="utf-8-sig")

    best_meta = overall.iloc[0].to_dict()
    print("\n" + "="*70)
    print(f"[{task_name}] OVERALL BEST (from A/B)")
    print("="*70)
    print(best_meta)

    # 复现该 best：重新生成 idx_keep/y 并全量重训保存
    scheme = best_meta["scheme"]
    q = float(best_meta["q"])
    idx_keep, y, meta = build_labels_log10(df[col_target], task_name, scheme, q)

    models = get_models()
    best_model_name = best_meta["best_model"]
    pipe = build_model_pipe(best_model_name, models[best_model_name], preprocess)

    X_use = X_all.iloc[idx_keep].copy()
    pipe.fit(X_use, y)

    df_use = df.iloc[idx_keep].copy()
    df_use["y"] = y
    df_use.to_csv(OUT_DIR / f"{task_name}_best_setting_labeled_data.csv", index=False, encoding="utf-8-sig")

    joblib.dump(
        {"task": task_name, "labeling_meta": meta, "best_model": best_model_name, "pipeline": pipe},
        OUT_DIR / f"{task_name}_best_setting_refit_full.joblib"
    )


def main():
    df = pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME)

    groups_all = df[COL_DOI].fillna("").astype(str).str.strip()
    miss = groups_all.eq("")
    groups_all.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df.index[miss]]
    groups_all = groups_all.values

    X_all = prepare_features(df)
    preprocess = make_preprocess()

    run_task("KmLow", df, X_all, groups_all, preprocess, COL_KM)
    run_task("VmaxHigh", df, X_all, groups_all, preprocess, COL_VMAX)

    print("\nSaved to:", OUT_DIR)


if __name__ == "__main__":
    main()


[KmLow] OVERALL BEST (from A/B)
{'scheme': 'extreme_q', 'q': 0.4, 'thr': nan, 'n_samples': 50, 'n_groups': 32, 'best_model': 'GradientBoosting', 'best_auc_mean': 0.7201823839412492, 'best_auc_median': 0.7333333333333333, 'best_n_splits': 47, 'lo': 1.0253058652647702, 'hi': 1.8055008581584002, 'dropped_mid': 0.19999999999999996}

[VmaxHigh] OVERALL BEST (from A/B)
{'scheme': 'single_q', 'q': 0.65, 'thr': 0.06519065318914162, 'n_samples': 57, 'n_groups': 35, 'best_model': 'DecisionTree', 'best_auc_mean': 0.6702681252681253, 'best_auc_median': 0.6904761904761905, 'best_n_splits': 37, 'lo': nan, 'hi': nan, 'dropped_mid': nan}

Saved to: C:\Users\86158\Desktop\cat_cls_threshold_tuning_AB


In [5]:
# -*- coding: utf-8 -*-
"""
CAT 最优模型全量重训 + SHAP + 模型重要性（按固定阈值，不再用q重算）

你给的最优结果：
1) KmLow: scheme=extreme_q, q=0.4
   固定阈值（log10尺度）:
     lo = 1.0253058652647702
     hi = 1.8055008581584002
   只保留两端：logKm <= lo 或 logKm >= hi（丢中间）
   标签：KmLow(好)=1 表示 logKm <= lo
   最优模型：GradientBoosting

2) VmaxHigh: scheme=single_q, q=0.65
   固定阈值（log10尺度）:
     thr = 0.06519065318914162
   保留全部样本（只要Vmax>0）
   标签：VmaxHigh(好)=1 表示 logVmax >= thr
   最优模型：DecisionTree

输出目录：
C:\\Users\\86158\\Desktop\\cat_best_refit_shap

输出文件（每个任务一套）：
- <task>_used_data_fixed_threshold.csv
- <task>_best_model_refit_full.joblib
- <task>_model_feature_importances.csv + png（transformed features）
- <task>_model_feature_importances_agg_by_original.csv
- <task>_shap_mean_abs_per_transformed_feature.csv
- <task>_shap_mean_abs_agg_by_original_feature.csv
- <task>_shap_beeswarm.png / <task>_shap_bar.png
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False


# =========================
# 0) 路径
# =========================
CAT_PATH = r"C:\Users\86158\Desktop\dataiscat.xlsx"
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\cat_best_refit_shap")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_DOI = "data reference doi"
COL_KM = "Km/mM"
COL_VMAX = "Vmax/μM s-1"

RANDOM_SEED = 42


# =========================
# 1) 固定阈值（log10尺度）
# =========================
# KmLow extreme thresholds (log10)
KM_LO = 1.0253058652647702
KM_HI = 1.8055008581584002

# VmaxHigh single threshold (log10)
VMAX_THR = 0.06519065318914162


# =========================
# 2) 特征定义（不改，沿用你体系）
# =========================
FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate "
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = [
    "shape", "surface treatment", "dispersion medium", "Substrate "
]


# =========================
# 3) 预处理（不改）
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s

def prepare_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    X = df_raw[FEATURE_COLS].copy()

    for c in NUMERIC_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    # 3rd metal 三列缺失补0（含 3rd metal type，按你设定不改）
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in CATEGORICAL_COLS:
        X[c] = clean_text_series(X[c])

    return X

def make_preprocess():
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True))])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[("num", num_pipe, NUMERIC_COLS), ("cat", cat_pipe, CATEGORICAL_COLS)],
        remainder="drop"
    )


# =========================
# 4) 名称映射：transformed -> original（用于聚合）
# =========================
def base_feature_from_transformed(name: str) -> str:
    n = str(name)
    if "__" in n:
        n = n.split("__", 1)[1]
    n = n.replace("missingindicator_", "")
    for c in CATEGORICAL_COLS:
        if n == c or n.startswith(c + "_"):
            return c
    for c in NUMERIC_COLS:
        if n == c:
            return c
    return n.split("_")[0]


# =========================
# 5) 图表工具
# =========================
def plot_top20_bar(df_imp, col, out_png, title):
    top = df_imp.sort_values(col, ascending=False).head(20).iloc[::-1]
    plt.figure(figsize=(10, 7))
    plt.barh(top["feature"], top[col])
    plt.xlabel(col)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=200)
    plt.close()


# =========================
# 6) SHAP 输出（TreeExplainer，必要时回退Kernel）
# =========================
def normalize_shap(shap_values):
    if isinstance(shap_values, list):
        if len(shap_values) == 2:
            return np.asarray(shap_values[1])
        return np.asarray(shap_values[0])
    arr = np.asarray(shap_values)
    if arr.ndim == 3 and arr.shape[-1] == 2:
        return arr[:, :, 1]
    return arr

def run_shap(task, model, X_t, feat_names, out_prefix):
    if not HAS_SHAP:
        print(f"[Warn] shap 未安装，跳过 {task} 的 SHAP。pip install shap")
        return

    try:
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_t)
    except Exception:
        # fallback：KernelExplainer（慢，但你的样本量不大）
        f = lambda X: model.predict_proba(X)[:, 1] if hasattr(model, "predict_proba") else model.predict(X)
        bg = shap.sample(X_t, min(12, X_t.shape[0]), random_state=RANDOM_SEED)
        explainer = shap.KernelExplainer(f, bg)
        shap_values = explainer.shap_values(X_t, nsamples=100)

    sv = normalize_shap(shap_values)
    mean_abs = np.mean(np.abs(sv), axis=0)

    # transformed-level
    df_shap = pd.DataFrame({
        "feature_transformed": feat_names,
        "mean_abs_shap": mean_abs
    }).sort_values("mean_abs_shap", ascending=False)
    df_shap.to_csv(OUT_DIR / f"{out_prefix}_shap_mean_abs_per_transformed_feature.csv",
                   index=False, encoding="utf-8-sig")

    # aggregated to original
    df_shap["feature_original"] = df_shap["feature_transformed"].apply(base_feature_from_transformed)
    df_agg = df_shap.groupby("feature_original", as_index=False)["mean_abs_shap"].sum() \
                    .sort_values("mean_abs_shap", ascending=False)
    df_agg.to_csv(OUT_DIR / f"{out_prefix}_shap_mean_abs_agg_by_original_feature.csv",
                  index=False, encoding="utf-8-sig")

    # plots
    plt.figure()
    shap.summary_plot(sv, X_t, feature_names=feat_names, show=False, max_display=20)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{out_prefix}_shap_beeswarm.png", dpi=200)
    plt.close()

    plt.figure()
    shap.summary_plot(sv, X_t, feature_names=feat_names, plot_type="bar", show=False, max_display=20)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{out_prefix}_shap_bar.png", dpi=200)
    plt.close()


# =========================
# 7) 模型重要性（feature_importances_）
# =========================
def run_model_importance(task, model, feat_names, out_prefix):
    if not hasattr(model, "feature_importances_"):
        return

    fi = np.asarray(model.feature_importances_, float)
    df_imp = pd.DataFrame({"feature": feat_names, "feature_importances_": fi}) \
                .sort_values("feature_importances_", ascending=False)

    df_imp.to_csv(OUT_DIR / f"{out_prefix}_model_feature_importances.csv",
                  index=False, encoding="utf-8-sig")

    plot_top20_bar(df_imp, "feature_importances_",
                   OUT_DIR / f"{out_prefix}_model_feature_importances.png",
                   f"{task} | model.feature_importances_ (transformed features)")

    # 聚合到原始特征
    df_imp["feature_original"] = df_imp["feature"].apply(base_feature_from_transformed)
    df_agg = df_imp.groupby("feature_original", as_index=False)["feature_importances_"].sum() \
                   .sort_values("feature_importances_", ascending=False)
    df_agg.to_csv(OUT_DIR / f"{out_prefix}_model_feature_importances_agg_by_original.csv",
                  index=False, encoding="utf-8-sig")


# =========================
# 8) 两个任务：固定阈值打标签 + 全量重训 + 解释
# =========================
def fit_and_explain_km(df):
    # 只用 Km>0 的行
    km = pd.to_numeric(df[COL_KM], errors="coerce")
    m = km.notna() & (km > 0)
    df0 = df.loc[m].copy()
    logkm = np.log10(km.loc[m].astype(float).values)

    # extreme：只保留两端（固定 lo/hi）
    keep = (logkm <= KM_LO) | (logkm >= KM_HI)
    df_use = df0.iloc[keep].copy()
    logkm_use = logkm[keep]

    y = (logkm_use <= KM_LO).astype(int)   # 低Km=好(1)

    X_raw = prepare_features(df_use)
    preprocess = make_preprocess()
    X_t = preprocess.fit_transform(X_raw)
    feat_names = [str(x) for x in preprocess.get_feature_names_out()]

    model = GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42
    )
    model.fit(X_t, y)

    # 保存 used data（含 logKm）
    out_prefix = "CAT_KmLow_best"
    df_save = df_use.copy()
    df_save["log10_Km"] = logkm_use
    df_save["y"] = y
    df_save.to_csv(OUT_DIR / f"{out_prefix}_used_data_fixed_threshold.csv",
                   index=False, encoding="utf-8-sig")

    # 保存 pipeline（把 preprocess + model 封装）
    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    joblib.dump(pipe, OUT_DIR / f"{out_prefix}_best_model_refit_full.joblib")

    print(f"[KmLow] n={len(y)} pos_rate={float(np.mean(y)):.3f} fixed lo={KM_LO:.6f} hi={KM_HI:.6f}")

    run_model_importance("CAT KmLow", model, feat_names, out_prefix)
    run_shap("CAT KmLow", model, X_t, feat_names, out_prefix)


def fit_and_explain_vmax(df):
    vmax = pd.to_numeric(df[COL_VMAX], errors="coerce")
    m = vmax.notna() & (vmax > 0)
    df_use = df.loc[m].copy()
    logv = np.log10(vmax.loc[m].astype(float).values)

    y = (logv >= VMAX_THR).astype(int)  # 高Vmax=好(1)

    X_raw = prepare_features(df_use)
    preprocess = make_preprocess()
    X_t = preprocess.fit_transform(X_raw)
    feat_names = [str(x) for x in preprocess.get_feature_names_out()]

    model = DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=2, class_weight="balanced", random_state=42
    )
    model.fit(X_t, y)

    out_prefix = "CAT_VmaxHigh_best"
    df_save = df_use.copy()
    df_save["log10_Vmax"] = logv
    df_save["y"] = y
    df_save.to_csv(OUT_DIR / f"{out_prefix}_used_data_fixed_threshold.csv",
                   index=False, encoding="utf-8-sig")

    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    joblib.dump(pipe, OUT_DIR / f"{out_prefix}_best_model_refit_full.joblib")

    print(f"[VmaxHigh] n={len(y)} pos_rate={float(np.mean(y)):.3f} fixed thr={VMAX_THR:.6f}")

    run_model_importance("CAT VmaxHigh", model, feat_names, out_prefix)
    run_shap("CAT VmaxHigh", model, X_t, feat_names, out_prefix)


def main():
    df = pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME)

    # 基本列检查
    need = FEATURE_COLS + [COL_KM, COL_VMAX]
    miss = [c for c in need if c not in df.columns]
    if miss:
        raise ValueError(f"缺少列：{miss}")

    fit_and_explain_km(df)
    fit_and_explain_vmax(df)

    print("\nSaved to:", OUT_DIR)


if __name__ == "__main__":
    main()

[KmLow] n=50 pos_rate=0.500 fixed lo=1.025306 hi=1.805501
[VmaxHigh] n=57 pos_rate=0.649 fixed thr=0.065191

Saved to: C:\Users\86158\Desktop\cat_best_refit_shap


In [11]:
# -*- coding: utf-8 -*-
"""
CAT regression: logKm / logVmax (separately) with DOI-GroupKFold OOF.
Multi-model + hierarchical parameter search (Stage1 coarse -> Stage2 fine) + save best model.

Targets:
  - logKm   = log10(Km/mM)
  - logVmax = log10(Vmax/μM s-1)

Pipeline (configurable):
  preprocess(num+cat+rare+onehot)
    -> VarianceThreshold
    -> (optional) SelectKBest(f_regression)
    -> (optional) TruncatedSVD
    -> (optional) StandardScaler
    -> regressor (Ridge/ElasticNet/BayesianRidge/Huber/SVR/KRR)

Outputs (per target):
  - <target>_search_summary.csv
  - <target>_best_config.json
  - <target>_oof_predictions_groupkfold.csv
  - <target>_best_model_refit_full.joblib

Also:
  - used_data_logKm.csv / used_data_logVmax.csv
  - feature_info.csv

Notes:
  - Column names normalized: strip + compress multiple spaces.
  - RareCategoryGrouper fitted fold-wise (no leakage).
  - k_best / svd_dim safe-adjusted to avoid invalid settings.
"""

import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import VarianceThreshold, f_regression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import Ridge, ElasticNet, HuberRegressor, BayesianRidge
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge


# =========================
# 0) Paths & global config
# =========================
CAT_PATH = r"C:\Users\86158\Desktop\dataiscat.xlsx"   # <-- 改成你的
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\cat_km_vmax_reg_doi_multistage_out")  # <-- 改成你的
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_DOI  = "data reference doi"
COL_KM   = "Km/mM"
COL_VMAX = "Vmax/μM s-1"

RANDOM_SEED = 42
MAX_FOLDS = 5

# True: 更快（推荐先同步跑通）；False: 网格更大更慢
FAST_MODE = False

# Stage2：每个 target 只对 Stage1 里最强的 TopK 模型族做精筛
TOPK_MODEL_FAMILIES_STAGE2 = 1


# =========================
# 1) Column normalize
# =========================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """strip + 多空格压成单空格（解决 main  metal proportion / Substrate 这种坑）"""
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


# =========================
# 2) Feature preprocessing
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw, feature_cols, numeric_cols, categorical_cols):
    X = df_raw[feature_cols].copy()

    for c in numeric_cols:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    # 3rd metal missing -> 0
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in categorical_cols:
        if c in X.columns:
            X[c] = clean_text_series(X[c])

    return X


class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    """Fold-wise rare category grouping: freq < min_freq -> __other__"""
    def __init__(self, min_freq=3, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


def build_preprocess(numeric_cols, categorical_cols, min_freq_cat=3, sparse=True):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=sparse)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=min_freq_cat)),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop",
    )


class ToDense(BaseEstimator, TransformerMixin):
    """Convert sparse matrix -> dense ndarray (ok for small datasets)."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if hasattr(X, "toarray"):
            return X.toarray()
        return np.asarray(X)


class SelectKBestSafe(BaseEstimator, TransformerMixin):
    """Safe SelectKBest: auto shrink k if k > n_features. If k is None -> pass-through."""
    def __init__(self, k=None):
        self.k = k
        self.selector_ = None
        self.k_effective_ = None

    def fit(self, X, y):
        if self.k is None:
            self.selector_ = None
            self.k_effective_ = None
            return self
        n_feat = X.shape[1]
        k_eff = int(min(int(self.k), int(n_feat)))
        if k_eff <= 0:
            self.selector_ = None
            self.k_effective_ = None
            return self
        from sklearn.feature_selection import SelectKBest
        self.selector_ = SelectKBest(score_func=f_regression, k=k_eff)
        self.selector_.fit(X, y)
        self.k_effective_ = k_eff
        return self

    def transform(self, X):
        if self.selector_ is None:
            return X
        return self.selector_.transform(X)


class TruncatedSVDSafe(BaseEstimator, TransformerMixin):
    """Safe TruncatedSVD: auto shrink n_components to n_features-1. If None -> pass-through."""
    def __init__(self, n_components=None, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.svd_ = None
        self.n_components_effective_ = None

    def fit(self, X, y=None):
        if self.n_components is None:
            self.svd_ = None
            self.n_components_effective_ = None
            return self
        n_feat = X.shape[1]
        n_comp = int(self.n_components)
        n_eff = min(n_comp, max(1, n_feat - 1))
        if n_feat <= 1:
            self.svd_ = None
            self.n_components_effective_ = None
            return self
        self.svd_ = TruncatedSVD(n_components=n_eff, random_state=self.random_state)
        self.svd_.fit(X)
        self.n_components_effective_ = n_eff
        return self

    def transform(self, X):
        if self.svd_ is None:
            return X
        return self.svd_.transform(X)


# =========================
# 3) Metrics + OOF
# =========================
def pearsonr_safe(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    r = pearsonr_safe(y_true, y_pred)
    return {
        "r2_log_oof": float(r2_score(y_true, y_pred)),
        "mae_log_oof": float(mean_absolute_error(y_true, y_pred)),
        "rmse_log_oof": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "pearson_r_oof": float(r) if not pd.isna(r) else np.nan,
        "pearson_r2_oof": float(r * r) if not pd.isna(r) else np.nan,
    }


def make_groups(df_unit, col_doi):
    groups = df_unit[col_doi].fillna("").astype(str).str.strip()
    miss = groups.eq("")
    groups.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df_unit.index[miss]]
    return groups.values


def groupkfold_oof_predict(X_raw, y_log, groups, pipe_builder):
    uniq_groups = np.unique(groups)
    n_folds = min(MAX_FOLDS, len(uniq_groups))
    if n_folds < 3:
        return None

    gkf = GroupKFold(n_splits=n_folds)
    oof_pred = np.full(len(y_log), np.nan, dtype=float)

    for _, (tr, te) in enumerate(gkf.split(X_raw, y_log, groups=groups), start=1):
        Xtr, Xte = X_raw.iloc[tr].copy(), X_raw.iloc[te].copy()
        ytr = y_log[tr]

        pipe = pipe_builder()
        pipe.fit(Xtr, ytr)
        oof_pred[te] = np.asarray(pipe.predict(Xte)).reshape(-1)

    ok = np.isfinite(oof_pred)
    return oof_pred, ok, n_folds


# =========================
# 4) Model factory + grids (Stage1/Stage2)
# =========================
def instantiate_model(model_name, params):
    if model_name == "Ridge":
        return Ridge(random_state=RANDOM_SEED, **params)
    if model_name == "ElasticNet":
        return ElasticNet(max_iter=50000, random_state=RANDOM_SEED, **params)
    if model_name == "BayesianRidge":
        return BayesianRidge(**params)
    if model_name == "Huber":
        return HuberRegressor(**params)
    if model_name == "SVR_RBF":
        return SVR(kernel="rbf", **params)
    if model_name == "KernelRidge_RBF":
        return KernelRidge(kernel="rbf", **params)
    raise ValueError(model_name)


def stage1_model_grids(fast=True):
    if fast:
        return {
            "Ridge": [{"alpha": a} for a in [1, 10, 100]],
            "ElasticNet": [{"alpha": a, "l1_ratio": r} for a in [0.01, 0.1] for r in [0.2, 0.5, 0.8]],
            "BayesianRidge": [{}],
            "Huber": [{"alpha": 1e-4, "epsilon": e} for e in [1.35, 1.5]],
            "SVR_RBF": [{"C": C, "gamma": g, "epsilon": e}
                        for C in [10, 30]
                        for g in ["scale", 0.03, 0.1]
                        for e in [0.05, 0.1]],
            "KernelRidge_RBF": [{"alpha": a, "gamma": g}
                                for a in [0.1, 1, 10]
                                for g in [0.03, 0.1]],
        }
    return {
        "Ridge": [{"alpha": a} for a in [0.3, 1, 3, 10, 30, 100, 300]],
        "ElasticNet": [{"alpha": a, "l1_ratio": r} for a in [0.003, 0.01, 0.03, 0.1, 0.3] for r in [0.1, 0.3, 0.5, 0.7, 0.9]],
        "BayesianRidge": [{}],
        "Huber": [{"alpha": a, "epsilon": e} for a in [1e-5, 1e-4, 1e-3] for e in [1.2, 1.35, 1.5, 1.8]],
        "SVR_RBF": [{"C": C, "gamma": g, "epsilon": e}
                    for C in [3, 10, 30, 100]
                    for g in ["scale", "auto", 0.01, 0.03, 0.1, 0.3]
                    for e in [0.03, 0.05, 0.1, 0.2]],
        "KernelRidge_RBF": [{"alpha": a, "gamma": g}
                            for a in [0.01, 0.1, 1, 10, 100]
                            for g in [0.01, 0.03, 0.1, 0.3]],
    }


def stage2_model_grids(model_name):
    if model_name == "Ridge":
        return [{"alpha": a} for a in [0.3, 1, 3, 10, 30, 100, 300]]
    if model_name == "ElasticNet":
        return [{"alpha": a, "l1_ratio": r} for a in [0.003, 0.01, 0.03, 0.1, 0.3] for r in [0.1, 0.3, 0.5, 0.7, 0.9]]
    if model_name == "BayesianRidge":
        return [{}]
    if model_name == "Huber":
        return [{"alpha": a, "epsilon": e} for a in [1e-5, 1e-4, 1e-3] for e in [1.2, 1.35, 1.5, 1.8]]
    if model_name == "SVR_RBF":
        return [{"C": C, "gamma": g, "epsilon": e}
                for C in [3, 10, 30, 100]
                for g in ["scale", "auto", 0.01, 0.03, 0.1, 0.3]
                for e in [0.03, 0.05, 0.1, 0.2]]
    if model_name == "KernelRidge_RBF":
        return [{"alpha": a, "gamma": g}
                for a in [0.01, 0.1, 1, 10, 100]
                for g in [0.01, 0.03, 0.1, 0.3]]
    raise ValueError(model_name)


def stage1_preprocess_grid(fast=True):
    if fast:
        return {
            "min_freq_cat": [2, 3],
            "var_thr": [0.0, 1e-5],
            "k_best": [None, 12],
            "svd_dim": [None, 8, 10],
        }
    return {
        "min_freq_cat": [2, 3],
        "var_thr": [0.0, 1e-5, 1e-4],
        "k_best": [None, 8, 10, 12, 15],
        "svd_dim": [None, 6, 8, 10, 12],
    }


def stage2_preprocess_grid():
    return {
        "min_freq_cat": [2, 3],
        "var_thr": [0.0, 1e-5, 1e-4],
        "k_best": [None, 8, 10, 12, 15],
        "svd_dim": [None, 6, 8, 10, 12],
    }


# =========================
# 5) Build pipeline for one config
# =========================
def build_pipeline_for_config(numeric_cols, categorical_cols,
                             min_freq_cat, var_thr, k_best, svd_dim,
                             model_name, model_params):
    needs_dense = model_name in {"BayesianRidge", "Huber", "SVR_RBF", "KernelRidge_RBF"}

    preprocess = build_preprocess(numeric_cols, categorical_cols, min_freq_cat=min_freq_cat, sparse=True)

    steps = []
    steps.append(("preprocess", preprocess))
    steps.append(("var", VarianceThreshold(threshold=float(var_thr))))

    if k_best is not None:
        steps.append(("kbest", SelectKBestSafe(k=int(k_best))))

    if svd_dim is not None:
        steps.append(("svd", TruncatedSVDSafe(n_components=int(svd_dim), random_state=RANDOM_SEED)))
        steps.append(("scaler", StandardScaler(with_mean=True)))
    else:
        if needs_dense:
            steps.append(("todense", ToDense()))
            steps.append(("scaler", StandardScaler(with_mean=True)))
        else:
            steps.append(("scaler", StandardScaler(with_mean=False)))

    steps.append(("model", instantiate_model(model_name, model_params)))
    return Pipeline(steps)


# =========================
# 6) Search for one target
# =========================
def run_target(target_name, df_target, feature_cols, numeric_cols, categorical_cols, y_col):
    df_target = df_target.copy()

    # y filter: numeric & > 0
    y_num = pd.to_numeric(df_target[y_col], errors="coerce")
    df_target = df_target[y_num.notna() & (y_num > 0)].copy()
    if len(df_target) < 10:
        print(f"[Skip] {target_name}: too few samples: {len(df_target)}")
        return None

    y_log = np.log10(pd.to_numeric(df_target[y_col], errors="coerce").astype(float).values)

    groups = make_groups(df_target, COL_DOI)
    X_raw = prepare_features(df_target, feature_cols, numeric_cols, categorical_cols)

    n_groups = int(len(np.unique(groups)))
    singleton_groups = int(pd.Series(groups).value_counts().eq(1).sum())
    print(f"\n[{target_name}] n_samples={len(df_target)} | n_groups={n_groups} | singleton_groups={singleton_groups}")

    all_rows = []

    # -------- Stage 1 --------
    pre_grid = stage1_preprocess_grid(fast=FAST_MODE)
    model_grids = stage1_model_grids(fast=FAST_MODE)

    for pre_params in ParameterGrid(pre_grid):
        min_freq_cat = pre_params["min_freq_cat"]
        var_thr = pre_params["var_thr"]
        k_best = pre_params["k_best"]
        svd_dim = pre_params["svd_dim"]

        if (k_best is not None) and (svd_dim is not None) and (int(svd_dim) >= int(k_best)):
            continue

        for model_name, grid_list in model_grids.items():
            for model_params in grid_list:
                def builder():
                    return build_pipeline_for_config(
                        numeric_cols, categorical_cols,
                        min_freq_cat=min_freq_cat,
                        var_thr=var_thr,
                        k_best=k_best,
                        svd_dim=svd_dim,
                        model_name=model_name,
                        model_params=model_params
                    )

                out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
                if out is None:
                    continue

                pred, ok, n_folds = out
                m = eval_metrics(y_log[ok], pred[ok])

                all_rows.append({
                    "stage": 1,
                    "target": target_name,
                    "model": model_name,
                    "min_freq_cat": int(min_freq_cat),
                    "var_thr": float(var_thr),
                    "k_best_req": None if k_best is None else int(k_best),
                    "svd_dim_req": None if svd_dim is None else int(svd_dim),
                    "model_params": json.dumps(model_params, ensure_ascii=False),
                    "n_samples": int(len(df_target)),
                    "n_groups": n_groups,
                    "singleton_groups": singleton_groups,
                    "n_folds": int(n_folds),
                    **m
                })

    res_stage1 = pd.DataFrame(all_rows)
    if res_stage1.empty:
        raise RuntimeError(f"{target_name}: no valid results in stage1 (too few DOI groups?)")

    fam_best = (
        res_stage1.sort_values(["r2_log_oof", "rmse_log_oof"], ascending=[False, True])
        .groupby("model", as_index=False)
        .head(1)
        .sort_values(["r2_log_oof", "rmse_log_oof"], ascending=[False, True])
    )
    shortlist = fam_best["model"].tolist()[:TOPK_MODEL_FAMILIES_STAGE2]
    print(f"[{target_name}] Stage1 shortlist -> {shortlist}")

    # -------- Stage 2 --------
    pre_grid2 = stage2_preprocess_grid()
    for model_name in shortlist:
        for pre_params in ParameterGrid(pre_grid2):
            min_freq_cat = pre_params["min_freq_cat"]
            var_thr = pre_params["var_thr"]
            k_best = pre_params["k_best"]
            svd_dim = pre_params["svd_dim"]
            if (k_best is not None) and (svd_dim is not None) and (int(svd_dim) >= int(k_best)):
                continue

            for model_params in stage2_model_grids(model_name):
                def builder():
                    return build_pipeline_for_config(
                        numeric_cols, categorical_cols,
                        min_freq_cat=min_freq_cat,
                        var_thr=var_thr,
                        k_best=k_best,
                        svd_dim=svd_dim,
                        model_name=model_name,
                        model_params=model_params
                    )

                out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
                if out is None:
                    continue

                pred, ok, n_folds = out
                m = eval_metrics(y_log[ok], pred[ok])

                all_rows.append({
                    "stage": 2,
                    "target": target_name,
                    "model": model_name,
                    "min_freq_cat": int(min_freq_cat),
                    "var_thr": float(var_thr),
                    "k_best_req": None if k_best is None else int(k_best),
                    "svd_dim_req": None if svd_dim is None else int(svd_dim),
                    "model_params": json.dumps(model_params, ensure_ascii=False),
                    "n_samples": int(len(df_target)),
                    "n_groups": n_groups,
                    "singleton_groups": singleton_groups,
                    "n_folds": int(n_folds),
                    **m
                })

    # -------- Select best (prefer stage2) --------
    res_all = pd.DataFrame(all_rows)
    res_all.to_csv(OUT_DIR / f"{target_name}_search_summary.csv", index=False, encoding="utf-8-sig")

    res_stage2 = res_all[res_all["stage"] == 2].copy()
    base = res_stage2 if not res_stage2.empty else res_all
    best = base.sort_values(by=["r2_log_oof", "rmse_log_oof"], ascending=[False, True]).iloc[0].to_dict()

    best_model = best["model"]
    best_min_freq = int(best["min_freq_cat"])
    best_var_thr = float(best["var_thr"])
    best_kbest = None if pd.isna(best["k_best_req"]) else int(best["k_best_req"])
    best_svd = None if pd.isna(best["svd_dim_req"]) else int(best["svd_dim_req"])
    best_model_params = json.loads(best["model_params"])

    def best_builder():
        return build_pipeline_for_config(
            numeric_cols, categorical_cols,
            min_freq_cat=best_min_freq,
            var_thr=best_var_thr,
            k_best=best_kbest,
            svd_dim=best_svd,
            model_name=best_model,
            model_params=best_model_params
        )

    oof, ok, n_folds = groupkfold_oof_predict(X_raw, y_log, groups, best_builder)
    pd.DataFrame({
        "row_index": df_target.index.values,
        "doi_group": groups,
        "y_log_true": y_log,
        "y_log_oof_pred": oof
    }).to_csv(OUT_DIR / f"{target_name}_oof_predictions_groupkfold.csv", index=False, encoding="utf-8-sig")

    final_pipe = best_builder()
    final_pipe.fit(X_raw, y_log)

    best_config = {
        "target": target_name,
        "y_col": y_col,
        "target_transform": "log10",
        "split": "GroupKFold by DOI",
        "data_stats": {
            "n_samples": int(len(df_target)),
            "n_groups": int(n_groups),
            "singleton_groups": int(singleton_groups),
            "n_folds": int(n_folds),
        },
        "best": {
            "stage": int(best["stage"]),
            "model": best_model,
            "preprocess_params": {
                "min_freq_cat": best_min_freq,
                "var_thr": best_var_thr,
                "k_best_req": best_kbest,
                "svd_dim_req": best_svd,
            },
            "model_params": best_model_params,
            "metrics_oof": {
                "r2_log_oof": float(best["r2_log_oof"]),
                "mae_log_oof": float(best["mae_log_oof"]),
                "rmse_log_oof": float(best["rmse_log_oof"]),
                "pearson_r_oof": float(best["pearson_r_oof"]) if pd.notna(best["pearson_r_oof"]) else None,
                "pearson_r2_oof": float(best["pearson_r2_oof"]) if pd.notna(best["pearson_r2_oof"]) else None,
            }
        },
        "features": {
            "feature_cols": feature_cols,
            "numeric_cols": numeric_cols,
            "categorical_cols": categorical_cols,
        }
    }

    with open(OUT_DIR / f"{target_name}_best_config.json", "w", encoding="utf-8") as f:
        json.dump(best_config, f, ensure_ascii=False, indent=2)

    joblib.dump({
        "target": target_name,
        "pipeline": final_pipe,
        "best_config": best_config
    }, OUT_DIR / f"{target_name}_best_model_refit_full.joblib")

    print("\n" + "=" * 72)
    print(f"[BEST] {target_name} | stage={best_config['best']['stage']} | model={best_model}")
    print("preprocess:", best_config["best"]["preprocess_params"])
    print("model_params:", best_model_params)
    print("metrics_oof:", best_config["best"]["metrics_oof"])
    print("=" * 72)

    return best_config


# =========================
# 7) Main
# =========================
def main():
    df = pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME)
    df = normalize_columns(df)

    if COL_DOI not in df.columns:
        raise KeyError(f"DOI column not found: {COL_DOI}")
    if COL_KM not in df.columns:
        raise KeyError(f"Km column not found: {COL_KM}")
    if COL_VMAX not in df.columns:
        raise KeyError(f"Vmax column not found: {COL_VMAX}")

    # -------- canonical feature names after normalize_columns --------
    FEATURE_COLS = [
        "contain O","contain N","contain P","contain S","contain Si","contain Se","contain B","contain F","contain Cl","contain Br","contain I",
        "main metal proportion","main metal number","main metal valence",
        "minor metal percentage","minor metal number","minor metal valence",
        "3rd metal ratio","3rd metal type","3rd metal valence",
        "shape","size (nm)","surface treatment","dispersion medium","pH","temperature/oC","Substrate"
    ]
    NUMERIC_COLS = [
        "contain O","contain N","contain P","contain S","contain Si","contain Se","contain B","contain F","contain Cl","contain Br","contain I",
        "main metal proportion","main metal number","main metal valence",
        "minor metal percentage","minor metal number","minor metal valence",
        "3rd metal ratio","3rd metal type","3rd metal valence",
        "size (nm)","pH","temperature/oC"
    ]
    CATEGORICAL_COLS = ["shape", "surface treatment", "dispersion medium", "Substrate"]

    # intersect with existing columns to be robust
    FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
    NUMERIC_COLS = [c for c in NUMERIC_COLS if c in df.columns]
    CATEGORICAL_COLS = [c for c in CATEGORICAL_COLS if c in df.columns]

    pd.DataFrame({
        "feature": FEATURE_COLS,
        "type": ["numeric" if c in NUMERIC_COLS else "categorical" for c in FEATURE_COLS]
    }).to_csv(OUT_DIR / "feature_info.csv", index=False, encoding="utf-8-sig")

    # -------- run logKm --------
    df_km = df.copy()
    km_num = pd.to_numeric(df_km[COL_KM], errors="coerce")
    df_km = df_km[km_num.notna() & (km_num > 0)].copy()
    df_km.to_csv(OUT_DIR / "used_data_logKm.csv", index=False, encoding="utf-8-sig")
    run_target("logKm", df_km, FEATURE_COLS, NUMERIC_COLS, CATEGORICAL_COLS, y_col=COL_KM)

    # -------- run logVmax --------
    df_v = df.copy()
    v_num = pd.to_numeric(df_v[COL_VMAX], errors="coerce")
    df_v = df_v[v_num.notna() & (v_num > 0)].copy()
    df_v.to_csv(OUT_DIR / "used_data_logVmax.csv", index=False, encoding="utf-8-sig")
    run_target("logVmax", df_v, FEATURE_COLS, NUMERIC_COLS, CATEGORICAL_COLS, y_col=COL_VMAX)

    print("\nSaved to:", OUT_DIR)


if __name__ == "__main__":
    main()


[logKm] n_samples=61 | n_groups=39 | singleton_groups=26
[logKm] Stage1 shortlist -> ['KernelRidge_RBF']

[BEST] logKm | stage=2 | model=KernelRidge_RBF
preprocess: {'min_freq_cat': 2, 'var_thr': 0.0, 'k_best_req': None, 'svd_dim_req': 12}
model_params: {'alpha': 0.1, 'gamma': 0.3}
metrics_oof: {'r2_log_oof': -0.02101245549547137, 'mae_log_oof': 1.1843870211524175, 'rmse_log_oof': 1.4334629466794484, 'pearson_r_oof': 0.2820243829318553, 'pearson_r2_oof': 0.07953775256809374}

[logVmax] n_samples=57 | n_groups=35 | singleton_groups=22
[logVmax] Stage1 shortlist -> ['SVR_RBF']

[BEST] logVmax | stage=2 | model=SVR_RBF
preprocess: {'min_freq_cat': 2, 'var_thr': 0.0, 'k_best_req': 10, 'svd_dim_req': 8}
model_params: {'C': 30, 'gamma': 0.1, 'epsilon': 0.03}
metrics_oof: {'r2_log_oof': 0.2476354387570726, 'mae_log_oof': 1.3862560044364394, 'rmse_log_oof': 1.7463409141084685, 'pearson_r_oof': 0.5150260799053337, 'pearson_r2_oof': 0.26525186298265513}

Saved to: C:\Users\86158\Desktop\cat_km_


[logKm] n_samples=61 | n_groups=39 | singleton_groups=26
[logKm] search space ~= 348 configs (before CV folds)
  - SVR_RBF          | best R2=-0.1639 | RMSE=1.5305 | feature=enhanced_pruned | k=None | svd=None | time=27.7s
  - KernelRidge_RBF  | best R2=0.0938 | RMSE=1.3505 | feature=enhanced_pruned | k=10 | svd=8 | time=20.9s
  - ExtraTrees       | best R2=-0.0824 | RMSE=1.4759 | feature=enhanced_pruned | k=10 | svd=8 | time=195.9s

[BEST] logKm | model=KernelRidge_RBF | feature=enhanced_pruned
preprocess: {'min_freq_cat': 2, 'var_thr': 0.0, 'k_best_req': 10, 'svd_dim_req': 8}
model_params: {'alpha': 0.03, 'gamma': 0.3, '__use_y_scaler__': True}
metrics_oof: {'r2_log_oof': 0.09378067339510954, 'mae_log_oof': 1.1370226085273827, 'rmse_log_oof': 1.35047830365423, 'pearson_r_oof': 0.38603284996605425, 'pearson_r2_oof': 0.14902136125291415}
runtime_sec: 244.887

[logVmax] n_samples=57 | n_groups=35 | singleton_groups=22
[logVmax] search space ~= 348 configs (before CV folds)
  - SVR_RBF 


[logKm] n_samples=61 | n_groups=39 | singleton_groups=26
[logKm] search space ~= 208 configs (before CV folds)
  - KernelRidge_RBF  | best R2=0.1908 | RMSE=1.2762 | feature=enhanced_pruned | k=10 | svd=8 | time=15.4s
  - SVR_RBF          | best R2=-0.0546 | RMSE=1.4568 | feature=enhanced_pruned | k=10 | svd=8 | time=0.8s

[BEST] logKm | model=KernelRidge_RBF | feature=enhanced_pruned
preprocess: {'min_freq_cat': 2, 'var_thr': 0.0, 'k_best_req': 10, 'svd_dim_req': 8}
model_params: {'alpha': 0.02, 'gamma': 0.5, '__use_y_scaler__': True}
metrics_oof: {'r2_log_oof': 0.190769966876014, 'mae_log_oof': 1.0723358391371405, 'rmse_log_oof': 1.2761653469220784, 'pearson_r_oof': 0.4666666033229816, 'pearson_r2_oof': 0.21777771865700907}
runtime_sec: 16.665

[logVmax] n_samples=57 | n_groups=35 | singleton_groups=22
[logVmax] search space ~= 688 configs (before CV folds)
  - KernelRidge_RBF  | best R2=0.1181 | RMSE=1.8907 | feature=baseline | k=10 | svd=8 | time=2.8s
  - SVR_RBF          | best R2


[logKm] n_samples=61 | n_groups=39 | singleton_groups=26
[logKm] search space ~= 56 configs (before CV folds)
  - KernelRidge_RBF  | best R2=0.2332 | RMSE=1.2423 | feature=enhanced_pruned | k=10 | svd=8 | time=4.7s
  - SVR_RBF          | best R2=-0.0905 | RMSE=1.4814 | feature=enhanced_pruned | k=10 | svd=8 | time=0.8s

[BEST] logKm | model=KernelRidge_RBF | feature=enhanced_pruned
preprocess: {'min_freq_cat': 2, 'var_thr': 0.0, 'k_best_req': 10, 'svd_dim_req': 8}
model_params: {'alpha': 0.01, 'gamma': 0.5, '__use_y_scaler__': True}
metrics_oof: {'r2_log_oof': 0.23316898149055876, 'mae_log_oof': 1.0420234722260375, 'rmse_log_oof': 1.2422837011845025, 'pearson_r_oof': 0.5107885917987501, 'pearson_r2_oof': 0.2609049855117502}
runtime_sec: 5.719

[logVmax] n_samples=57 | n_groups=35 | singleton_groups=22
[logVmax] search space ~= 444 configs (before CV folds)
  - GPR_RBF          | best R2=0.0805 | RMSE=1.9306 | feature=baseline | k=10 | svd=8 | time=1.4s
  - KernelRidge_RBF  | best R2=0


[logKm] n_samples=61 | n_groups=39 | singleton_groups=26
[logKm] search space ~= 56 configs (before CV folds)
  - KernelRidge_RBF  | best R2=0.2332 | RMSE=1.2423 | feature=enhanced_pruned | k=10 | svd=8 | time=4.4s
  - SVR_RBF          | best R2=-0.0905 | RMSE=1.4814 | feature=enhanced_pruned | k=10 | svd=8 | time=0.8s

[BEST] logKm | model=KernelRidge_RBF | feature=enhanced_pruned
preprocess: {'min_freq_cat': 2, 'var_thr': 0.0, 'k_best_req': 10, 'svd_dim_req': 8}
model_params: {'alpha': 0.01, 'gamma': 0.5, '__use_y_scaler__': True}
metrics_oof: {'r2_log_oof': 0.23316898149055876, 'mae_log_oof': 1.0420234722260375, 'rmse_log_oof': 1.2422837011845025, 'pearson_r_oof': 0.5107885917987501, 'pearson_r2_oof': 0.2609049855117502}
runtime_sec: 5.436

[logVmax] n_samples=57 | n_groups=35 | singleton_groups=22
[logVmax] search space ~= 1036 configs (before CV folds)
  - GPR_Matern       | best R2=-0.0001 | RMSE=2.0134 | feature=baseline | k=10 | svd=8 | time=2.4s
  - GPR_RBF          | best R2

In [7]:
# -*- coding: utf-8 -*-
"""
CAT regression (reasonable expand v4): focus on Vmax with low-to-moderate compute.

核心思路：
1) 不改原始 Excel，只在内存里做特征工程。
2) 继续用 DOI-GroupKFold OOF 作为唯一搜索和汇报标准。
3) 主攻 logVmax，但不再盲目加很多新模型；重点尝试：
   - SVR_RBF（局部精修）
   - NuSVR_RBF（局部精修）
   - KernelRidge_RBF（小范围兜底）
   - KNN_distance（新模型，低计算量）
4) 新增两种更克制的特征模式：
   - baseline_no_substrate
   - baseline_slim
5) 新增两个合理但低成本的技巧：
   - 组均衡 sample_weight（按 DOI 组大小的倒数）
   - RobustScaler（替代 StandardScaler）

输出（每个 target）：
- <target>_search_summary.csv
- <target>_best_config.json
- <target>_oof_predictions_groupkfold.csv
- <target>_best_model_refit_full.joblib
"""

import json
import time
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import VarianceThreshold, f_regression, SelectKBest
from sklearn.impute import SimpleImputer
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.svm import SVR, NuSVR


# =========================
# 0) Paths & config
# =========================
CAT_PATH = Path(r"C:\Users\86158\Desktop\dataiscat.xlsx")  # <-- 改成你的输入文件
SHEET_NAME = 0

OUT_DIR = CAT_PATH.parent / "cat_km_vmax_reg_doi_multistage_out_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_DOI  = "data reference doi"
COL_KM   = "Km/mM"
COL_VMAX = "Vmax/μM s-1"

RANDOM_SEED = 42
MAX_FOLDS = 5

RUN_LOGKM = True
RUN_LOGVMAX = True
TRY_VMAX_ENSEMBLE = True


# =========================
# 1) Basic helpers
# =========================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan, "na": np.nan, "n/a": np.nan})
    return s


def _safe_num(s):
    return pd.to_numeric(s, errors="coerce")


def pearsonr_safe(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    r = pearsonr_safe(y_true, y_pred)
    return {
        "r2_log_oof": float(r2_score(y_true, y_pred)),
        "mae_log_oof": float(mean_absolute_error(y_true, y_pred)),
        "rmse_log_oof": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "pearson_r_oof": float(r) if not pd.isna(r) else np.nan,
        "pearson_r2_oof": float(r * r) if not pd.isna(r) else np.nan,
    }


def make_groups(df_unit, col_doi):
    groups = df_unit[col_doi].fillna("").astype(str).str.strip()
    miss = groups.eq("")
    groups.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df_unit.index[miss]]
    return groups.values


def make_inverse_group_weights(groups_1d):
    counts = pd.Series(groups_1d).value_counts()
    sw = np.array([1.0 / counts[g] for g in groups_1d], dtype=float)
    sw = sw / np.mean(sw)
    return sw


# =========================
# 2) Chemistry / feature engineering
# =========================
ATOMIC_INFO = {
    0:  {"symbol": "None", "period": 0, "group": 0,  "en": 0.0,  "radius_pm": 0.0,  "noble": 0, "rare_earth": 0},
    12: {"symbol": "Mg",   "period": 3, "group": 2,  "en": 1.31, "radius_pm": 160.0, "noble": 0, "rare_earth": 0},
    20: {"symbol": "Ca",   "period": 4, "group": 2,  "en": 1.00, "radius_pm": 197.0, "noble": 0, "rare_earth": 0},
    23: {"symbol": "V",    "period": 4, "group": 5,  "en": 1.63, "radius_pm": 135.0, "noble": 0, "rare_earth": 0},
    25: {"symbol": "Mn",   "period": 4, "group": 7,  "en": 1.55, "radius_pm": 127.0, "noble": 0, "rare_earth": 0},
    26: {"symbol": "Fe",   "period": 4, "group": 8,  "en": 1.83, "radius_pm": 126.0, "noble": 0, "rare_earth": 0},
    27: {"symbol": "Co",   "period": 4, "group": 9,  "en": 1.88, "radius_pm": 125.0, "noble": 0, "rare_earth": 0},
    28: {"symbol": "Ni",   "period": 4, "group": 10, "en": 1.91, "radius_pm": 124.0, "noble": 0, "rare_earth": 0},
    29: {"symbol": "Cu",   "period": 4, "group": 11, "en": 1.90, "radius_pm": 128.0, "noble": 0, "rare_earth": 0},
    40: {"symbol": "Zr",   "period": 5, "group": 4,  "en": 1.33, "radius_pm": 160.0, "noble": 0, "rare_earth": 0},
    42: {"symbol": "Mo",   "period": 5, "group": 6,  "en": 2.16, "radius_pm": 139.0, "noble": 0, "rare_earth": 0},
    44: {"symbol": "Ru",   "period": 5, "group": 8,  "en": 2.20, "radius_pm": 134.0, "noble": 1, "rare_earth": 0},
    45: {"symbol": "Rh",   "period": 5, "group": 9,  "en": 2.28, "radius_pm": 134.0, "noble": 1, "rare_earth": 0},
    46: {"symbol": "Pd",   "period": 5, "group": 10, "en": 2.20, "radius_pm": 137.0, "noble": 1, "rare_earth": 0},
    47: {"symbol": "Ag",   "period": 5, "group": 11, "en": 1.93, "radius_pm": 144.0, "noble": 1, "rare_earth": 0},
    58: {"symbol": "Ce",   "period": 6, "group": 3,  "en": 1.12, "radius_pm": 182.0, "noble": 0, "rare_earth": 1},
    68: {"symbol": "Er",   "period": 6, "group": 3,  "en": 1.24, "radius_pm": 176.0, "noble": 0, "rare_earth": 1},
    77: {"symbol": "Ir",   "period": 6, "group": 9,  "en": 2.20, "radius_pm": 136.0, "noble": 1, "rare_earth": 0},
    78: {"symbol": "Pt",   "period": 6, "group": 10, "en": 2.28, "radius_pm": 139.0, "noble": 1, "rare_earth": 0},
    79: {"symbol": "Au",   "period": 6, "group": 11, "en": 2.54, "radius_pm": 144.0, "noble": 1, "rare_earth": 0},
}
TRANSITION_GROUPS = set(range(3, 13))


def map_atomic_property(series_atomic_num: pd.Series, prop: str) -> pd.Series:
    vals = _safe_num(series_atomic_num).fillna(0).astype(int)
    return vals.map(lambda z: ATOMIC_INFO.get(int(z), ATOMIC_INFO[0])[prop])


def map_atomic_symbol(series_atomic_num: pd.Series) -> pd.Series:
    vals = _safe_num(series_atomic_num).fillna(0).astype(int)
    return vals.map(lambda z: ATOMIC_INFO.get(int(z), ATOMIC_INFO[0])["symbol"])


def coarse_shape(x):
    if pd.isna(x):
        return "unknown"
    s = str(x).strip().lower()
    if any(k in s for k in ["sphere", "spherical", "nanosphere", "particle", "nanoparticle", "cluster", "nanocluster"]):
        return "sphere_particle"
    if any(k in s for k in ["rod", "wire", "fiber", "tube"]):
        return "rod_wire_tube"
    if any(k in s for k in ["sheet", "plate", "flake", "layer"]):
        return "sheet_plate"
    if any(k in s for k in ["cube", "cubic", "octa", "dodeca", "rhombic"]):
        return "polyhedral"
    if any(k in s for k in ["porous", "hollow", "mesoporous"]):
        return "porous_hollow"
    if any(k in s for k in ["flower", "urchin", "aggregate"]):
        return "flower_aggregate"
    return "other"


def coarse_surface(x):
    if pd.isna(x):
        return "unknown"
    s = str(x).strip().lower()
    if any(k in s for k in ["none", "bare", "unmodified", "without"]):
        return "bare"
    if any(k in s for k in ["peg", "pvp", "pva", "poly", "dextran", "chitosan", "pamam"]):
        return "polymer"
    if any(k in s for k in ["bsa", "albumin", "protein", "peptide", "enzyme", "silk", "apo"]):
        return "protein_biomolecule"
    if any(k in s for k in ["sio2", "silica"]):
        return "silica"
    if any(k in s for k in ["citrate", "ctab", "surfactant", "oleic", "ligand", "mpa", "gsh", "histidine"]):
        return "small_molecule_surfactant"
    return "other"


def coarse_medium(x):
    if pd.isna(x):
        return "unknown"
    s = str(x).strip().lower()
    if "pbs" in s or "phosphate" in s:
        return "phosphate"
    if "tris" in s:
        return "tris"
    if "acetate" in s:
        return "acetate"
    if "mops" in s:
        return "mops"
    if "water" in s:
        return "water"
    return "other"


def coarse_substrate(x):
    if pd.isna(x):
        return "unknown"
    s = str(x).strip().lower()
    if "h2o2" in s:
        return "h2o2"
    return "other"


BASE_NUMERIC = [
    "contain O", "contain N", "contain P", "contain S", "contain Si", "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]
BASE_CATEGORICAL = ["shape", "surface treatment", "dispersion medium", "Substrate"]

EXTRA_NUMERIC_LIGHT = [
    "size_log1p", "size_inv", "abs_pH_from_7", "size_x_pH", "size_x_temp", "pH_x_temp",
    "metal_component_count", "ratio_balance_abs", "valence_sum_weighted",
    "weighted_en", "weighted_radius_pm", "delta_en_main_minor", "delta_radius_main_minor"
]

ENG_NUMERIC = [
    "has_minor_metal", "has_third_metal", "metal_component_count",
    "main_ratio_filled", "minor_ratio_filled", "third_ratio_filled",
    "main_minus_minor_ratio", "ratio_balance_abs",
    "main_valence_filled", "minor_valence_filled", "third_valence_filled",
    "valence_sum_weighted", "valence_diff_main_minor",
    "main_period", "main_group", "main_en", "main_radius_pm", "main_is_noble", "main_is_rare_earth", "main_is_transition",
    "minor_period", "minor_group", "minor_en", "minor_radius_pm", "minor_is_noble", "minor_is_rare_earth", "minor_is_transition",
    "third_period", "third_group", "third_en", "third_radius_pm", "third_is_noble", "third_is_rare_earth", "third_is_transition",
    "delta_en_main_minor", "delta_radius_main_minor",
    "weighted_en", "weighted_radius_pm", "n_noble_metals", "n_rare_earth_metals",
    "size_missing", "size_log1p", "size_inv",
    "pH_missing", "pH_centered_7", "abs_pH_from_7",
    "temp_centered_25", "temp_centered_37",
    "size_x_pH", "size_x_temp", "pH_x_temp"
]
ENG_CATEGORICAL = [
    "shape_coarse", "surface_coarse", "medium_coarse", "substrate_coarse",
    "main_symbol", "minor_symbol", "third_symbol", "main_minor_pair"
]


def engineer_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()

    num_candidates = [
        "contain O", "contain N", "contain P", "contain S", "contain Si", "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
        "main metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "3rd metal ratio", "3rd metal type", "3rd metal valence",
        "size (nm)", "pH", "temperature/oC"
    ]
    for c in num_candidates:
        if c in df.columns:
            df[c] = _safe_num(df[c])

    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in df.columns:
            df[c] = df[c].fillna(0.0)

    for c in ["shape", "surface treatment", "dispersion medium", "Substrate", "nanozyme"]:
        if c in df.columns:
            df[c] = clean_text_series(df[c])

    if "shape" in df.columns:
        df["shape_coarse"] = df["shape"].map(coarse_shape)
    if "surface treatment" in df.columns:
        df["surface_coarse"] = df["surface treatment"].map(coarse_surface)
    if "dispersion medium" in df.columns:
        df["medium_coarse"] = df["dispersion medium"].map(coarse_medium)
    if "Substrate" in df.columns:
        df["substrate_coarse"] = df["Substrate"].map(coarse_substrate)

    main_num = df.get("main metal number", pd.Series(0, index=df.index)).fillna(0)
    minor_num = df.get("minor metal number", pd.Series(0, index=df.index)).fillna(0)
    third_num = df.get("3rd metal type", pd.Series(0, index=df.index)).fillna(0)

    main_ratio = df.get("main metal proportion", pd.Series(0, index=df.index)).fillna(0.0)
    minor_ratio = df.get("minor metal percentage", pd.Series(0, index=df.index)).fillna(0.0)
    third_ratio = df.get("3rd metal ratio", pd.Series(0, index=df.index)).fillna(0.0)

    main_val = df.get("main metal valence", pd.Series(0, index=df.index)).fillna(0.0)
    minor_val = df.get("minor metal valence", pd.Series(0, index=df.index)).fillna(0.0)
    third_val = df.get("3rd metal valence", pd.Series(0, index=df.index)).fillna(0.0)

    for prefix, series in [("main", main_num), ("minor", minor_num), ("third", third_num)]:
        df[f"{prefix}_symbol"] = map_atomic_symbol(series)
        df[f"{prefix}_period"] = map_atomic_property(series, "period")
        df[f"{prefix}_group"] = map_atomic_property(series, "group")
        df[f"{prefix}_en"] = map_atomic_property(series, "en")
        df[f"{prefix}_radius_pm"] = map_atomic_property(series, "radius_pm")
        df[f"{prefix}_is_noble"] = map_atomic_property(series, "noble")
        df[f"{prefix}_is_rare_earth"] = map_atomic_property(series, "rare_earth")
        df[f"{prefix}_is_transition"] = df[f"{prefix}_group"].isin(TRANSITION_GROUPS).astype(float)

    df["has_minor_metal"] = (minor_ratio > 0).astype(float)
    df["has_third_metal"] = (third_ratio > 0).astype(float)
    df["metal_component_count"] = 1.0 + df["has_minor_metal"] + df["has_third_metal"]

    df["main_ratio_filled"] = main_ratio
    df["minor_ratio_filled"] = minor_ratio
    df["third_ratio_filled"] = third_ratio
    df["main_minus_minor_ratio"] = main_ratio - minor_ratio
    df["ratio_balance_abs"] = (main_ratio - minor_ratio).abs()

    df["main_valence_filled"] = main_val
    df["minor_valence_filled"] = minor_val
    df["third_valence_filled"] = third_val
    df["valence_sum_weighted"] = (
        main_ratio * main_val + minor_ratio * minor_val + third_ratio * third_val
    ) / (main_ratio + minor_ratio + third_ratio + 1e-9)
    df["valence_diff_main_minor"] = main_val - minor_val

    df["delta_en_main_minor"] = (df["main_en"] - df["minor_en"]).abs()
    df["delta_radius_main_minor"] = (df["main_radius_pm"] - df["minor_radius_pm"]).abs()
    df["weighted_en"] = (
        main_ratio * df["main_en"] + minor_ratio * df["minor_en"] + third_ratio * df["third_en"]
    ) / (main_ratio + minor_ratio + third_ratio + 1e-9)
    df["weighted_radius_pm"] = (
        main_ratio * df["main_radius_pm"] + minor_ratio * df["minor_radius_pm"] + third_ratio * df["third_radius_pm"]
    ) / (main_ratio + minor_ratio + third_ratio + 1e-9)
    df["n_noble_metals"] = df["main_is_noble"] + df["minor_is_noble"] + df["third_is_noble"]
    df["n_rare_earth_metals"] = df["main_is_rare_earth"] + df["minor_is_rare_earth"] + df["third_is_rare_earth"]

    size = _safe_num(df.get("size (nm)", pd.Series(np.nan, index=df.index)))
    ph = _safe_num(df.get("pH", pd.Series(np.nan, index=df.index)))
    temp = _safe_num(df.get("temperature/oC", pd.Series(np.nan, index=df.index)))

    df["size_missing"] = size.isna().astype(float)
    df["size_log1p"] = np.log1p(size.clip(lower=0))
    df["size_inv"] = 1.0 / (size.clip(lower=1e-6) + 1.0)
    df["pH_missing"] = ph.isna().astype(float)
    df["pH_centered_7"] = ph - 7.0
    df["abs_pH_from_7"] = (ph - 7.0).abs()
    df["temp_centered_25"] = temp - 25.0
    df["temp_centered_37"] = temp - 37.0
    df["size_x_pH"] = size * ph
    df["size_x_temp"] = size * temp
    df["pH_x_temp"] = ph * temp

    df["main_minor_pair"] = df["main_symbol"].astype(str) + "_" + df["minor_symbol"].astype(str)

    return df


def get_feature_set(df_eng: pd.DataFrame, mode: str):
    if mode == "baseline":
        num_cols = [c for c in BASE_NUMERIC if c in df_eng.columns]
        cat_cols = [c for c in BASE_CATEGORICAL if c in df_eng.columns]

    elif mode == "baseline_no_substrate":
        num_cols = [c for c in BASE_NUMERIC if c in df_eng.columns]
        cat_cols = [c for c in ["shape", "surface treatment", "dispersion medium"] if c in df_eng.columns]

    elif mode == "baseline_slim":
        num_cols = [c for c in BASE_NUMERIC if c in df_eng.columns]
        cat_cols = [c for c in ["shape_coarse", "medium_coarse"] if c in df_eng.columns]

    elif mode == "enhanced_pruned":
        drop_low_info_num = {"contain Se", "contain B", "contain Cl", "contain I", "3rd metal valence", "3rd metal type"}
        drop_low_info_cat = {"Substrate"}
        num_cols = [c for c in (BASE_NUMERIC + ENG_NUMERIC) if c in df_eng.columns and c not in drop_low_info_num]
        cat_cols = [c for c in (BASE_CATEGORICAL + ENG_CATEGORICAL) if c in df_eng.columns and c not in drop_low_info_cat]

    elif mode == "enhanced_km_light":
        km_light_num = [
            "main metal proportion", "main metal number", "main metal valence",
            "minor metal percentage", "minor metal number", "minor metal valence",
            "size (nm)", "pH", "temperature/oC",
            "size_log1p", "abs_pH_from_7", "weighted_en", "weighted_radius_pm",
            "valence_sum_weighted", "delta_en_main_minor", "delta_radius_main_minor",
            "metal_component_count", "ratio_balance_abs"
        ]
        km_light_cat = ["shape", "shape_coarse", "main_minor_pair"]
        num_cols = [c for c in km_light_num if c in df_eng.columns]
        cat_cols = [c for c in km_light_cat if c in df_eng.columns]

    else:
        raise ValueError(f"Unknown feature mode: {mode}")

    keep_num = []
    for c in num_cols:
        s = pd.to_numeric(df_eng[c], errors="coerce")
        if s.nunique(dropna=True) > 1:
            keep_num.append(c)

    keep_cat = []
    for c in cat_cols:
        s = df_eng[c].astype(str)
        if s.nunique(dropna=True) > 1:
            keep_cat.append(c)

    return keep_num + keep_cat, keep_num, keep_cat


# =========================
# 3) Preprocess components
# =========================
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_freq=2, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        X.columns = [str(c) for c in X.columns]
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        X.columns = [str(c) for c in X.columns]
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


class ToDense(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if hasattr(X, "toarray"):
            return X.toarray()
        return np.asarray(X)


class SelectKBestSafe(BaseEstimator, TransformerMixin):
    def __init__(self, k=None):
        self.k = k
        self.selector_ = None

    def fit(self, X, y):
        if self.k is None:
            self.selector_ = None
            return self
        n_feat = X.shape[1]
        k_eff = int(min(int(self.k), int(n_feat)))
        if k_eff <= 0:
            self.selector_ = None
            return self
        self.selector_ = SelectKBest(score_func=f_regression, k=k_eff)
        self.selector_.fit(X, y)
        return self

    def transform(self, X):
        if self.selector_ is None:
            return X
        return self.selector_.transform(X)


class TruncatedSVDSafe(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=None, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.svd_ = None

    def fit(self, X, y=None):
        if self.n_components is None:
            self.svd_ = None
            return self
        n_feat = X.shape[1]
        if n_feat <= 1:
            self.svd_ = None
            return self
        n_eff = min(int(self.n_components), max(1, n_feat - 1))
        self.svd_ = TruncatedSVD(n_components=n_eff, random_state=self.random_state)
        self.svd_.fit(X)
        return self

    def transform(self, X):
        if self.svd_ is None:
            return X
        return self.svd_.transform(X)


def build_preprocess(numeric_cols, categorical_cols, min_freq_cat=2, sparse=True):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=sparse)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=sparse)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=min_freq_cat)),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop",
    )


# =========================
# 4) Models + pipeline
# =========================
def instantiate_model(model_name, params):
    params = dict(params)
    use_y_scaler = bool(params.pop("__use_y_scaler__", False))
    params.pop("__sample_weight_mode__", None)

    if model_name == "SVR_RBF":
        base_model = SVR(kernel="rbf", **params)
    elif model_name == "NuSVR_RBF":
        base_model = NuSVR(kernel="rbf", **params)
    elif model_name == "KernelRidge_RBF":
        base_model = KernelRidge(kernel="rbf", **params)
    elif model_name == "KNN_distance":
        base_model = KNeighborsRegressor(weights="distance", **params)
    else:
        raise ValueError(model_name)

    if use_y_scaler:
        return TransformedTargetRegressor(regressor=base_model, transformer=StandardScaler())
    return base_model


def build_pipeline_for_config(numeric_cols, categorical_cols,
                              min_freq_cat, var_thr, k_best, svd_dim,
                              scaler_choice,
                              model_name, model_params):
    params = dict(model_params)
    sample_weight_mode = params.get("__sample_weight_mode__", "none")

    preprocess = build_preprocess(numeric_cols, categorical_cols, min_freq_cat=min_freq_cat, sparse=True)

    scaler_cls = RobustScaler if scaler_choice == "robust" else StandardScaler
    scaler_with_mean = True

    steps = []
    steps.append(("preprocess", preprocess))
    steps.append(("var", VarianceThreshold(threshold=float(var_thr))))

    if k_best is not None:
        steps.append(("kbest", SelectKBestSafe(k=int(k_best))))

    if svd_dim is not None:
        steps.append(("svd", TruncatedSVDSafe(n_components=int(svd_dim), random_state=RANDOM_SEED)))
        steps.append(("scaler", scaler_cls(with_centering=True) if scaler_choice == "robust" else scaler_cls(with_mean=True)))
    else:
        steps.append(("todense", ToDense()))
        steps.append(("scaler", scaler_cls(with_centering=True) if scaler_choice == "robust" else scaler_cls(with_mean=True)))

    steps.append(("model", instantiate_model(model_name, params)))
    pipe = Pipeline(steps)
    pipe._sample_weight_mode = sample_weight_mode
    pipe._scaler_choice = scaler_choice
    return pipe


# =========================
# 5) OOF / search
# =========================
def groupkfold_oof_predict(X_raw, y_log, groups, pipe_builder):
    uniq_groups = np.unique(groups)
    n_folds = min(MAX_FOLDS, len(uniq_groups))
    if n_folds < 3:
        return None

    gkf = GroupKFold(n_splits=n_folds)
    oof_pred = np.full(len(y_log), np.nan, dtype=float)

    for tr, te in gkf.split(X_raw, y_log, groups=groups):
        Xtr, Xte = X_raw.iloc[tr].copy(), X_raw.iloc[te].copy()
        ytr = y_log[tr]

        pipe = pipe_builder()
        fit_params = {}
        sw_mode = getattr(pipe, "_sample_weight_mode", "none")
        if sw_mode == "inv_group":
            sw = make_inverse_group_weights(groups[tr])
            fit_params["model__sample_weight"] = sw

        try:
            pipe.fit(Xtr, ytr, **fit_params)
        except TypeError:
            # Some models (e.g. KNN) do not accept sample_weight
            if "model__sample_weight" in fit_params:
                continue
            raise

        oof_pred[te] = np.asarray(pipe.predict(Xte)).reshape(-1)

    ok = np.isfinite(oof_pred)
    if ok.sum() == 0:
        return None
    return oof_pred, ok, n_folds


def build_vmax_search_space():
    rows = []

    preprocs = [
        {"feature_mode": "baseline",              "min_freq_cat": 2, "var_thr": 0.0, "k_best": 10, "svd_dim": 8, "scaler_choice": "standard"},
        {"feature_mode": "baseline",              "min_freq_cat": 2, "var_thr": 0.0, "k_best": 10, "svd_dim": 8, "scaler_choice": "robust"},
        {"feature_mode": "baseline",              "min_freq_cat": 2, "var_thr": 0.0, "k_best": 12, "svd_dim": 8, "scaler_choice": "standard"},
        {"feature_mode": "baseline_no_substrate", "min_freq_cat": 2, "var_thr": 0.0, "k_best": 10, "svd_dim": 8, "scaler_choice": "robust"},
        {"feature_mode": "baseline_slim",         "min_freq_cat": 2, "var_thr": 0.0, "k_best": 8,  "svd_dim": 6, "scaler_choice": "robust"},
    ]

    svr_grid = [
        {"model": "SVR_RBF", "model_params": {"C": C, "gamma": g, "epsilon": e, "__use_y_scaler__": True, "__sample_weight_mode__": sw}}
        for C in [10, 15, 20, 25]
        for g in [0.06, 0.08, 0.10, 0.12]
        for e in [0.05, 0.08, 0.10]
        for sw in ["none", "inv_group"]
    ]

    nusvr_grid = [
        {"model": "NuSVR_RBF", "model_params": {"C": C, "gamma": g, "nu": nu, "__use_y_scaler__": True, "__sample_weight_mode__": sw}}
        for C in [10, 15, 20]
        for g in [0.06, 0.08, 0.10]
        for nu in [0.35, 0.50]
        for sw in ["none", "inv_group"]
    ]

    krr_grid = [
        {"model": "KernelRidge_RBF", "model_params": {"alpha": a, "gamma": g, "__use_y_scaler__": True, "__sample_weight_mode__": sw}}
        for a in [0.005, 0.01, 0.02]
        for g in [0.30, 0.50, 0.80]
        for sw in ["none", "inv_group"]
    ]

    knn_grid = [
        {"model": "KNN_distance", "model_params": {"n_neighbors": k, "p": p, "__use_y_scaler__": False, "__sample_weight_mode__": "none"}}
        for k in [3, 5, 7]
        for p in [1, 2]
    ]

    for pp in preprocs:
        for mg in (svr_grid + nusvr_grid + krr_grid + knn_grid):
            rows.append({**pp, **mg})
    return rows


def build_km_search_space():
    rows = []
    preprocs = [
        {"feature_mode": "enhanced_pruned", "min_freq_cat": 2, "var_thr": 0.0, "k_best": 10, "svd_dim": 8, "scaler_choice": "standard"},
    ]
    model_rows = [
        {"model": "KernelRidge_RBF", "model_params": {"alpha": a, "gamma": g, "__use_y_scaler__": True, "__sample_weight_mode__": "none"}}
        for a in [0.005, 0.01, 0.02]
        for g in [0.30, 0.50, 0.80]
    ] + [
        {"model": "SVR_RBF", "model_params": {"C": C, "gamma": g, "epsilon": e, "__use_y_scaler__": True, "__sample_weight_mode__": "none"}}
        for C in [10, 15]
        for g in [0.08, 0.10]
        for e in [0.08, 0.10]
    ]
    for pp in preprocs:
        for mr in model_rows:
            rows.append({**pp, **mr})
    return rows


def build_component_from_row(row, numeric_cols, categorical_cols):
    model_params = json.loads(row["model_params"]) if isinstance(row["model_params"], str) else dict(row["model_params"])

    def builder():
        return build_pipeline_for_config(
            numeric_cols, categorical_cols,
            min_freq_cat=int(row["min_freq_cat"]),
            var_thr=float(row["var_thr"]),
            k_best=None if pd.isna(row["k_best_req"]) else int(row["k_best_req"]),
            svd_dim=None if pd.isna(row["svd_dim_req"]) else int(row["svd_dim_req"]),
            scaler_choice=row.get("scaler_choice", "standard"),
            model_name=row["model"],
            model_params=model_params,
        )

    return builder, model_params


def fit_full_with_possible_weights(pipe, X_raw, y_log, groups):
    fit_params = {}
    sw_mode = getattr(pipe, "_sample_weight_mode", "none")
    if sw_mode == "inv_group":
        fit_params["model__sample_weight"] = make_inverse_group_weights(groups)
    pipe.fit(X_raw, y_log, **fit_params)
    return pipe


def run_vmax_ensemble_scan(rows_df, best_oof_by_model, y_log):
    if not TRY_VMAX_ENSEMBLE:
        return []

    available = [m for m in ["SVR_RBF", "NuSVR_RBF", "KernelRidge_RBF", "KNN_distance"] if m in best_oof_by_model]
    if len(available) < 2:
        return []

    ensemble_rows = []
    for i in range(len(available)):
        for j in range(i + 1, len(available)):
            m1, m2 = available[i], available[j]
            pred1 = best_oof_by_model[m1]["oof_pred"]
            pred2 = best_oof_by_model[m2]["oof_pred"]
            row1 = best_oof_by_model[m1]["row"]
            row2 = best_oof_by_model[m2]["row"]
            for w1 in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
                pred_ens = w1 * pred1 + (1.0 - w1) * pred2
                m = eval_metrics(y_log, pred_ens)
                ensemble_rows.append({
                    "target": "logVmax",
                    "model": f"Ensemble_{m1}_{m2}",
                    "feature_mode": f"{row1['feature_mode']} + {row2['feature_mode']}",
                    "min_freq_cat": np.nan,
                    "var_thr": np.nan,
                    "k_best_req": np.nan,
                    "svd_dim_req": np.nan,
                    "scaler_choice": np.nan,
                    "model_params": json.dumps({"weight_first": w1, "model_first": m1, "model_second": m2}, ensure_ascii=False),
                    "n_samples": int(row1["n_samples"]),
                    "n_groups": int(row1["n_groups"]),
                    "singleton_groups": int(row1["singleton_groups"]),
                    "n_folds": int(row1["n_folds"]),
                    **m,
                    "is_ensemble": True,
                })
    return ensemble_rows


# =========================
# 6) Main target runner
# =========================
def run_target(target_name, df_target, y_col):
    t0_target = time.time()
    df_target = df_target.copy()

    y_num = pd.to_numeric(df_target[y_col], errors="coerce")
    df_target = df_target[y_num.notna() & (y_num > 0)].copy()
    if len(df_target) < 10:
        print(f"[Skip] {target_name}: too few samples: {len(df_target)}")
        return None

    y_log = np.log10(pd.to_numeric(df_target[y_col], errors="coerce").astype(float).values)
    groups = make_groups(df_target, COL_DOI)
    n_groups = int(len(np.unique(groups)))
    singleton_groups = int(pd.Series(groups).value_counts().eq(1).sum())

    print(f"\n[{target_name}] n_samples={len(df_target)} | n_groups={n_groups} | singleton_groups={singleton_groups}")

    df_eng = engineer_features(df_target)
    feature_modes_to_cache = ["baseline", "baseline_no_substrate", "baseline_slim", "enhanced_pruned", "enhanced_km_light"]
    feature_cache = {mode: get_feature_set(df_eng, mode) for mode in feature_modes_to_cache}

    search_space = build_vmax_search_space() if target_name == "logVmax" else build_km_search_space()
    print(f"[{target_name}] search space ~= {len(search_space)} configs (before CV folds)")

    rows = []
    model_time = {}
    model_best = {}
    best_oof_by_model = {}

    for cfg in search_space:
        model_name = cfg["model"]
        feature_mode = cfg["feature_mode"]
        min_freq_cat = cfg["min_freq_cat"]
        var_thr = cfg["var_thr"]
        k_best = cfg["k_best"]
        svd_dim = cfg["svd_dim"]
        scaler_choice = cfg["scaler_choice"]
        model_params = cfg["model_params"]

        if model_name not in model_time:
            model_time[model_name] = 0.0
            model_best[model_name] = None

        feature_cols, numeric_cols, categorical_cols = feature_cache[feature_mode]
        if len(feature_cols) == 0:
            continue
        X_raw = df_eng[feature_cols].copy()

        t0 = time.time()

        def builder():
            return build_pipeline_for_config(
                numeric_cols, categorical_cols,
                min_freq_cat=min_freq_cat,
                var_thr=var_thr,
                k_best=k_best,
                svd_dim=svd_dim,
                scaler_choice=scaler_choice,
                model_name=model_name,
                model_params=model_params,
            )

        out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
        model_time[model_name] += time.time() - t0
        if out is None:
            continue

        pred, ok, n_folds = out
        m = eval_metrics(y_log[ok], pred[ok])

        row = {
            "target": target_name,
            "model": model_name,
            "feature_mode": feature_mode,
            "min_freq_cat": int(min_freq_cat),
            "var_thr": float(var_thr),
            "k_best_req": None if k_best is None else int(k_best),
            "svd_dim_req": None if svd_dim is None else int(svd_dim),
            "scaler_choice": scaler_choice,
            "model_params": json.dumps(model_params, ensure_ascii=False),
            "n_samples": int(len(df_target)),
            "n_groups": n_groups,
            "singleton_groups": singleton_groups,
            "n_folds": int(n_folds),
            **m,
            "is_ensemble": False,
        }
        rows.append(row)

        best_for_model = model_best[model_name]
        if (best_for_model is None) or ((row["r2_log_oof"], -row["rmse_log_oof"]) > (best_for_model["r2_log_oof"], -best_for_model["rmse_log_oof"])):
            model_best[model_name] = row
            best_oof_by_model[model_name] = {
                "row": row,
                "oof_pred": pred.copy(),
                "feature_cols": feature_cols,
                "numeric_cols": numeric_cols,
                "categorical_cols": categorical_cols,
            }

    rows_df = pd.DataFrame(rows)
    if rows_df.empty:
        raise RuntimeError(f"{target_name}: no valid results")

    if target_name == "logVmax":
        ens_rows = run_vmax_ensemble_scan(rows_df, best_oof_by_model, y_log)
        if len(ens_rows) > 0:
            rows_df = pd.concat([rows_df, pd.DataFrame(ens_rows)], axis=0, ignore_index=True)

    rows_df.to_csv(OUT_DIR / f"{target_name}_search_summary.csv", index=False, encoding="utf-8-sig")

    # Per-model print
    for mname in sorted(model_time.keys()):
        best_row = None
        if mname in rows_df["model"].values:
            tmp = rows_df[rows_df["model"] == mname].sort_values(["r2_log_oof", "rmse_log_oof"], ascending=[False, True])
            if not tmp.empty:
                best_row = tmp.iloc[0]
        if best_row is not None:
            print(
                f"  - {mname:<16} | best R2={best_row['r2_log_oof']:.4f} | "
                f"RMSE={best_row['rmse_log_oof']:.4f} | feature={best_row['feature_mode']} | "
                f"k={best_row['k_best_req']} | svd={best_row['svd_dim_req']} | time={model_time.get(mname, 0.0):.1f}s"
            )
    if target_name == "logVmax" and "Ensemble_" in "".join(rows_df["model"].astype(str).tolist()):
        tmp = rows_df[rows_df["is_ensemble"] == True].sort_values(["r2_log_oof", "rmse_log_oof"], ascending=[False, True])
        if not tmp.empty:
            best_row = tmp.iloc[0]
            print(
                f"  - {'Ensemble':<16} | best R2={best_row['r2_log_oof']:.4f} | "
                f"RMSE={best_row['rmse_log_oof']:.4f} | feature={best_row['feature_mode']} | "
                f"k={best_row['k_best_req']} | svd={best_row['svd_dim_req']} | time~0.0s"
            )

    best = rows_df.sort_values(["r2_log_oof", "rmse_log_oof"], ascending=[False, True]).iloc[0].to_dict()
    best_is_ensemble = bool(best.get("is_ensemble", False))

    if best_is_ensemble:
        params_ens = json.loads(best["model_params"])
        m1 = params_ens["model_first"]
        m2 = params_ens["model_second"]
        w1 = float(params_ens["weight_first"])

        oof = w1 * best_oof_by_model[m1]["oof_pred"] + (1.0 - w1) * best_oof_by_model[m2]["oof_pred"]
        ok = np.isfinite(oof)
        n_folds = int(best_oof_by_model[m1]["row"]["n_folds"])
        oof_metrics = eval_metrics(y_log[ok], oof[ok])

        components = []
        for comp_name in [m1, m2]:
            comp_row = best_oof_by_model[comp_name]["row"]
            feature_cols, numeric_cols, categorical_cols = feature_cache[comp_row["feature_mode"]]
            X_raw_comp = df_eng[feature_cols].copy()
            builder, comp_params = build_component_from_row(comp_row, numeric_cols, categorical_cols)
            pipe = builder()
            pipe = fit_full_with_possible_weights(pipe, X_raw_comp, y_log, groups)
            components.append({
                "name": comp_name,
                "feature_mode": comp_row["feature_mode"],
                "feature_cols": feature_cols,
                "numeric_cols": numeric_cols,
                "categorical_cols": categorical_cols,
                "pipeline": pipe,
                "model_params": comp_params,
                "preprocess_params": {
                    "min_freq_cat": int(comp_row["min_freq_cat"]),
                    "var_thr": float(comp_row["var_thr"]),
                    "k_best_req": None if pd.isna(comp_row["k_best_req"]) else int(comp_row["k_best_req"]),
                    "svd_dim_req": None if pd.isna(comp_row["svd_dim_req"]) else int(comp_row["svd_dim_req"]),
                    "scaler_choice": comp_row.get("scaler_choice", "standard"),
                }
            })

        final_model_obj = {
            "type": "weighted_ensemble",
            "weight_first": w1,
            "components": components
        }
        pd.DataFrame({
            "row_index": df_target.index.values,
            "doi_group": groups,
            "y_log_true": y_log,
            "y_log_oof_pred": oof
        }).to_csv(OUT_DIR / f"{target_name}_oof_predictions_groupkfold.csv", index=False, encoding="utf-8-sig")

    else:
        best_model = best["model"]
        best_feature_mode = best["feature_mode"]
        best_min_freq = int(best["min_freq_cat"])
        best_var_thr = float(best["var_thr"])
        best_kbest = None if pd.isna(best["k_best_req"]) else int(best["k_best_req"])
        best_svd = None if pd.isna(best["svd_dim_req"]) else int(best["svd_dim_req"])
        best_scaler = best.get("scaler_choice", "standard")
        best_model_params = json.loads(best["model_params"])

        feature_cols, numeric_cols, categorical_cols = feature_cache[best_feature_mode]
        X_raw_best = df_eng[feature_cols].copy()

        def best_builder():
            return build_pipeline_for_config(
                numeric_cols, categorical_cols,
                min_freq_cat=best_min_freq,
                var_thr=best_var_thr,
                k_best=best_kbest,
                svd_dim=best_svd,
                scaler_choice=best_scaler,
                model_name=best_model,
                model_params=best_model_params,
            )

        oof, ok, n_folds = groupkfold_oof_predict(X_raw_best, y_log, groups, best_builder)
        oof_metrics = eval_metrics(y_log[ok], oof[ok])

        pd.DataFrame({
            "row_index": df_target.index.values,
            "doi_group": groups,
            "y_log_true": y_log,
            "y_log_oof_pred": oof
        }).to_csv(OUT_DIR / f"{target_name}_oof_predictions_groupkfold.csv", index=False, encoding="utf-8-sig")

        final_pipe = best_builder()
        final_pipe = fit_full_with_possible_weights(final_pipe, X_raw_best, y_log, groups)

        final_model_obj = {
            "type": "single_model",
            "target": target_name,
            "feature_mode": best_feature_mode,
            "feature_cols": feature_cols,
            "numeric_cols": numeric_cols,
            "categorical_cols": categorical_cols,
            "pipeline": final_pipe,
        }

    best_config = {
        "target": target_name,
        "y_col": y_col,
        "target_transform": "log10",
        "split": "GroupKFold by DOI",
        "data_stats": {
            "n_samples": int(len(df_target)),
            "n_groups": int(n_groups),
            "singleton_groups": int(singleton_groups),
            "n_folds": int(n_folds),
        },
        "best": {
            "model": best["model"],
            "feature_mode": best["feature_mode"],
            "preprocess_params": {
                "min_freq_cat": None if pd.isna(best["min_freq_cat"]) else int(best["min_freq_cat"]),
                "var_thr": None if pd.isna(best["var_thr"]) else float(best["var_thr"]),
                "k_best_req": None if pd.isna(best["k_best_req"]) else int(best["k_best_req"]),
                "svd_dim_req": None if pd.isna(best["svd_dim_req"]) else int(best["svd_dim_req"]),
                "scaler_choice": None if pd.isna(best.get("scaler_choice", np.nan)) else best.get("scaler_choice"),
            },
            "model_params": json.loads(best["model_params"]),
            "metrics_oof": {
                "r2_log_oof": float(oof_metrics["r2_log_oof"]),
                "mae_log_oof": float(oof_metrics["mae_log_oof"]),
                "rmse_log_oof": float(oof_metrics["rmse_log_oof"]),
                "pearson_r_oof": float(oof_metrics["pearson_r_oof"]) if pd.notna(oof_metrics["pearson_r_oof"]) else None,
                "pearson_r2_oof": float(oof_metrics["pearson_r2_oof"]) if pd.notna(oof_metrics["pearson_r2_oof"]) else None,
            }
        },
        "runtime_sec": round(time.time() - t0_target, 3),
    }

    with open(OUT_DIR / f"{target_name}_best_config.json", "w", encoding="utf-8") as f:
        json.dump(best_config, f, ensure_ascii=False, indent=2)

    joblib.dump({
        "target": target_name,
        "model_object": final_model_obj,
        "best_config": best_config
    }, OUT_DIR / f"{target_name}_best_model_refit_full.joblib")

    print("\n" + "=" * 72)
    print(f"[BEST] {target_name} | model={best['model']} | feature={best['feature_mode']}")
    print("preprocess:", best_config["best"]["preprocess_params"])
    print("model_params:", best_config["best"]["model_params"])
    print("metrics_oof:", best_config["best"]["metrics_oof"])
    print(f"runtime_sec: {best_config['runtime_sec']}")
    print("=" * 72)

    return best_config


# =========================
# 7) Main
# =========================
def main():
    t0 = time.time()

    df = pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME)
    df = normalize_columns(df)

    if COL_DOI not in df.columns:
        raise KeyError(f"DOI column not found: {COL_DOI}")
    if COL_KM not in df.columns:
        raise KeyError(f"Km column not found: {COL_KM}")
    if COL_VMAX not in df.columns:
        raise KeyError(f"Vmax column not found: {COL_VMAX}")

    # save used data
    df_km = df.copy()
    km_num = pd.to_numeric(df_km[COL_KM], errors="coerce")
    df_km = df_km[km_num.notna() & (km_num > 0)].copy()
    df_km.to_csv(OUT_DIR / "used_data_logKm.csv", index=False, encoding="utf-8-sig")

    df_v = df.copy()
    v_num = pd.to_numeric(df_v[COL_VMAX], errors="coerce")
    df_v = df_v[v_num.notna() & (v_num > 0)].copy()
    df_v.to_csv(OUT_DIR / "used_data_logVmax.csv", index=False, encoding="utf-8-sig")

    # export feature info for modes
    df_eng_all = engineer_features(df.copy())
    for mode in ["baseline", "baseline_no_substrate", "baseline_slim", "enhanced_pruned", "enhanced_km_light"]:
        feature_cols, numeric_cols, categorical_cols = get_feature_set(df_eng_all, mode)
        pd.DataFrame({
            "feature": feature_cols,
            "type": ["numeric" if c in numeric_cols else "categorical" for c in feature_cols]
        }).to_csv(OUT_DIR / f"feature_info_{mode}.csv", index=False, encoding="utf-8-sig")

    if RUN_LOGKM:
        run_target("logKm", df_km, y_col=COL_KM)

    if RUN_LOGVMAX:
        run_target("logVmax", df_v, y_col=COL_VMAX)

    print("\nSaved to:", OUT_DIR)
    print(f"Total runtime: {round(time.time() - t0, 2)} sec")


if __name__ == "__main__":
    main()



[logKm] n_samples=61 | n_groups=39 | singleton_groups=26
[logKm] search space ~= 17 configs (before CV folds)
  - KernelRidge_RBF  | best R2=0.2843 | RMSE=1.2001 | feature=enhanced_pruned | k=10 | svd=8 | time=0.9s
  - SVR_RBF          | best R2=-0.2496 | RMSE=1.5858 | feature=enhanced_pruned | k=10 | svd=8 | time=0.8s

[BEST] logKm | model=KernelRidge_RBF | feature=enhanced_pruned
preprocess: {'min_freq_cat': 2, 'var_thr': 0.0, 'k_best_req': 10, 'svd_dim_req': 8, 'scaler_choice': 'standard'}
model_params: {'alpha': 0.005, 'gamma': 0.3, '__use_y_scaler__': True, '__sample_weight_mode__': 'none'}
metrics_oof: {'r2_log_oof': 0.28430709037621893, 'mae_log_oof': 0.9891005311818448, 'rmse_log_oof': 1.2001466264658507, 'pearson_r_oof': 0.5751488291451686, 'pearson_r2_oof': 0.3307961756670584}
runtime_sec: 1.939

[logVmax] n_samples=57 | n_groups=35 | singleton_groups=22
[logVmax] search space ~= 780 configs (before CV folds)
  - KNN_distance     | best R2=0.0821 | RMSE=1.9289 | feature=base

In [9]:
# -*- coding: utf-8 -*-
"""
CAT regression (focus on logVmax) with DOI-GroupKFold OOF.

Goal of this version:
1) Keep the original Excel untouched; all feature/sample filtering is in-memory.
2) Explore *feature screening* + *sample screening* + a few additional reasonable models
   without blowing up compute.
3) Keep the evaluation protocol fixed: DOI-GroupKFold OOF for both search and final report.

Compared with previous versions, this script adds:
- Sample modes (full / complete_core / common_buffer / assay_core)
- Feature modes (baseline / baseline_slim / metal_env / enhanced_pruned)
- Feature selectors (f_regression / mutual_info / tree_embed / none)
- More model families for Vmax, especially tree-based models:
    SVR_RBF / KernelRidge_RBF / RandomForest / ExtraTrees / HistGBR / KNN_distance
- RobustScaler option

Notes:
- Because CAT Vmax data are small and heterogeneous across DOI groups, a very large grid is
  not recommended. This script uses a curated search space instead of a huge Cartesian product.
- By default, logVmax is the main task. logKm is kept lighter.
"""

import json
import math
import re
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.feature_selection import VarianceThreshold, SelectKBest, SelectFromModel, f_regression, mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.svm import SVR

warnings.filterwarnings("ignore")

# =========================
# 0) Paths & config
# =========================
CAT_PATH = Path(r"C:\Users\86158\Desktop\dataiscat.xlsx")  # <-- 改成你的输入文件
SHEET_NAME = 0

OUT_DIR = CAT_PATH.parent / "cat_km_vmax_reg_doi_multistage_out_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_DOI = "data reference doi"
COL_KM = "Km/mM"
COL_VMAX = "Vmax/μM s-1"

RANDOM_SEED = 42
MAX_FOLDS = 5
MIN_SAMPLES_FOR_RUN = 14
MIN_GROUPS_FOR_RUN = 8

# 为了控制算量，默认主攻 Vmax；需要的话把 RUN_LOGKM 改成 True
RUN_LOGKM = True
RUN_LOGVMAX = True


# =========================
# 1) Utils
# =========================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def coarse_shape(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if "sphere" in x:
        return "sphere"
    if "particle" in x:
        return "particle"
    if "cluster" in x:
        return "cluster"
    if "sheet" in x:
        return "sheet"
    if "rod" in x:
        return "rod"
    if "cube" in x:
        return "cube"
    if "flower" in x:
        return "flower"
    return "other"


def coarse_surface(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if x in {"none", "na", "nan"}:
        return "none"
    if any(k in x for k in ["peg", "pvp", "paa", "pamam", "dspe", "β-cd", "beta-cd", "mpa", "aa"]):
        return "poly_small"
    if "sio2" in x or "silica" in x:
        return "silica"
    if any(k in x for k in ["apo", "ferritin", "histidine", "gsh", "silk", "amp", "nta"]):
        return "bio_ligand"
    return "other"


def coarse_medium(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if x in {"pbs", "phosphate buffer", "pb", "phosphate-buffered saline"}:
        return "phosphate"
    if "tris" in x:
        return "tris"
    if "water" in x:
        return "water"
    if "acetate" in x or "hac-naac" in x:
        return "acetate"
    if "mops" in x:
        return "mops"
    return "other"


# Atomic properties for metal-number feature engineering
# columns: Z -> (period, group, pauling_en, radius_pm, noble_flag, rare_earth_flag, transition_flag)
ATOMIC_PROPS = {
    12: (3, 2, 1.31, 160, 0, 0, 0),   # Mg
    20: (4, 2, 1.00, 197, 0, 0, 0),   # Ca
    23: (4, 5, 1.63, 134, 0, 0, 1),   # V
    25: (4, 7, 1.55, 127, 0, 0, 1),   # Mn
    26: (4, 8, 1.83, 126, 0, 0, 1),   # Fe
    27: (4, 9, 1.88, 125, 0, 0, 1),   # Co
    28: (4,10, 1.91, 124, 0, 0, 1),   # Ni
    29: (4,11, 1.90, 128, 0, 0, 1),   # Cu
    40: (5, 4, 1.33, 160, 0, 0, 1),   # Zr
    42: (5, 6, 2.16, 139, 0, 0, 1),   # Mo
    44: (5, 8, 2.20, 134, 0, 0, 1),   # Ru
    45: (5, 9, 2.28, 134, 0, 0, 1),   # Rh
    46: (5,10, 2.20, 137, 0, 0, 1),   # Pd
    47: (5,11, 1.93, 144, 0, 0, 1),   # Ag
    58: (6, 3, 1.12, 182, 0, 1, 0),   # Ce
    68: (6,17, 1.24, 176, 0, 1, 0),   # Er
    77: (6, 9, 2.20, 136, 1, 0, 1),   # Ir
    78: (6,10, 2.28, 139, 1, 0, 1),   # Pt
    79: (6,11, 2.54, 144, 1, 0, 1),   # Au
}


def atomic_prop(z, idx, default=np.nan):
    try:
        z = int(z)
    except Exception:
        return default
    if z <= 0:
        return default
    return ATOMIC_PROPS.get(z, (default, default, default, default, default, default, default))[idx]


# =========================
# 2) Preprocessing transformers
# =========================
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_freq=2, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


class ToDense(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if hasattr(X, "toarray"):
            return X.toarray()
        return np.asarray(X)


class SelectKBestSafe(BaseEstimator, TransformerMixin):
    def __init__(self, mode=None, k=None, random_state=42):
        self.mode = mode
        self.k = k
        self.random_state = random_state
        self.selector_ = None

    def fit(self, X, y):
        X = np.asarray(X)
        n_feat = X.shape[1]
        if self.mode is None:
            self.selector_ = None
            return self

        if self.mode == "f_regression":
            k_eff = n_feat if self.k is None else max(1, min(int(self.k), n_feat))
            self.selector_ = SelectKBest(score_func=f_regression, k=k_eff)
            self.selector_.fit(X, y)
            return self

        if self.mode == "mutual_info":
            k_eff = n_feat if self.k is None else max(1, min(int(self.k), n_feat))
            self.selector_ = SelectKBest(
                score_func=lambda a, b: mutual_info_regression(a, b, random_state=self.random_state),
                k=k_eff,
            )
            self.selector_.fit(X, y)
            return self

        if self.mode == "tree_embed":
            base = ExtraTreesRegressor(
                n_estimators=250,
                random_state=self.random_state,
                max_depth=None,
                min_samples_leaf=1,
                n_jobs=-1,
            )
            max_features = None
            if self.k is not None:
                max_features = max(1, min(int(self.k), n_feat))
            self.selector_ = SelectFromModel(
                estimator=base,
                threshold="median",
                max_features=max_features,
            )
            self.selector_.fit(X, y)
            return self

        raise ValueError(f"Unknown selector mode: {self.mode}")

    def transform(self, X):
        if self.selector_ is None:
            return X
        return self.selector_.transform(np.asarray(X))


class TruncatedSVDSafe(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=None, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.svd_ = None

    def fit(self, X, y=None):
        X = np.asarray(X)
        if self.n_components is None or X.shape[1] <= 1:
            self.svd_ = None
            return self
        n_eff = min(int(self.n_components), max(1, X.shape[1] - 1))
        self.svd_ = TruncatedSVD(n_components=n_eff, random_state=self.random_state)
        self.svd_.fit(X)
        return self

    def transform(self, X):
        if self.svd_ is None:
            return X
        return self.svd_.transform(np.asarray(X))


# =========================
# 3) Feature engineering
# =========================
def engineer_base_frame(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()

    # normalize key categoricals
    for c in ["shape", "surface treatment", "dispersion medium", "Substrate"]:
        if c in df.columns:
            df[c] = clean_text_series(df[c])

    # numeric coercion
    numeric_like = [
        "contain O", "contain N", "contain P", "contain S", "contain Si", "contain Se", "contain B",
        "contain F", "contain Cl", "contain Br", "contain I",
        "main metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "3rd metal ratio", "3rd metal type", "3rd metal valence",
        "size (nm)", "pH", "temperature/oC",
    ]
    for c in numeric_like:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # sparse 3rd metal -> zero
    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in df.columns:
            df[c] = df[c].fillna(0.0)

    # coarse categories
    df["shape_coarse"] = df["shape"].map(coarse_shape) if "shape" in df.columns else np.nan
    df["surface_coarse"] = df["surface treatment"].map(coarse_surface) if "surface treatment" in df.columns else np.nan
    df["medium_coarse"] = df["dispersion medium"].map(coarse_medium) if "dispersion medium" in df.columns else np.nan

    # simple continuous engineering
    size = pd.to_numeric(df.get("size (nm)"), errors="coerce")
    pH = pd.to_numeric(df.get("pH"), errors="coerce")
    temp = pd.to_numeric(df.get("temperature/oC"), errors="coerce")

    df["size_log1p"] = np.log1p(size)
    df["abs_pH_from_7"] = (pH - 7.0).abs()
    df["size_x_pH"] = size * pH
    df["size_x_temp"] = size * temp
    df["pH_x_temp"] = pH * temp

    # metal-property engineering
    for prefix, zcol, rcol, vcol in [
        ("main", "main metal number", "main metal proportion", "main metal valence"),
        ("minor", "minor metal number", "minor metal percentage", "minor metal valence"),
        ("third", "3rd metal type", "3rd metal ratio", "3rd metal valence"),
    ]:
        z = pd.to_numeric(df.get(zcol), errors="coerce")
        ratio = pd.to_numeric(df.get(rcol), errors="coerce").fillna(0.0) / 100.0
        val = pd.to_numeric(df.get(vcol), errors="coerce")

        df[f"{prefix}_period"] = z.map(lambda a: atomic_prop(a, 0))
        df[f"{prefix}_group"] = z.map(lambda a: atomic_prop(a, 1))
        df[f"{prefix}_en"] = z.map(lambda a: atomic_prop(a, 2))
        df[f"{prefix}_radius"] = z.map(lambda a: atomic_prop(a, 3))
        df[f"{prefix}_noble"] = z.map(lambda a: atomic_prop(a, 4, 0)).fillna(0.0)
        df[f"{prefix}_rare_earth"] = z.map(lambda a: atomic_prop(a, 5, 0)).fillna(0.0)
        df[f"{prefix}_transition"] = z.map(lambda a: atomic_prop(a, 6, 0)).fillna(0.0)
        df[f"{prefix}_weighted_valence"] = ratio * val
        df[f"{prefix}_weighted_en"] = ratio * df[f"{prefix}_en"]
        df[f"{prefix}_weighted_radius"] = ratio * df[f"{prefix}_radius"]

    # pairwise metal contrasts
    df["main_minor_en_gap"] = (df["main_en"] - df["minor_en"]).abs()
    df["main_minor_radius_gap"] = (df["main_radius"] - df["minor_radius"]).abs()
    df["main_minor_valence_gap"] = (
        pd.to_numeric(df.get("main metal valence"), errors="coerce") - pd.to_numeric(df.get("minor metal valence"), errors="coerce")
    ).abs()

    # weighted composition descriptors
    df["weighted_mean_en"] = df[["main_weighted_en", "minor_weighted_en", "third_weighted_en"]].sum(axis=1)
    df["weighted_mean_radius"] = df[["main_weighted_radius", "minor_weighted_radius", "third_weighted_radius"]].sum(axis=1)
    df["weighted_mean_valence"] = df[["main_weighted_valence", "minor_weighted_valence", "third_weighted_valence"]].sum(axis=1)
    df["n_noble_metals"] = df[["main_noble", "minor_noble", "third_noble"]].sum(axis=1)
    df["n_transition_metals"] = df[["main_transition", "minor_transition", "third_transition"]].sum(axis=1)

    return df


def get_feature_bundle(df_feat: pd.DataFrame, mode: str):
    baseline_num = [
        "contain O", "contain N", "contain P", "contain S", "contain Si", "contain F", "contain Br",
        "main metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "size (nm)", "pH", "temperature/oC",
    ]
    baseline_cat = ["shape_coarse", "surface_coarse", "medium_coarse"]

    metal_num = [
        "main metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "main_period", "main_group", "main_en", "main_radius", "main_noble", "main_transition",
        "minor_period", "minor_group", "minor_en", "minor_radius", "minor_noble", "minor_transition",
        "third_period", "third_group", "third_en", "third_radius", "third_noble", "third_transition",
        "weighted_mean_en", "weighted_mean_radius", "weighted_mean_valence",
        "main_minor_en_gap", "main_minor_radius_gap", "main_minor_valence_gap",
        "n_noble_metals", "n_transition_metals",
    ]
    env_num = ["size (nm)", "size_log1p", "pH", "temperature/oC", "abs_pH_from_7", "size_x_pH", "size_x_temp", "pH_x_temp"]
    env_cat = ["shape_coarse", "medium_coarse", "surface_coarse"]

    feature_modes = {
        "baseline": (baseline_num, baseline_cat),
        "baseline_slim": (baseline_num, ["shape_coarse", "medium_coarse"]),
        "metal_env": (metal_num + env_num, ["shape_coarse", "medium_coarse"]),
        "enhanced_pruned": (baseline_num + metal_num + env_num, env_cat),
    }
    if mode not in feature_modes:
        raise ValueError(mode)

    num_cols, cat_cols = feature_modes[mode]

    # de-duplicate while preserving order; otherwise repeated names (especially in
    # enhanced_pruned) make X[c] return a DataFrame instead of a Series.
    num_cols = [c for c in dict.fromkeys(num_cols) if c in df_feat.columns]
    cat_cols = [c for c in dict.fromkeys(cat_cols) if c in df_feat.columns and c not in num_cols]

    X = df_feat[num_cols + cat_cols].copy()
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    for c in cat_cols:
        X[c] = clean_text_series(X[c])
    return X, num_cols, cat_cols


# =========================
# 4) Sample selection modes
# =========================
def build_target_subset(df_feat: pd.DataFrame, y_col: str, sample_mode: str):
    y_num = pd.to_numeric(df_feat[y_col], errors="coerce")
    mask = y_num.notna() & (y_num > 0)

    size = pd.to_numeric(df_feat.get("size (nm)"), errors="coerce")
    pH = pd.to_numeric(df_feat.get("pH"), errors="coerce")
    temp = pd.to_numeric(df_feat.get("temperature/oC"), errors="coerce")
    med = df_feat.get("medium_coarse")

    if sample_mode == "full":
        pass
    elif sample_mode == "complete_core":
        mask &= size.notna() & pH.notna() & temp.notna() & med.notna()
    elif sample_mode == "common_buffer":
        mask &= med.isin(["phosphate", "tris", "water", "acetate"])
    elif sample_mode == "assay_core":
        mask &= med.isin(["phosphate", "tris", "water", "acetate"]) & pH.between(6.0, 8.0) & temp.isin([20, 25, 30, 37, 40])
    else:
        raise ValueError(sample_mode)

    df_sub = df_feat[mask].copy()
    if len(df_sub) < MIN_SAMPLES_FOR_RUN:
        return None
    n_groups = df_sub[COL_DOI].fillna("").astype(str).str.strip().replace("", np.nan).nunique(dropna=True)
    if n_groups < MIN_GROUPS_FOR_RUN:
        return None
    return df_sub


# =========================
# 5) Metrics + group CV
# =========================
def pearsonr_safe(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    r = pearsonr_safe(y_true, y_pred)
    return {
        "r2_log_oof": float(r2_score(y_true, y_pred)),
        "mae_log_oof": float(mean_absolute_error(y_true, y_pred)),
        "rmse_log_oof": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "pearson_r_oof": float(r) if not pd.isna(r) else np.nan,
        "pearson_r2_oof": float(r * r) if not pd.isna(r) else np.nan,
    }


def make_groups(df_unit: pd.DataFrame, col_doi: str):
    groups = df_unit[col_doi].fillna("").astype(str).str.strip()
    miss = groups.eq("")
    groups.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df_unit.index[miss]]
    return groups.values


# =========================
# 6) Model factory + pipelines
# =========================
def instantiate_model(model_name, params):
    if model_name == "SVR_RBF":
        return SVR(kernel="rbf", **params)
    if model_name == "KernelRidge_RBF":
        return KernelRidge(kernel="rbf", **params)
    if model_name == "RandomForest":
        return RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1, **params)
    if model_name == "ExtraTrees":
        return ExtraTreesRegressor(random_state=RANDOM_SEED, n_jobs=-1, **params)
    if model_name == "HistGBR":
        return HistGradientBoostingRegressor(random_state=RANDOM_SEED, **params)
    if model_name == "KNN_distance":
        return KNeighborsRegressor(weights="distance", metric="minkowski", **params)
    raise ValueError(model_name)


def build_preprocess(numeric_cols, categorical_cols, min_freq_cat=2):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=True)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=min_freq_cat)),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop",
    )


def build_pipeline_for_config(numeric_cols, categorical_cols,
                              selector_mode, k_best, svd_dim,
                              scaler_choice, model_name, model_params,
                              min_freq_cat=2, var_thr=0.0):
    preprocess = build_preprocess(numeric_cols, categorical_cols, min_freq_cat=min_freq_cat)

    if scaler_choice == "robust":
        scaler = RobustScaler(with_centering=True)
    else:
        scaler = StandardScaler(with_mean=True)

    # Strip internal control flags before constructing the sklearn estimator.
    model_params = model_params.copy()
    use_y_scaler = model_params.pop("__use_y_scaler__", True)

    steps = [
        ("preprocess", preprocess),
        ("var", VarianceThreshold(threshold=float(var_thr))),
        ("todense", ToDense()),
        ("selector", SelectKBestSafe(mode=selector_mode, k=k_best, random_state=RANDOM_SEED)),
        ("svd", TruncatedSVDSafe(n_components=svd_dim, random_state=RANDOM_SEED)),
        ("scaler", scaler),
        ("model", instantiate_model(model_name, model_params)),
    ]
    reg = Pipeline(steps)

    if use_y_scaler:
        return TransformedTargetRegressor(
            regressor=reg,
            transformer=StandardScaler(),
        )
    return reg


def groupkfold_oof_predict(X_raw, y_log, groups, pipe_builder):
    uniq_groups = np.unique(groups)
    n_folds = min(MAX_FOLDS, len(uniq_groups))
    if n_folds < 3:
        return None

    gkf = GroupKFold(n_splits=n_folds)
    oof_pred = np.full(len(y_log), np.nan, dtype=float)

    for tr, te in gkf.split(X_raw, y_log, groups=groups):
        Xtr, Xte = X_raw.iloc[tr].copy(), X_raw.iloc[te].copy()
        ytr = y_log[tr]
        pipe = pipe_builder()
        pipe.fit(Xtr, ytr)
        oof_pred[te] = np.asarray(pipe.predict(Xte)).reshape(-1)

    ok = np.isfinite(oof_pred)
    return oof_pred, ok, n_folds


# =========================
# 7) Search spaces
# =========================
V_PREPROCESS_CANDIDATES = [
    # good old baseline line
    {"sample_mode": "full", "feature_mode": "baseline",       "selector": "f_regression", "k_best": 10, "svd_dim": 8, "scaler_choice": "standard"},
    {"sample_mode": "full", "feature_mode": "baseline",       "selector": "f_regression", "k_best": 12, "svd_dim": 8, "scaler_choice": "standard"},
    {"sample_mode": "full", "feature_mode": "baseline",       "selector": "mutual_info",  "k_best": 10, "svd_dim": 8, "scaler_choice": "standard"},
    {"sample_mode": "full", "feature_mode": "baseline",       "selector": "tree_embed",   "k_best": 10, "svd_dim": 8, "scaler_choice": "robust"},

    # slim categories
    {"sample_mode": "full", "feature_mode": "baseline_slim",  "selector": "f_regression", "k_best": 8,  "svd_dim": 8, "scaler_choice": "standard"},
    {"sample_mode": "full", "feature_mode": "baseline_slim",  "selector": "mutual_info",  "k_best": 8,  "svd_dim": 8, "scaler_choice": "robust"},

    # engineered metal/env
    {"sample_mode": "full", "feature_mode": "enhanced_pruned", "selector": "f_regression", "k_best": 10, "svd_dim": 8, "scaler_choice": "standard"},
    {"sample_mode": "full", "feature_mode": "enhanced_pruned", "selector": "tree_embed",   "k_best": 10, "svd_dim": 8, "scaler_choice": "robust"},
    {"sample_mode": "full", "feature_mode": "metal_env",      "selector": "f_regression", "k_best": 10, "svd_dim": 8, "scaler_choice": "standard"},

    # sample filtering
    {"sample_mode": "complete_core", "feature_mode": "metal_env",     "selector": "f_regression", "k_best": 10, "svd_dim": 8, "scaler_choice": "standard"},
    {"sample_mode": "common_buffer", "feature_mode": "baseline",      "selector": "f_regression", "k_best": 10, "svd_dim": 8, "scaler_choice": "robust"},
    {"sample_mode": "common_buffer", "feature_mode": "baseline_slim", "selector": "f_regression", "k_best": 10, "svd_dim": 8, "scaler_choice": "standard"},
    {"sample_mode": "assay_core",    "feature_mode": "baseline",      "selector": "f_regression", "k_best": 10, "svd_dim": 8, "scaler_choice": "standard"},
]

V_MODEL_GRIDS = {
    "SVR_RBF": [
        {"C": C, "gamma": g, "epsilon": e, "__use_y_scaler__": True}
        for C in [10, 15, 20, 30]
        for g in [0.05, 0.08, 0.10, 0.15]
        for e in [0.05, 0.08, 0.10]
    ],
    "KernelRidge_RBF": [
        {"alpha": a, "gamma": g, "__use_y_scaler__": True}
        for a in [0.005, 0.01, 0.03]
        for g in [0.10, 0.30, 0.50]
    ],
    "RandomForest": [
        {"n_estimators": 400, "max_depth": d, "min_samples_leaf": leaf, "max_features": mf, "__use_y_scaler__": False}
        for d in [None, 4, 8]
        for leaf in [1, 2]
        for mf in ["sqrt", 0.7]
    ],
    "ExtraTrees": [
        {"n_estimators": 500, "max_depth": d, "min_samples_leaf": leaf, "max_features": mf, "__use_y_scaler__": False}
        for d in [None, 4, 8]
        for leaf in [1, 2]
        for mf in ["sqrt", 0.7]
    ],
    "HistGBR": [
        {"learning_rate": lr, "max_depth": d, "max_iter": it, "min_samples_leaf": leaf, "__use_y_scaler__": False}
        for lr in [0.03, 0.06, 0.10]
        for d in [3, None]
        for it in [200, 400]
        for leaf in [3, 5]
    ],
    "KNN_distance": [
        {"n_neighbors": n, "p": p, "__use_y_scaler__": True}
        for n in [3, 5, 7]
        for p in [1, 2]
    ],
}

KM_PREPROCESS_CANDIDATES = [
    {"sample_mode": "full", "feature_mode": "enhanced_pruned", "selector": "f_regression", "k_best": 10, "svd_dim": 8, "scaler_choice": "standard"},
    {"sample_mode": "full", "feature_mode": "enhanced_pruned", "selector": "mutual_info",  "k_best": 10, "svd_dim": 8, "scaler_choice": "standard"},
    {"sample_mode": "full", "feature_mode": "metal_env",      "selector": "f_regression", "k_best": 10, "svd_dim": 8, "scaler_choice": "robust"},
    {"sample_mode": "complete_core", "feature_mode": "enhanced_pruned", "selector": "f_regression", "k_best": 10, "svd_dim": 8, "scaler_choice": "standard"},
]

KM_MODEL_GRIDS = {
    "KernelRidge_RBF": [
        {"alpha": a, "gamma": g, "__use_y_scaler__": True}
        for a in [0.003, 0.005, 0.01, 0.02]
        for g in [0.20, 0.30, 0.50]
    ],
    "SVR_RBF": [
        {"C": C, "gamma": g, "epsilon": e, "__use_y_scaler__": True}
        for C in [8, 12, 15]
        for g in [0.08, 0.10, 0.15]
        for e in [0.08, 0.10]
    ],
    "ExtraTrees": [
        {"n_estimators": 400, "max_depth": d, "min_samples_leaf": leaf, "max_features": mf, "__use_y_scaler__": False}
        for d in [None, 4]
        for leaf in [1, 2]
        for mf in ["sqrt", 0.7]
    ],
}


# =========================
# 8) Search one target
# =========================
def run_target(target_name, df_feat_all, y_col, preprocess_candidates, model_grids):
    all_rows = []
    best_row = None
    best_subset = None
    best_X = None
    best_y_log = None
    best_groups = None
    start_all = time.time()

    # rough config count for logging
    n_cfg = len(preprocess_candidates) * sum(len(v) for v in model_grids.values())
    print(f"\n[{target_name}] search space ~= {n_cfg} configs (before CV folds)")

    family_best_rows = []

    for model_name, grid_list in model_grids.items():
        fam_best = None
        fam_start = time.time()

        for pre in preprocess_candidates:
            df_sub = build_target_subset(df_feat_all, y_col, pre["sample_mode"])
            if df_sub is None:
                continue

            X_raw, num_cols, cat_cols = get_feature_bundle(df_sub, pre["feature_mode"])
            y = pd.to_numeric(df_sub[y_col], errors="coerce").astype(float).values
            y_log = np.log10(y)
            groups = make_groups(df_sub, COL_DOI)

            # skip too-small after filtering
            if len(df_sub) < MIN_SAMPLES_FOR_RUN or len(np.unique(groups)) < MIN_GROUPS_FOR_RUN:
                continue

            for model_params in grid_list:
                def builder(pre=pre, num_cols=num_cols, cat_cols=cat_cols, model_name=model_name, model_params=model_params.copy()):
                    return build_pipeline_for_config(
                        numeric_cols=num_cols,
                        categorical_cols=cat_cols,
                        selector_mode=pre["selector"],
                        k_best=pre["k_best"],
                        svd_dim=pre["svd_dim"],
                        scaler_choice=pre["scaler_choice"],
                        model_name=model_name,
                        model_params=model_params.copy(),
                        min_freq_cat=2,
                        var_thr=0.0,
                    )

                out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
                if out is None:
                    continue
                pred, ok, n_folds = out
                m = eval_metrics(y_log[ok], pred[ok])

                row = {
                    "target": target_name,
                    "model": model_name,
                    "sample_mode": pre["sample_mode"],
                    "feature_mode": pre["feature_mode"],
                    "selector": pre["selector"],
                    "min_freq_cat": 2,
                    "var_thr": 0.0,
                    "k_best_req": pre["k_best"],
                    "svd_dim_req": pre["svd_dim"],
                    "scaler_choice": pre["scaler_choice"],
                    "model_params": json.dumps(model_params, ensure_ascii=False),
                    "n_samples": int(len(df_sub)),
                    "n_groups": int(len(np.unique(groups))),
                    "singleton_groups": int(pd.Series(groups).value_counts().eq(1).sum()),
                    "n_folds": int(n_folds),
                    **m,
                }
                all_rows.append(row)

                if fam_best is None or (row["r2_log_oof"] > fam_best["r2_log_oof"]) or (
                    row["r2_log_oof"] == fam_best["r2_log_oof"] and row["rmse_log_oof"] < fam_best["rmse_log_oof"]
                ):
                    fam_best = row.copy()

                if best_row is None or (row["r2_log_oof"] > best_row["r2_log_oof"]) or (
                    row["r2_log_oof"] == best_row["r2_log_oof"] and row["rmse_log_oof"] < best_row["rmse_log_oof"]
                ):
                    best_row = row.copy()
                    best_subset = df_sub.copy()
                    best_X = X_raw.copy()
                    best_y_log = y_log.copy()
                    best_groups = groups.copy()

        if fam_best is not None:
            family_best_rows.append(fam_best)
            print(
                f"  - {model_name:<16} | best R2={fam_best['r2_log_oof']:.4f} | RMSE={fam_best['rmse_log_oof']:.4f} "
                f"| sample={fam_best['sample_mode']} | feature={fam_best['feature_mode']} | selector={fam_best['selector']} "
                f"| k={fam_best['k_best_req']} | svd={fam_best['svd_dim_req']} | scaler={fam_best['scaler_choice']} "
                f"| time={time.time()-fam_start:.1f}s"
            )

    if best_row is None:
        raise RuntimeError(f"{target_name}: no valid result")

    # save full search table
    res = pd.DataFrame(all_rows)
    res.to_csv(OUT_DIR / f"{target_name}_search_summary.csv", index=False, encoding="utf-8-sig")

    # rebuild best model for OOF save + full refit
    best_model_params = json.loads(best_row["model_params"])
    X_best, num_best, cat_best = get_feature_bundle(best_subset, best_row["feature_mode"])

    def best_builder():
        return build_pipeline_for_config(
            numeric_cols=num_best,
            categorical_cols=cat_best,
            selector_mode=best_row["selector"],
            k_best=best_row["k_best_req"],
            svd_dim=best_row["svd_dim_req"],
            scaler_choice=best_row["scaler_choice"],
            model_name=best_row["model"],
            model_params=best_model_params.copy(),
            min_freq_cat=2,
            var_thr=0.0,
        )

    oof, ok, n_folds = groupkfold_oof_predict(X_best, best_y_log, best_groups, best_builder)
    pd.DataFrame({
        "row_index": best_subset.index.values,
        "doi_group": best_groups,
        "y_log_true": best_y_log,
        "y_log_oof_pred": oof,
        "sample_mode": best_row["sample_mode"],
        "feature_mode": best_row["feature_mode"],
        "selector": best_row["selector"],
    }).to_csv(OUT_DIR / f"{target_name}_oof_predictions_groupkfold.csv", index=False, encoding="utf-8-sig")

    final_pipe = best_builder()
    final_pipe.fit(X_best, best_y_log)

    best_config = {
        "target": target_name,
        "y_col": y_col,
        "target_transform": "log10",
        "split": "GroupKFold by DOI",
        "sample_mode": best_row["sample_mode"],
        "data_stats": {
            "n_samples": int(best_row["n_samples"]),
            "n_groups": int(best_row["n_groups"]),
            "singleton_groups": int(best_row["singleton_groups"]),
            "n_folds": int(n_folds),
        },
        "best": {
            "model": best_row["model"],
            "feature_mode": best_row["feature_mode"],
            "selector": best_row["selector"],
            "preprocess_params": {
                "min_freq_cat": 2,
                "var_thr": 0.0,
                "k_best_req": best_row["k_best_req"],
                "svd_dim_req": best_row["svd_dim_req"],
                "scaler_choice": best_row["scaler_choice"],
            },
            "model_params": best_model_params,
            "metrics_oof": {
                "r2_log_oof": float(best_row["r2_log_oof"]),
                "mae_log_oof": float(best_row["mae_log_oof"]),
                "rmse_log_oof": float(best_row["rmse_log_oof"]),
                "pearson_r_oof": float(best_row["pearson_r_oof"]) if pd.notna(best_row["pearson_r_oof"]) else None,
                "pearson_r2_oof": float(best_row["pearson_r2_oof"]) if pd.notna(best_row["pearson_r2_oof"]) else None,
            },
        },
        "runtime_sec": round(time.time() - start_all, 3),
    }

    with open(OUT_DIR / f"{target_name}_best_config.json", "w", encoding="utf-8") as f:
        json.dump(best_config, f, ensure_ascii=False, indent=2)

    joblib.dump({
        "target": target_name,
        "sample_mode": best_row["sample_mode"],
        "pipeline": final_pipe,
        "best_config": best_config,
        "row_index_used": best_subset.index.tolist(),
    }, OUT_DIR / f"{target_name}_best_model_refit_full.joblib")

    print("\n" + "=" * 72)
    print(f"[BEST] {target_name} | model={best_row['model']} | sample={best_row['sample_mode']} | feature={best_row['feature_mode']}")
    print("selector:", best_row["selector"])
    print("preprocess:", {
        "min_freq_cat": 2,
        "var_thr": 0.0,
        "k_best_req": best_row["k_best_req"],
        "svd_dim_req": best_row["svd_dim_req"],
        "scaler_choice": best_row["scaler_choice"],
    })
    print("model_params:", best_model_params)
    print("metrics_oof:", best_config["best"]["metrics_oof"])
    print("runtime_sec:", best_config["runtime_sec"])
    print("=" * 72)

    return best_config


# =========================
# 9) Main
# =========================
def main():
    t0 = time.time()
    df = pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME)
    df = normalize_columns(df)

    for c in [COL_DOI, COL_KM, COL_VMAX]:
        if c not in df.columns:
            raise KeyError(f"Required column not found: {c}")

    df_feat = engineer_base_frame(df)

    # basic feature info export
    info_rows = []
    for mode in ["baseline", "baseline_slim", "metal_env", "enhanced_pruned"]:
        X_mode, num_cols, cat_cols = get_feature_bundle(df_feat, mode)
        for c in num_cols:
            info_rows.append({"feature_mode": mode, "feature": c, "type": "numeric"})
        for c in cat_cols:
            info_rows.append({"feature_mode": mode, "feature": c, "type": "categorical"})
    pd.DataFrame(info_rows).drop_duplicates().to_csv(OUT_DIR / "feature_info_by_mode.csv", index=False, encoding="utf-8-sig")

    # save sample-mode sizes for transparency
    sample_rows = []
    for target_name, ycol in [("logKm", COL_KM), ("logVmax", COL_VMAX)]:
        for sm in ["full", "complete_core", "common_buffer", "assay_core"]:
            sub = build_target_subset(df_feat, ycol, sm)
            if sub is None:
                sample_rows.append({"target": target_name, "sample_mode": sm, "n_samples": 0, "n_groups": 0})
            else:
                gps = make_groups(sub, COL_DOI)
                sample_rows.append({"target": target_name, "sample_mode": sm, "n_samples": len(sub), "n_groups": len(np.unique(gps))})
    pd.DataFrame(sample_rows).to_csv(OUT_DIR / "sample_mode_counts.csv", index=False, encoding="utf-8-sig")

    if RUN_LOGKM:
        df_km = df[pd.to_numeric(df[COL_KM], errors="coerce").gt(0)].copy()
        df_km.to_csv(OUT_DIR / "used_data_logKm_raw_positive.csv", index=False, encoding="utf-8-sig")
        run_target("logKm", df_feat, COL_KM, KM_PREPROCESS_CANDIDATES, KM_MODEL_GRIDS)

    if RUN_LOGVMAX:
        df_v = df[pd.to_numeric(df[COL_VMAX], errors="coerce").gt(0)].copy()
        df_v.to_csv(OUT_DIR / "used_data_logVmax_raw_positive.csv", index=False, encoding="utf-8-sig")
        run_target("logVmax", df_feat, COL_VMAX, V_PREPROCESS_CANDIDATES, V_MODEL_GRIDS)

    print("\nSaved to:", OUT_DIR)
    print("Total runtime:", round(time.time() - t0, 2), "sec")


if __name__ == "__main__":
    main()



[logKm] search space ~= 152 configs (before CV folds)
  - KernelRidge_RBF  | best R2=0.2587 | RMSE=1.2214 | sample=full | feature=metal_env | selector=f_regression | k=10 | svd=8 | scaler=robust | time=6.3s
  - SVR_RBF          | best R2=-0.0695 | RMSE=1.4671 | sample=full | feature=metal_env | selector=f_regression | k=10 | svd=8 | scaler=robust | time=9.2s
  - ExtraTrees       | best R2=-0.1060 | RMSE=1.5056 | sample=complete_core | feature=enhanced_pruned | selector=f_regression | k=10 | svd=8 | scaler=standard | time=53.3s

[BEST] logKm | model=KernelRidge_RBF | sample=full | feature=metal_env
selector: f_regression
preprocess: {'min_freq_cat': 2, 'var_thr': 0.0, 'k_best_req': 10, 'svd_dim_req': 8, 'scaler_choice': 'robust'}
model_params: {'alpha': 0.005, 'gamma': 0.2, '__use_y_scaler__': True}
metrics_oof: {'r2_log_oof': 0.258737516560149, 'mae_log_oof': 1.0080063110732447, 'rmse_log_oof': 1.2213973165502423, 'pearson_r_oof': 0.5511670444112227, 'pearson_r2_oof': 0.30378511084500

In [17]:
# -*- coding: utf-8 -*-
"""
CAT regression v11: hyper-local refine Vmax around the 0.4009 best point with a narrower targeted search.

What this version does
----------------------
1) Keep the evaluation protocol strict and consistent:
   DOI-GroupKFold OOF only.
2) Stop wasting compute on broad weak directions.
3) Hyper-focus around the strongest lane found so far:
   SVR_RBF + full + baseline + k=12 + svd=8 + standard scaler.
4) Add only a few nearby, still-plausible alternatives:
   - baseline_metal_small for SVR / NuSVR
   - KNN on metal_env / complete_core
   - tree-specific lanes without SVD / scaler
   - optional CatBoost if installed
5) Keep the original Excel untouched; all feature engineering is in memory.

Expected use
------------
- Best for continuing the current Vmax push.
- Default runs logVmax only.
"""

import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_regression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.svm import NuSVR, SVR

warnings.filterwarnings("ignore")

try:
    from catboost import CatBoostRegressor
    HAS_CATBOOST = True
except Exception:
    CatBoostRegressor = None
    HAS_CATBOOST = False

# =========================
# 0) Paths & config
# =========================
CAT_PATH = Path(r"C:\Users\86158\Desktop\dataiscat.xlsx")  # <-- 改成你的输入文件
SHEET_NAME = 0

OUT_DIR = CAT_PATH.parent / "cat_km_vmax_reg_doi_multistage_out_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_DOI = "data reference doi"
COL_KM = "Km/mM"
COL_VMAX = "Vmax/μM s-1"

RANDOM_SEED = 42
MAX_FOLDS = 5
MIN_SAMPLES_FOR_RUN = 14
MIN_GROUPS_FOR_RUN = 8

RUN_LOGKM = False
RUN_LOGVMAX = True

NUMERIC_MISSING_THRESHOLD = 0.50
CATEGORICAL_MISSING_THRESHOLD = 0.75

# =========================
# 1) Basic utils
# =========================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def coarse_shape(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if "sphere" in x:
        return "sphere"
    if "particle" in x:
        return "particle"
    if "cluster" in x:
        return "cluster"
    if "sheet" in x:
        return "sheet"
    if "rod" in x:
        return "rod"
    if "cube" in x:
        return "cube"
    if "flower" in x:
        return "flower"
    return "other"


def coarse_surface(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if x in {"none", "na", "nan"}:
        return "none"
    if any(k in x for k in ["peg", "pvp", "paa", "pamam", "dspe", "β-cd", "beta-cd", "mpa", "aa"]):
        return "poly_small"
    if "sio2" in x or "silica" in x:
        return "silica"
    if any(k in x for k in ["apo", "ferritin", "histidine", "gsh", "silk", "amp", "nta"]):
        return "bio_ligand"
    return "other"


def coarse_medium(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if x in {"pbs", "phosphate buffer", "pb", "phosphate-buffered saline"}:
        return "phosphate"
    if "tris" in x:
        return "tris"
    if "water" in x:
        return "water"
    if "acetate" in x or "hac-naac" in x:
        return "acetate"
    if "mops" in x:
        return "mops"
    return "other"


ATOMIC_PROPS = {
    12: (3, 2, 1.31, 160, 0, 0, 0),
    20: (4, 2, 1.00, 197, 0, 0, 0),
    23: (4, 5, 1.63, 134, 0, 0, 1),
    25: (4, 7, 1.55, 127, 0, 0, 1),
    26: (4, 8, 1.83, 126, 0, 0, 1),
    27: (4, 9, 1.88, 125, 0, 0, 1),
    28: (4, 10, 1.91, 124, 0, 0, 1),
    29: (4, 11, 1.90, 128, 0, 0, 1),
    40: (5, 4, 1.33, 160, 0, 0, 1),
    42: (5, 6, 2.16, 139, 0, 0, 1),
    44: (5, 8, 2.20, 134, 0, 0, 1),
    45: (5, 9, 2.28, 134, 0, 0, 1),
    46: (5, 10, 2.20, 137, 0, 0, 1),
    47: (5, 11, 1.93, 144, 0, 0, 1),
    58: (6, 3, 1.12, 182, 0, 1, 0),
    68: (6, 17, 1.24, 176, 0, 1, 0),
    77: (6, 9, 2.20, 136, 1, 0, 1),
    78: (6, 10, 2.28, 139, 1, 0, 1),
    79: (6, 11, 2.54, 144, 1, 0, 1),
}


def atomic_prop(z, idx, default=np.nan):
    try:
        z = int(z)
    except Exception:
        return default
    if z <= 0:
        return default
    return ATOMIC_PROPS.get(z, (default, default, default, default, default, default, default))[idx]


# =========================
# 2) Transformers
# =========================
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_freq=2, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


class ToDense(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if hasattr(X, "toarray"):
            return X.toarray()
        return np.asarray(X)


class SelectKBestSafe(BaseEstimator, TransformerMixin):
    def __init__(self, k=None):
        self.k = k
        self.selector_ = None

    def fit(self, X, y):
        if self.k is None:
            self.selector_ = None
            return self
        n_feat = X.shape[1]
        k_eff = int(min(int(self.k), int(n_feat)))
        if k_eff <= 0:
            self.selector_ = None
            return self
        self.selector_ = SelectKBest(score_func=f_regression, k=k_eff)
        self.selector_.fit(X, y)
        return self

    def transform(self, X):
        if self.selector_ is None:
            return X
        return self.selector_.transform(X)


class TruncatedSVDSafe(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=None, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.svd_ = None

    def fit(self, X, y=None):
        if self.n_components is None:
            self.svd_ = None
            return self
        n_feat = X.shape[1]
        if n_feat <= 1:
            self.svd_ = None
            return self
        n_eff = min(int(self.n_components), max(1, n_feat - 1))
        self.svd_ = TruncatedSVD(n_components=n_eff, random_state=self.random_state)
        self.svd_.fit(X)
        return self

    def transform(self, X):
        if self.svd_ is None:
            return X
        return self.svd_.transform(X)


# =========================
# 3) Feature engineering / bundles
# =========================
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["shape_coarse"] = df.get("shape", np.nan).map(coarse_shape)
    df["surface_coarse"] = df.get("surface treatment", np.nan).map(coarse_surface)
    df["medium_coarse"] = df.get("dispersion medium", np.nan).map(coarse_medium)

    size = pd.to_numeric(df.get("size (nm)"), errors="coerce")
    pH = pd.to_numeric(df.get("pH"), errors="coerce")
    temp = pd.to_numeric(df.get("temperature/oC"), errors="coerce")

    df["size_log1p"] = np.log1p(size)
    df["abs_pH_from_7"] = (pH - 7.0).abs()
    df["size_x_pH"] = size * pH
    df["size_x_temp"] = size * temp
    df["pH_x_temp"] = pH * temp

    for prefix, zcol, rcol, vcol in [
        ("main", "main metal number", "main metal proportion", "main metal valence"),
        ("minor", "minor metal number", "minor metal percentage", "minor metal valence"),
        ("third", "3rd metal type", "3rd metal ratio", "3rd metal valence"),
    ]:
        z = pd.to_numeric(df.get(zcol), errors="coerce")
        ratio = pd.to_numeric(df.get(rcol), errors="coerce").fillna(0.0) / 100.0
        val = pd.to_numeric(df.get(vcol), errors="coerce")

        df[f"{prefix}_period"] = z.map(lambda a: atomic_prop(a, 0))
        df[f"{prefix}_group"] = z.map(lambda a: atomic_prop(a, 1))
        df[f"{prefix}_en"] = z.map(lambda a: atomic_prop(a, 2))
        df[f"{prefix}_radius"] = z.map(lambda a: atomic_prop(a, 3))
        df[f"{prefix}_noble"] = z.map(lambda a: atomic_prop(a, 4, 0)).fillna(0.0)
        df[f"{prefix}_transition"] = z.map(lambda a: atomic_prop(a, 6, 0)).fillna(0.0)
        df[f"{prefix}_weighted_valence"] = ratio * val
        df[f"{prefix}_weighted_en"] = ratio * df[f"{prefix}_en"]
        df[f"{prefix}_weighted_radius"] = ratio * df[f"{prefix}_radius"]

    df["main_minor_en_gap"] = (df["main_en"] - df["minor_en"]).abs()
    df["main_minor_radius_gap"] = (df["main_radius"] - df["minor_radius"]).abs()
    df["main_minor_valence_gap"] = (
        pd.to_numeric(df.get("main metal valence"), errors="coerce") - pd.to_numeric(df.get("minor metal valence"), errors="coerce")
    ).abs()

    df["weighted_mean_en"] = df[["main_weighted_en", "minor_weighted_en", "third_weighted_en"]].sum(axis=1)
    df["weighted_mean_radius"] = df[["main_weighted_radius", "minor_weighted_radius", "third_weighted_radius"]].sum(axis=1)
    df["weighted_mean_valence"] = df[["main_weighted_valence", "minor_weighted_valence", "third_weighted_valence"]].sum(axis=1)
    df["n_transition_metals"] = df[["main_transition", "minor_transition", "third_transition"]].sum(axis=1)
    return df


def get_feature_bundle(df_feat: pd.DataFrame, mode: str):
    baseline_num = [
        "contain O", "contain N", "contain P", "contain S", "contain Si", "contain F", "contain Br",
        "main metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "size (nm)", "pH", "temperature/oC",
    ]
    baseline_cat = ["shape_coarse", "surface_coarse", "medium_coarse"]

    metal_small_num = [
        "weighted_mean_en", "weighted_mean_radius", "weighted_mean_valence",
        "main_minor_en_gap", "main_minor_radius_gap", "main_minor_valence_gap",
        "n_transition_metals",
    ]

    metal_env_num = [
        "main metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "main_en", "main_radius", "main_transition",
        "minor_en", "minor_radius", "minor_transition",
        "weighted_mean_en", "weighted_mean_radius", "weighted_mean_valence",
        "main_minor_en_gap", "main_minor_radius_gap", "main_minor_valence_gap",
        "n_transition_metals",
        "size (nm)", "size_log1p", "pH", "temperature/oC", "abs_pH_from_7", "size_x_pH", "size_x_temp", "pH_x_temp",
    ]

    feature_modes = {
        "baseline": (baseline_num, baseline_cat),
        "baseline_metal_small": (baseline_num + metal_small_num, baseline_cat),
        "metal_env": (metal_env_num, ["shape_coarse", "medium_coarse"]),
    }
    if mode not in feature_modes:
        raise ValueError(mode)

    num_cols, cat_cols = feature_modes[mode]
    num_cols = [c for c in dict.fromkeys(num_cols) if c in df_feat.columns]
    cat_cols = [c for c in dict.fromkeys(cat_cols) if c in df_feat.columns and c not in num_cols]

    num_cols = [c for c in num_cols if df_feat[c].isna().mean() <= NUMERIC_MISSING_THRESHOLD]
    cat_cols = [
        c for c in cat_cols
        if df_feat[c].isna().mean() <= CATEGORICAL_MISSING_THRESHOLD
        and df_feat[c].dropna().astype(str).str.strip().nunique() >= 2
    ]

    X = df_feat[num_cols + cat_cols].copy()
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    for c in cat_cols:
        X[c] = clean_text_series(X[c])
    return X, num_cols, cat_cols


# =========================
# 4) Sample subsets
# =========================
def build_target_subset(df_feat: pd.DataFrame, y_col: str, sample_mode: str):
    y_num = pd.to_numeric(df_feat[y_col], errors="coerce")
    mask = y_num.notna() & (y_num > 0)

    size = pd.to_numeric(df_feat.get("size (nm)"), errors="coerce")
    pH = pd.to_numeric(df_feat.get("pH"), errors="coerce")
    temp = pd.to_numeric(df_feat.get("temperature/oC"), errors="coerce")
    med = df_feat.get("medium_coarse")

    if sample_mode == "full":
        pass
    elif sample_mode == "complete_core":
        mask &= size.notna() & pH.notna() & temp.notna() & med.notna()
    else:
        raise ValueError(sample_mode)

    df_sub = df_feat[mask].copy()
    if len(df_sub) < MIN_SAMPLES_FOR_RUN:
        return None
    groups = df_sub[COL_DOI].fillna("").astype(str).str.strip().replace("", np.nan)
    if groups.nunique(dropna=True) < MIN_GROUPS_FOR_RUN:
        return None
    return df_sub


# =========================
# 5) Metrics / CV
# =========================
def pearsonr_safe(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    r = pearsonr_safe(y_true, y_pred)
    return {
        "r2_log_oof": float(r2_score(y_true, y_pred)),
        "mae_log_oof": float(mean_absolute_error(y_true, y_pred)),
        "rmse_log_oof": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "pearson_r_oof": float(r) if not pd.isna(r) else np.nan,
        "pearson_r2_oof": float(r * r) if not pd.isna(r) else np.nan,
    }


def make_groups(df_unit: pd.DataFrame, col_doi: str):
    groups = df_unit[col_doi].fillna("").astype(str).str.strip()
    miss = groups.eq("")
    groups.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df_unit.index[miss]]
    return groups.values


def groupkfold_oof_predict(X_raw, y_log, groups, pipe_builder):
    uniq_groups = np.unique(groups)
    n_folds = min(MAX_FOLDS, len(uniq_groups))
    if n_folds < 3:
        return None
    gkf = GroupKFold(n_splits=n_folds)
    oof_pred = np.full(len(y_log), np.nan, dtype=float)
    for tr, te in gkf.split(X_raw, y_log, groups=groups):
        Xtr, Xte = X_raw.iloc[tr].copy(), X_raw.iloc[te].copy()
        ytr = y_log[tr]
        pipe = pipe_builder()
        pipe.fit(Xtr, ytr)
        oof_pred[te] = np.asarray(pipe.predict(Xte)).reshape(-1)
    ok = np.isfinite(oof_pred)
    return oof_pred, ok, n_folds


def groupkfold_oof_predict_catboost(X_raw, y_log, groups, model_params):
    uniq_groups = np.unique(groups)
    n_folds = min(MAX_FOLDS, len(uniq_groups))
    if n_folds < 3:
        return None
    gkf = GroupKFold(n_splits=n_folds)
    oof_pred = np.full(len(y_log), np.nan, dtype=float)

    cat_cols = [c for c in X_raw.columns if X_raw[c].dtype == object]
    cat_idx = [X_raw.columns.get_loc(c) for c in cat_cols]

    for tr, te in gkf.split(X_raw, y_log, groups=groups):
        Xtr = X_raw.iloc[tr].copy()
        Xte = X_raw.iloc[te].copy()
        ytr = y_log[tr]

        for c in Xtr.columns:
            if c in cat_cols:
                Xtr[c] = Xtr[c].fillna("not_reported").astype(str)
                Xte[c] = Xte[c].fillna("not_reported").astype(str)
            else:
                med = pd.to_numeric(Xtr[c], errors="coerce").median()
                Xtr[c] = pd.to_numeric(Xtr[c], errors="coerce").fillna(med)
                Xte[c] = pd.to_numeric(Xte[c], errors="coerce").fillna(med)

        model = CatBoostRegressor(
            loss_function="RMSE",
            random_seed=RANDOM_SEED,
            verbose=False,
            **model_params,
        )
        model.fit(Xtr, ytr, cat_features=cat_idx)
        oof_pred[te] = np.asarray(model.predict(Xte)).reshape(-1)

    ok = np.isfinite(oof_pred)
    return oof_pred, ok, n_folds


# =========================
# 6) Model / pipeline builders
# =========================
def instantiate_model(model_name, params):
    if model_name == "SVR_RBF":
        return SVR(kernel="rbf", **params)
    if model_name == "NuSVR_RBF":
        return NuSVR(kernel="rbf", **params)
    if model_name == "KNN_distance":
        return KNeighborsRegressor(weights="distance", metric="minkowski", **params)
    if model_name == "RandomForest":
        return RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1, **params)
    if model_name == "ExtraTrees":
        return ExtraTreesRegressor(random_state=RANDOM_SEED, n_jobs=-1, **params)
    raise ValueError(model_name)


def build_preprocess(numeric_cols, categorical_cols, min_freq_cat=2):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=True)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=min_freq_cat)),
        ("onehot", oh),
    ])
    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop",
    )


def build_pipeline_for_config(numeric_cols, categorical_cols, k_best, svd_dim, scaler_choice, model_name, model_params, min_freq_cat=2, var_thr=0.0):
    preprocess = build_preprocess(numeric_cols, categorical_cols, min_freq_cat=min_freq_cat)
    model_params = model_params.copy()
    use_y_scaler = model_params.pop("__use_y_scaler__", True)

    scaler = None
    if scaler_choice == "standard":
        scaler = StandardScaler(with_mean=True)
    elif scaler_choice == "robust":
        scaler = RobustScaler(with_centering=True)

    steps = [("preprocess", preprocess), ("var", VarianceThreshold(threshold=float(var_thr)))]

    needs_dense = model_name in {"SVR_RBF", "NuSVR_RBF", "KNN_distance"}
    if needs_dense:
        steps.append(("todense", ToDense()))

    if k_best is not None:
        steps.append(("kbest", SelectKBestSafe(k=k_best)))
    if svd_dim is not None:
        steps.append(("svd", TruncatedSVDSafe(n_components=svd_dim, random_state=RANDOM_SEED)))
    if scaler is not None:
        steps.append(("scaler", scaler))
    steps.append(("model", instantiate_model(model_name, model_params)))

    reg = Pipeline(steps)
    if use_y_scaler:
        return TransformedTargetRegressor(regressor=reg, transformer=StandardScaler())
    return reg


# =========================
# 7) Search spaces
# =========================
V_BRANCHES = {
    "SVR_RBF": {
        "preprocess": [
            {"sample_mode": "full", "feature_mode": "baseline", "k_best": 12, "svd_dim": 8, "scaler_choice": "standard"},
            {"sample_mode": "full", "feature_mode": "baseline", "k_best": 11, "svd_dim": 8, "scaler_choice": "standard"},
            {"sample_mode": "full", "feature_mode": "baseline", "k_best": 13, "svd_dim": 8, "scaler_choice": "standard"},
            {"sample_mode": "full", "feature_mode": "baseline", "k_best": 12, "svd_dim": 7, "scaler_choice": "standard"},
            {"sample_mode": "full", "feature_mode": "baseline", "k_best": 12, "svd_dim": 9, "scaler_choice": "standard"},
            {"sample_mode": "full", "feature_mode": "baseline_metal_small", "k_best": 12, "svd_dim": 8, "scaler_choice": "standard"},
        ],
        "params": [
            {"C": C, "gamma": g, "epsilon": e, "__use_y_scaler__": True}
            for C in [80, 100, 120, 150, 180, 220, 300]
            for g in [0.015, 0.02, 0.025, 0.03, 0.035]
            for e in [0.04, 0.05, 0.06, 0.07]
        ],
    },
    "NuSVR_RBF": {
        "preprocess": [
            {"sample_mode": "full", "feature_mode": "baseline", "k_best": 12, "svd_dim": 8, "scaler_choice": "standard"},
        ],
        "params": [
            {"nu": nu, "C": C, "gamma": g, "__use_y_scaler__": True}
            for nu in [0.35, 0.45, 0.55, 0.65]
            for C in [60, 80, 100, 120]
            for g in [0.015, 0.02, 0.025, 0.03, 0.035]
        ],
    },
    "KNN_distance": {
        "preprocess": [
            {"sample_mode": "complete_core", "feature_mode": "metal_env", "k_best": 10, "svd_dim": 8, "scaler_choice": "robust"},
            {"sample_mode": "full", "feature_mode": "metal_env", "k_best": 10, "svd_dim": 8, "scaler_choice": "robust"},
        ],
        "params": [
            {"n_neighbors": n, "p": p, "__use_y_scaler__": True}
            for n in [3, 4, 5, 6, 7, 8]
            for p in [1, 2]
        ],
    },
}

KM_BRANCHES = {
    "SVR_RBF": {
        "preprocess": [
            {"sample_mode": "full", "feature_mode": "metal_env", "k_best": 10, "svd_dim": 8, "scaler_choice": "robust"},
        ],
        "params": [
            {"C": C, "gamma": g, "epsilon": e, "__use_y_scaler__": True}
            for C in [10, 20]
            for g in [0.1, 0.2, 0.3]
            for e in [0.05, 0.10]
        ],
    }
}

# =========================
# 8) Target runner
# =========================
def run_target(target_name, df_feat_all, y_col, branches, enable_catboost=False):
    print()
    y_num_all = pd.to_numeric(df_feat_all[y_col], errors="coerce")
    base_mask = y_num_all.notna() & (y_num_all > 0)
    df_base = df_feat_all[base_mask].copy()
    groups_base = make_groups(df_base, COL_DOI)
    print(f"[{target_name}] n_samples={len(df_base)} | n_groups={pd.Series(groups_base).nunique()} | singleton_groups={pd.Series(groups_base).value_counts().eq(1).sum()}")

    all_rows = []
    best_overall = None
    best_payload = None

    total_configs = 0
    for cfg in branches.values():
        total_configs += len(cfg["preprocess"]) * len(cfg["params"])
    if enable_catboost and HAS_CATBOOST:
        total_configs += len(CATBOOST_BRANCH["preprocess"]) * len(CATBOOST_BRANCH["params"])
    print(f"[{target_name}] search space ~= {total_configs} configs (before CV folds)")

    # regular branches
    for model_name, cfg in branches.items():
        t0 = time.time()
        best_row_model = None
        best_payload_model = None

        for pre in cfg["preprocess"]:
            df_sub = build_target_subset(df_feat_all, y_col, pre["sample_mode"])
            if df_sub is None:
                continue
            y_log = np.log10(pd.to_numeric(df_sub[y_col], errors="coerce").astype(float).values)
            groups = make_groups(df_sub, COL_DOI)
            X_raw, numeric_cols, categorical_cols = get_feature_bundle(df_sub, pre["feature_mode"])
            if X_raw.shape[1] < 3:
                continue

            for model_params in cfg["params"]:
                def builder(mp=model_params, pp=pre):
                    return build_pipeline_for_config(
                        numeric_cols, categorical_cols,
                        k_best=pp.get("k_best"),
                        svd_dim=pp.get("svd_dim"),
                        scaler_choice=pp.get("scaler_choice"),
                        model_name=model_name,
                        model_params=mp,
                        min_freq_cat=2,
                        var_thr=0.0,
                    )

                out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
                if out is None:
                    continue
                pred, ok, n_folds = out
                m = eval_metrics(y_log[ok], pred[ok])
                row = {
                    "target": target_name,
                    "model": model_name,
                    "sample_mode": pre["sample_mode"],
                    "feature_mode": pre["feature_mode"],
                    "k_best_req": pre.get("k_best"),
                    "svd_dim_req": pre.get("svd_dim"),
                    "scaler_choice": pre.get("scaler_choice"),
                    "n_samples": len(df_sub),
                    "n_groups": pd.Series(groups).nunique(),
                    "n_folds": n_folds,
                    "model_params": json.dumps(model_params, ensure_ascii=False),
                    **m,
                }
                all_rows.append(row)

                if (best_row_model is None) or ((row["r2_log_oof"], -row["rmse_log_oof"]) > (best_row_model["r2_log_oof"], -best_row_model["rmse_log_oof"])):
                    best_row_model = row
                    best_payload_model = (df_sub.copy(), X_raw.copy(), y_log.copy(), groups.copy(), numeric_cols[:], categorical_cols[:], pre.copy(), model_params.copy(), model_name)

        elapsed = time.time() - t0
        if best_row_model is not None:
            print(
                f"  - {model_name:<16} | best R2={best_row_model['r2_log_oof']:.4f} | "
                f"RMSE={best_row_model['rmse_log_oof']:.4f} | sample={best_row_model['sample_mode']} | "
                f"feature={best_row_model['feature_mode']} | k={best_row_model['k_best_req']} | "
                f"svd={best_row_model['svd_dim_req']} | scaler={best_row_model['scaler_choice']} | time={elapsed:.1f}s"
            )
            if (best_overall is None) or ((best_row_model["r2_log_oof"], -best_row_model["rmse_log_oof"]) > (best_overall["r2_log_oof"], -best_overall["rmse_log_oof"])):
                best_overall = best_row_model
                best_payload = best_payload_model

    # optional CatBoost branch
    if enable_catboost and HAS_CATBOOST:
        t0 = time.time()
        best_row_model = None
        best_payload_model = None
        for pre in CATBOOST_BRANCH["preprocess"]:
            df_sub = build_target_subset(df_feat_all, y_col, pre["sample_mode"])
            if df_sub is None:
                continue
            y_log = np.log10(pd.to_numeric(df_sub[y_col], errors="coerce").astype(float).values)
            groups = make_groups(df_sub, COL_DOI)
            X_raw, _, _ = get_feature_bundle(df_sub, pre["feature_mode"])
            if X_raw.shape[1] < 3:
                continue
            for model_params in CATBOOST_BRANCH["params"]:
                out = groupkfold_oof_predict_catboost(X_raw, y_log, groups, model_params)
                if out is None:
                    continue
                pred, ok, n_folds = out
                m = eval_metrics(y_log[ok], pred[ok])
                row = {
                    "target": target_name,
                    "model": "CatBoost",
                    "sample_mode": pre["sample_mode"],
                    "feature_mode": pre["feature_mode"],
                    "k_best_req": None,
                    "svd_dim_req": None,
                    "scaler_choice": None,
                    "n_samples": len(df_sub),
                    "n_groups": pd.Series(groups).nunique(),
                    "n_folds": n_folds,
                    "model_params": json.dumps(model_params, ensure_ascii=False),
                    **m,
                }
                all_rows.append(row)
                if (best_row_model is None) or ((row["r2_log_oof"], -row["rmse_log_oof"]) > (best_row_model["r2_log_oof"], -best_row_model["rmse_log_oof"])):
                    best_row_model = row
                    best_payload_model = (df_sub.copy(), X_raw.copy(), y_log.copy(), groups.copy(), [], [], pre.copy(), model_params.copy(), "CatBoost")

        elapsed = time.time() - t0
        if best_row_model is not None:
            print(
                f"  - {'CatBoost':<16} | best R2={best_row_model['r2_log_oof']:.4f} | "
                f"RMSE={best_row_model['rmse_log_oof']:.4f} | sample={best_row_model['sample_mode']} | "
                f"feature={best_row_model['feature_mode']} | k=None | svd=None | scaler=None | time={elapsed:.1f}s"
            )
            if (best_overall is None) or ((best_row_model["r2_log_oof"], -best_row_model["rmse_log_oof"]) > (best_overall["r2_log_oof"], -best_overall["rmse_log_oof"])):
                best_overall = best_row_model
                best_payload = best_payload_model
    elif enable_catboost:
        print("  - CatBoost         | skipped (catboost not installed)")

    if best_overall is None:
        raise RuntimeError(f"{target_name}: no valid configuration.")

    res_all = pd.DataFrame(all_rows).sort_values(["r2_log_oof", "rmse_log_oof"], ascending=[False, True])
    res_all.to_csv(OUT_DIR / f"{target_name}_search_summary.csv", index=False, encoding="utf-8-sig")

    df_sub, X_raw, y_log, groups, numeric_cols, categorical_cols, pre, model_params, model_name = best_payload

    if model_name == "CatBoost":
        oof, ok, n_folds = groupkfold_oof_predict_catboost(X_raw, y_log, groups, model_params)
        final_pack = {
            "target": target_name,
            "best_config": best_overall,
            "model_name": model_name,
            "preprocess": pre,
            "model_params": model_params,
        }
        joblib.dump(final_pack, OUT_DIR / f"{target_name}_best_model_refit_full.joblib")
    else:
        def best_builder():
            return build_pipeline_for_config(
                numeric_cols, categorical_cols,
                k_best=pre.get("k_best"),
                svd_dim=pre.get("svd_dim"),
                scaler_choice=pre.get("scaler_choice"),
                model_name=model_name,
                model_params=model_params,
                min_freq_cat=2,
                var_thr=0.0,
            )
        oof, ok, n_folds = groupkfold_oof_predict(X_raw, y_log, groups, best_builder)
        final_model = best_builder()
        final_model.fit(X_raw, y_log)
        joblib.dump(
            {
                "target": target_name,
                "pipeline": final_model,
                "best_config": best_overall,
                "feature_mode": pre["feature_mode"],
                "sample_mode": pre["sample_mode"],
                "numeric_cols": numeric_cols,
                "categorical_cols": categorical_cols,
            },
            OUT_DIR / f"{target_name}_best_model_refit_full.joblib",
        )

    pd.DataFrame({
        "row_index": df_sub.index.values,
        "doi_group": groups,
        "y_log_true": y_log,
        "y_log_oof_pred": oof,
    }).to_csv(OUT_DIR / f"{target_name}_oof_predictions_groupkfold.csv", index=False, encoding="utf-8-sig")

    best_config = {
        "target": target_name,
        "y_col": y_col,
        "target_transform": "log10",
        "split": "GroupKFold by DOI",
        "best": {
            "model": model_name,
            "sample_mode": pre.get("sample_mode"),
            "feature_mode": pre.get("feature_mode"),
            "preprocess": {
                "k_best_req": pre.get("k_best"),
                "svd_dim_req": pre.get("svd_dim"),
                "scaler_choice": pre.get("scaler_choice"),
            },
            "model_params": model_params,
            "metrics_oof": eval_metrics(y_log[ok], oof[ok]),
        }
    }
    with open(OUT_DIR / f"{target_name}_best_config.json", "w", encoding="utf-8") as f:
        json.dump(best_config, f, ensure_ascii=False, indent=2)

    print("\n" + "=" * 72)
    print(f"[BEST] {target_name} | model={model_name} | sample={pre.get('sample_mode')} | feature={pre.get('feature_mode')}")
    print("preprocess:", best_config["best"]["preprocess"])
    print("model_params:", model_params)
    print("metrics_oof:", best_config["best"]["metrics_oof"])
    print("=" * 72)

    return best_config


# =========================
# 9) Main
# =========================
def main():
    t_all = time.time()
    df = pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME)
    df = normalize_columns(df)

    for c in [COL_DOI, COL_KM, COL_VMAX]:
        if c not in df.columns:
            raise KeyError(f"Required column not found: {c}")

    df_feat = engineer_features(df)

    if RUN_LOGKM:
        t0 = time.time()
        run_target("logKm", df_feat, COL_KM, KM_BRANCHES, enable_catboost=False)
        print(f"[logKm] runtime_sec: {time.time() - t0:.3f}")

    if RUN_LOGVMAX:
        t0 = time.time()
        run_target("logVmax", df_feat, COL_VMAX, V_BRANCHES, enable_catboost=False)
        print(f"[logVmax] runtime_sec: {time.time() - t0:.3f}")

    print(f"\nSaved to: {OUT_DIR}")
    print(f"Total runtime: {time.time() - t_all:.2f} sec")


if __name__ == "__main__":
    main()



[logVmax] n_samples=57 | n_groups=35 | singleton_groups=22
[logVmax] search space ~= 944 configs (before CV folds)
  - SVR_RBF          | best R2=0.4027 | RMSE=1.5560 | sample=full | feature=baseline | k=12 | svd=8 | scaler=standard | time=61.9s
  - NuSVR_RBF        | best R2=0.3876 | RMSE=1.5755 | sample=full | feature=baseline | k=12 | svd=8 | scaler=standard | time=6.5s
  - KNN_distance     | best R2=0.3617 | RMSE=1.5613 | sample=complete_core | feature=metal_env | k=10 | svd=8 | scaler=robust | time=1.6s

[BEST] logVmax | model=SVR_RBF | sample=full | feature=baseline
preprocess: {'k_best_req': 12, 'svd_dim_req': 8, 'scaler_choice': 'standard'}
model_params: {'C': 150, 'gamma': 0.02, 'epsilon': 0.07, '__use_y_scaler__': True}
metrics_oof: {'r2_log_oof': 0.4027337181111341, 'mae_log_oof': 1.2935234697264555, 'rmse_log_oof': 1.5559615562412323, 'pearson_r_oof': 0.6623311975933772, 'pearson_r2_oof': 0.4386826153054773}
[logVmax] runtime_sec: 70.173

Saved to: C:\Users\86158\Desktop\c

In [5]:
# -*- coding: utf-8 -*-
"""
CAT regression v13-final: pure-baseline final sprint for logVmax.

Purpose
-------
Continue the strongest and most stable lane found so far, while keeping the
feature definition clean and comparable:
  - target: log10(Vmax)
  - split: DOI-GroupKFold OOF only
  - sample mode: full
  - feature mode: pure baseline (no external atomic-property table)
  - main model: SVR_RBF

This version intentionally does NOT inject external element-property features.
It only uses:
  1) original numeric columns from the spreadsheet
  2) coarse recoding of shape / surface treatment / dispersion medium

A small NuSVR branch is kept as a nearby reference.
"""

import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_regression
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import NuSVR, SVR

warnings.filterwarnings("ignore")

# =========================
# 0) Paths & config
# =========================
CAT_PATH = Path(r"C:\Users\86158\Desktop\dataiscat.xlsx")  # <-- 改成你的输入文件
SHEET_NAME = 0

OUT_DIR = CAT_PATH.parent / "cat_km_vmax_reg_doi_multistage_out_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_DOI = "data reference doi"
COL_VMAX = "Vmax/μM s-1"

RANDOM_SEED = 42
MAX_FOLDS = 5
MIN_SAMPLES_FOR_RUN = 14
MIN_GROUPS_FOR_RUN = 8
MIN_FREQ_CAT = 2
VAR_THR = 0.0

# Optional switch: if you want to try one light engineered variant, turn this on.
# It still does NOT use any external atomic-property lookup table.
TRY_LIGHT_NUMERIC_DERIVED = True


# =========================
# 1) Basic utils
# =========================
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def coarse_shape(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if "sphere" in x:
        return "sphere"
    if "particle" in x:
        return "particle"
    if "cluster" in x:
        return "cluster"
    if "sheet" in x:
        return "sheet"
    if "rod" in x:
        return "rod"
    if "cube" in x:
        return "cube"
    if "flower" in x:
        return "flower"
    return "other"


def coarse_surface(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if x in {"none", "na", "nan"}:
        return "none"
    if any(k in x for k in ["peg", "pvp", "paa", "pamam", "dspe", "β-cd", "beta-cd", "mpa", "aa"]):
        return "poly_small"
    if "sio2" in x or "silica" in x:
        return "silica"
    if any(k in x for k in ["apo", "ferritin", "histidine", "gsh", "silk", "amp", "nta"]):
        return "bio_ligand"
    return "other"


def coarse_medium(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if x in {"pbs", "phosphate buffer", "pb", "phosphate-buffered saline"}:
        return "phosphate"
    if "tris" in x:
        return "tris"
    if "water" in x:
        return "water"
    if "acetate" in x or "hac-naac" in x:
        return "acetate"
    if "mops" in x:
        return "mops"
    return "other"


# =========================
# 2) Feature engineering (pure baseline)
# =========================
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["shape_coarse"] = df.get("shape", np.nan).map(coarse_shape)
    df["surface_coarse"] = df.get("surface treatment", np.nan).map(coarse_surface)
    df["medium_coarse"] = df.get("dispersion medium", np.nan).map(coarse_medium)

    if TRY_LIGHT_NUMERIC_DERIVED:
        size = pd.to_numeric(df.get("size (nm)"), errors="coerce")
        pH = pd.to_numeric(df.get("pH"), errors="coerce")
        temp = pd.to_numeric(df.get("temperature/oC"), errors="coerce")
        df["size_log1p"] = np.log1p(size)
        df["abs_pH_from_7"] = (pH - 7.0).abs()
        df["size_x_pH"] = size * pH
        df["size_x_temp"] = size * temp
    return df


def get_feature_bundle(df_feat: pd.DataFrame, mode: str):
    baseline_num = [
        "contain O", "contain N", "contain P", "contain S", "contain Si", "contain F", "contain Br",
        "main metal proportion", "main metal number", "main metal valence",
        "minor metal percentage", "minor metal number", "minor metal valence",
        "size (nm)", "pH", "temperature/oC",
    ]
    baseline_cat = ["shape_coarse", "surface_coarse", "medium_coarse"]

    light_plus_num = baseline_num + ["size_log1p", "abs_pH_from_7", "size_x_pH", "size_x_temp"]

    modes = {
        "baseline": (baseline_num, baseline_cat),
        "baseline_plus_light": (light_plus_num, baseline_cat),
    }
    if mode not in modes:
        raise ValueError(mode)

    num_cols, cat_cols = modes[mode]
    num_cols = [c for c in dict.fromkeys(num_cols) if c in df_feat.columns]
    cat_cols = [c for c in dict.fromkeys(cat_cols) if c in df_feat.columns and c not in num_cols]

    X = df_feat[num_cols + cat_cols].copy()
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")
    for c in cat_cols:
        X[c] = clean_text_series(X[c])
    return X, num_cols, cat_cols


# =========================
# 3) Transformers
# =========================
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_freq=2, other_label="__other__"):
        self.min_freq = min_freq
        self.other_label = other_label
        self.keep_levels_ = {}

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        self.keep_levels_ = {}
        for c in X.columns:
            vc = X[c].value_counts(dropna=True)
            keep = vc[vc >= self.min_freq].index.tolist()
            self.keep_levels_[c] = set(keep)
        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for c in X.columns:
            keep = self.keep_levels_.get(c, set())
            X[c] = X[c].where(X[c].isin(keep), self.other_label)
        return X


class ToDense(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if hasattr(X, "toarray"):
            return X.toarray()
        return np.asarray(X)


class SelectKBestSafe(BaseEstimator, TransformerMixin):
    def __init__(self, k=None):
        self.k = k
        self.selector_ = None

    def fit(self, X, y):
        if self.k is None:
            self.selector_ = None
            return self
        n_feat = X.shape[1]
        k_eff = int(min(int(self.k), int(n_feat)))
        if k_eff <= 0:
            self.selector_ = None
            return self
        self.selector_ = SelectKBest(score_func=f_regression, k=k_eff)
        self.selector_.fit(X, y)
        return self

    def transform(self, X):
        if self.selector_ is None:
            return X
        return self.selector_.transform(X)


class TruncatedSVDSafe(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=None, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.svd_ = None

    def fit(self, X, y=None):
        if self.n_components is None:
            self.svd_ = None
            return self
        n_feat = X.shape[1]
        if n_feat <= 1:
            self.svd_ = None
            return self
        n_eff = min(int(self.n_components), max(1, n_feat - 1))
        self.svd_ = TruncatedSVD(n_components=n_eff, random_state=self.random_state)
        self.svd_.fit(X)
        return self

    def transform(self, X):
        if self.svd_ is None:
            return X
        return self.svd_.transform(X)


# =========================
# 4) Sample / target utils
# =========================
def build_target_subset(df_feat: pd.DataFrame, y_col: str, sample_mode: str):
    y_num = pd.to_numeric(df_feat[y_col], errors="coerce")
    mask = y_num.notna() & (y_num > 0)
    if sample_mode != "full":
        raise ValueError(sample_mode)
    df_sub = df_feat[mask].copy()
    if len(df_sub) < MIN_SAMPLES_FOR_RUN:
        return None
    groups = df_sub[COL_DOI].fillna("").astype(str).str.strip().replace("", np.nan)
    if groups.nunique(dropna=True) < MIN_GROUPS_FOR_RUN:
        return None
    return df_sub


def pearsonr_safe(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    if len(a) < 2:
        return np.nan
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def eval_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    r = pearsonr_safe(y_true, y_pred)
    return {
        "r2_log_oof": float(r2_score(y_true, y_pred)),
        "mae_log_oof": float(mean_absolute_error(y_true, y_pred)),
        "rmse_log_oof": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "pearson_r_oof": float(r) if not pd.isna(r) else np.nan,
        "pearson_r2_oof": float(r * r) if not pd.isna(r) else np.nan,
    }


def make_groups(df_unit: pd.DataFrame, col_doi: str):
    groups = df_unit[col_doi].fillna("").astype(str).str.strip()
    miss = groups.eq("")
    groups.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df_unit.index[miss]]
    return groups.values


def groupkfold_oof_predict(X_raw, y_log, groups, pipe_builder):
    uniq_groups = np.unique(groups)
    n_folds = min(MAX_FOLDS, len(uniq_groups))
    if n_folds < 3:
        return None
    gkf = GroupKFold(n_splits=n_folds)
    oof_pred = np.full(len(y_log), np.nan, dtype=float)
    for tr, te in gkf.split(X_raw, y_log, groups=groups):
        Xtr, Xte = X_raw.iloc[tr].copy(), X_raw.iloc[te].copy()
        ytr = y_log[tr]
        pipe = pipe_builder()
        pipe.fit(Xtr, ytr)
        oof_pred[te] = np.asarray(pipe.predict(Xte)).reshape(-1)
    ok = np.isfinite(oof_pred)
    return oof_pred, ok, n_folds


# =========================
# 5) Model / pipeline
# =========================
def instantiate_model(model_name, params):
    if model_name == "SVR_RBF":
        return SVR(kernel="rbf", **params)
    if model_name == "NuSVR_RBF":
        return NuSVR(kernel="rbf", **params)
    raise ValueError(model_name)


def build_preprocess(numeric_cols, categorical_cols):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=True)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("rare", RareCategoryGrouper(min_freq=MIN_FREQ_CAT)),
        ("onehot", oh),
    ])
    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop",
    )


def build_pipeline_for_config(numeric_cols, categorical_cols, k_best, svd_dim, model_name, model_params):
    model_params = model_params.copy()
    use_y_scaler = model_params.pop("__use_y_scaler__", True)

    preprocess = build_preprocess(numeric_cols, categorical_cols)
    steps = [
        ("preprocess", preprocess),
        ("var", VarianceThreshold(threshold=float(VAR_THR))),
        ("todense", ToDense()),
    ]
    if k_best is not None:
        steps.append(("kbest", SelectKBestSafe(k=k_best)))
    if svd_dim is not None:
        steps.append(("svd", TruncatedSVDSafe(n_components=svd_dim, random_state=RANDOM_SEED)))
    steps.append(("scaler", StandardScaler()))
    steps.append(("model", instantiate_model(model_name, model_params)))

    reg = Pipeline(steps)
    if use_y_scaler:
        return TransformedTargetRegressor(regressor=reg, transformer=StandardScaler())
    return reg


# =========================
# 6) Search spaces
# =========================
PREPROCESS_CANDIDATES = [
    {"sample_mode": "full", "feature_mode": "baseline", "k_best": 12, "svd_dim": 8},
    {"sample_mode": "full", "feature_mode": "baseline", "k_best": 13, "svd_dim": 8},
    {"sample_mode": "full", "feature_mode": "baseline", "k_best": 12, "svd_dim": 9},
]
if TRY_LIGHT_NUMERIC_DERIVED:
    PREPROCESS_CANDIDATES.extend([
        {"sample_mode": "full", "feature_mode": "baseline_plus_light", "k_best": 12, "svd_dim": 8},
        {"sample_mode": "full", "feature_mode": "baseline_plus_light", "k_best": 13, "svd_dim": 8},
    ])

BRANCHES = {
    "SVR_RBF": {
        "preprocess": PREPROCESS_CANDIDATES,
        "params": [
            {"C": C, "gamma": g, "epsilon": e, "__use_y_scaler__": True}
            for C in [120, 150, 180, 220]
            for g in [0.018, 0.020, 0.022]
            for e in [0.06, 0.07, 0.08]
        ],
    },
    "NuSVR_RBF": {
        "preprocess": [
            {"sample_mode": "full", "feature_mode": "baseline", "k_best": 12, "svd_dim": 8},
            {"sample_mode": "full", "feature_mode": "baseline_plus_light", "k_best": 12, "svd_dim": 8},
        ] if TRY_LIGHT_NUMERIC_DERIVED else [
            {"sample_mode": "full", "feature_mode": "baseline", "k_best": 12, "svd_dim": 8},
        ],
        "params": [
            {"nu": nu, "C": C, "gamma": g, "__use_y_scaler__": True}
            for nu in [0.45, 0.55, 0.65]
            for C in [120, 150, 180]
            for g in [0.018, 0.020, 0.022]
        ],
    },
}


# =========================
# 7) Runner
# =========================
def run_target(target_name, df_feat_all, y_col, branches):
    print()
    y_num_all = pd.to_numeric(df_feat_all[y_col], errors="coerce")
    df_base = df_feat_all[y_num_all.notna() & (y_num_all > 0)].copy()
    groups_base = make_groups(df_base, COL_DOI)
    print(f"[{target_name}] n_samples={len(df_base)} | n_groups={pd.Series(groups_base).nunique()} | singleton_groups={pd.Series(groups_base).value_counts().eq(1).sum()}")

    total_configs = sum(len(cfg["preprocess"]) * len(cfg["params"]) for cfg in branches.values())
    print(f"[{target_name}] search space ~= {total_configs} configs (before CV folds)")

    all_rows = []
    best_overall = None
    best_payload = None

    for model_name, cfg in branches.items():
        t0 = time.time()
        best_row_model = None
        best_payload_model = None

        for pre in cfg["preprocess"]:
            df_sub = build_target_subset(df_feat_all, y_col, pre["sample_mode"])
            if df_sub is None:
                continue
            y_log = np.log10(pd.to_numeric(df_sub[y_col], errors="coerce").astype(float).values)
            groups = make_groups(df_sub, COL_DOI)
            X_raw, numeric_cols, categorical_cols = get_feature_bundle(df_sub, pre["feature_mode"])
            if X_raw.shape[1] < 3:
                continue

            for model_params in cfg["params"]:
                def builder(mp=model_params, pp=pre):
                    return build_pipeline_for_config(
                        numeric_cols, categorical_cols,
                        k_best=pp["k_best"],
                        svd_dim=pp["svd_dim"],
                        model_name=model_name,
                        model_params=mp,
                    )

                out = groupkfold_oof_predict(X_raw, y_log, groups, builder)
                if out is None:
                    continue
                pred, ok, n_folds = out
                m = eval_metrics(y_log[ok], pred[ok])
                row = {
                    "target": target_name,
                    "model": model_name,
                    "sample_mode": pre["sample_mode"],
                    "feature_mode": pre["feature_mode"],
                    "k_best_req": pre["k_best"],
                    "svd_dim_req": pre["svd_dim"],
                    "scaler_choice": "standard",
                    "n_samples": len(df_sub),
                    "n_groups": pd.Series(groups).nunique(),
                    "n_folds": n_folds,
                    "model_params": json.dumps(model_params, ensure_ascii=False),
                    **m,
                }
                all_rows.append(row)

                key = (row["r2_log_oof"], -row["rmse_log_oof"])
                if (best_row_model is None) or (key > (best_row_model["r2_log_oof"], -best_row_model["rmse_log_oof"])):
                    best_row_model = row
                    best_payload_model = (df_sub.copy(), X_raw.copy(), y_log.copy(), groups.copy(), numeric_cols[:], categorical_cols[:], pre.copy(), model_params.copy(), model_name)

        elapsed = time.time() - t0
        if best_row_model is not None:
            print(
                f"  - {model_name:<16} | best R2={best_row_model['r2_log_oof']:.4f} | RMSE={best_row_model['rmse_log_oof']:.4f} | "
                f"sample={best_row_model['sample_mode']} | feature={best_row_model['feature_mode']} | "
                f"k={best_row_model['k_best_req']} | svd={best_row_model['svd_dim_req']} | scaler=standard | time={elapsed:.1f}s"
            )
            key = (best_row_model["r2_log_oof"], -best_row_model["rmse_log_oof"])
            if (best_overall is None) or (key > (best_overall["r2_log_oof"], -best_overall["rmse_log_oof"])):
                best_overall = best_row_model
                best_payload = best_payload_model

    if best_overall is None:
        raise RuntimeError(f"{target_name}: no valid result")

    res = pd.DataFrame(all_rows).sort_values(["r2_log_oof", "rmse_log_oof"], ascending=[False, True])
    res.to_csv(OUT_DIR / f"{target_name}_search_summary.csv", index=False, encoding="utf-8-sig")

    df_sub, X_raw, y_log, groups, numeric_cols, categorical_cols, pre, model_params, model_name = best_payload

    def best_builder():
        return build_pipeline_for_config(
            numeric_cols, categorical_cols,
            k_best=pre["k_best"],
            svd_dim=pre["svd_dim"],
            model_name=model_name,
            model_params=model_params,
        )

    oof, ok, n_folds = groupkfold_oof_predict(X_raw, y_log, groups, best_builder)
    pd.DataFrame({
        "row_index": df_sub.index.values,
        "doi_group": groups,
        "y_log_true": y_log,
        "y_log_oof_pred": oof,
    }).to_csv(OUT_DIR / f"{target_name}_oof_predictions_groupkfold.csv", index=False, encoding="utf-8-sig")

    final_pipe = best_builder()
    final_pipe.fit(X_raw, y_log)
    joblib.dump({
        "target": target_name,
        "pipeline": final_pipe,
    }, OUT_DIR / f"{target_name}_best_model_refit_full.joblib")

    best_config = {
        "target": target_name,
        "y_col": y_col,
        "target_transform": "log10",
        "split": "GroupKFold by DOI",
        "best": {
            "model": model_name,
            "sample_mode": pre["sample_mode"],
            "feature_mode": pre["feature_mode"],
            "preprocess": {
                "k_best_req": pre["k_best"],
                "svd_dim_req": pre["svd_dim"],
                "scaler_choice": "standard",
            },
            "model_params": model_params,
            "metrics_oof": eval_metrics(y_log[ok], oof[ok]),
        }
    }
    with open(OUT_DIR / f"{target_name}_best_config.json", "w", encoding="utf-8") as f:
        json.dump(best_config, f, ensure_ascii=False, indent=2)

    print("\n" + "=" * 72)
    print(f"[BEST] {target_name} | model={model_name} | sample={pre['sample_mode']} | feature={pre['feature_mode']}")
    print("preprocess:", best_config["best"]["preprocess"])
    print("model_params:", model_params)
    print("metrics_oof:", best_config["best"]["metrics_oof"])
    print("=" * 72)
    return best_config


# =========================
# 8) Main
# =========================
def main():
    t_all = time.time()
    df = pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME)
    df = normalize_columns(df)

    for c in [COL_DOI, COL_VMAX]:
        if c not in df.columns:
            raise KeyError(f"Required column not found: {c}")

    df_feat = engineer_features(df)

    t0 = time.time()
    run_target("logVmax", df_feat, COL_VMAX, BRANCHES)
    print(f"[logVmax] runtime_sec: {time.time() - t0:.3f}")

    print(f"\nSaved to: {OUT_DIR}")
    print(f"Total runtime: {time.time() - t_all:.2f} sec")


if __name__ == "__main__":
    main()



[logVmax] n_samples=57 | n_groups=35 | singleton_groups=22
[logVmax] search space ~= 234 configs (before CV folds)
  - SVR_RBF          | best R2=0.4027 | RMSE=1.5560 | sample=full | feature=baseline | k=12 | svd=8 | scaler=standard | time=13.5s
  - NuSVR_RBF        | best R2=0.3902 | RMSE=1.5722 | sample=full | feature=baseline | k=12 | svd=8 | scaler=standard | time=5.2s

[BEST] logVmax | model=SVR_RBF | sample=full | feature=baseline
preprocess: {'k_best_req': 12, 'svd_dim_req': 8, 'scaler_choice': 'standard'}
model_params: {'C': 150, 'gamma': 0.02, 'epsilon': 0.07, '__use_y_scaler__': True}
metrics_oof: {'r2_log_oof': 0.4027337181111341, 'mae_log_oof': 1.2935234697264555, 'rmse_log_oof': 1.5559615562412323, 'pearson_r_oof': 0.6623311975933772, 'pearson_r2_oof': 0.4386826153054773}
[logVmax] runtime_sec: 18.753

Saved to: C:\Users\86158\Desktop\cat_km_vmax_reg_doi_multistage_out_v2
Total runtime: 19.52 sec


In [ ]:
CAT_PATH = Path(r"C:\Users\86158\Desktop\dataiscat.xlsx")  # <-- 改成你的输入文件
SHEET_NAME = 0

OUT_DIR = CAT_PATH.parent / "cat_km_vmax_reg_doi_multistage_out_v2"

In [1]:
# -*- coding: utf-8 -*-
"""
CAT KcatHigh：标签方案 A+B 同时搜索（只改标签，不改特征）
A: 单阈值 (single_q)  q ∈ [0.30, 0.70] step=0.05
B: 两端切分 (extreme_q) 丢中间  q ∈ {0.30, 0.35, 0.40}

严格 group split + 12模型不变（若安装 xgboost）。
额外约束：
- 标签样本数至少 >= MIN_LABEL_SAMPLES
- 只有 n_splits > MIN_VALID_SPLITS 的模型才参与该设置下的排名

输出：
- <task>_A_singleq_grid.csv
- <task>_B_extremeq_grid.csv
- <task>_overall_best.csv
- <task>_best_setting_model_summary.csv
- <task>_best_setting_labeled_data.csv
- <task>_best_setting_refit_full.joblib
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.metrics import roc_auc_score, cohen_kappa_score, matthews_corrcoef, accuracy_score

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


# =========================
# 0) 路径与配置
# =========================
CAT_PATH = r"C:\Users\86158\Desktop\dataiscat.xlsx"
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\cat_kcat_cls_threshold_tuning_AB")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_DOI = "data reference doi"
COL_KCAT = "Kcat/s-1"

# 严格 split（主体不变）
TEST_SIZE = 0.2
N_SPLITS = 50
SPLIT_SEED = 42

# 小样本放宽版门槛（适配 Kcat）
MIN_LABEL_SAMPLES = 10
MIN_TEST_SAMPLES = 4
MIN_TRAIN_CLASS_COUNT = 2
MIN_TEST_CLASS_COUNT = 1

# 只保留 n_splits > 10 的模型
MIN_VALID_SPLITS = 10

# 搜索参数
QS_SINGLE = np.round(np.arange(0.30, 0.70 + 1e-9, 0.05), 2).tolist()
QS_EXTREME = [0.30, 0.35, 0.40]


# =========================
# 1) 特征定义（不改）
# =========================
FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate "
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = [
    "shape", "surface treatment", "dispersion medium", "Substrate "
]


# =========================
# 2) 预处理（不改）
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    X = df_raw[FEATURE_COLS].copy()

    for c in NUMERIC_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in CATEGORICAL_COLS:
        X[c] = clean_text_series(X[c])

    return X


def make_preprocess():
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, NUMERIC_COLS),
            ("cat", cat_pipe, CATEGORICAL_COLS),
        ],
        remainder="drop"
    )


# =========================
# 3) 模型（不改）
# =========================
def get_models():
    models = {
        "LogReg": LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42
        ),
        "LinearSVC": LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=42
        ),
        "SVC_RBF": SVC(
            C=1.0,
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=42
        ),
        "KNN": KNeighborsClassifier(
            n_neighbors=5,
            weights="distance"
        ),
        "GaussianNB": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42
        ),
        "RF": RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=800,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ),
        "HistGB": HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            max_depth=4,
            random_state=42
        ),
        "AdaBoost": AdaBoostClassifier(
            n_estimators=300,
            learning_rate=0.05,
            random_state=42
        ),
    }

    if HAS_XGBOOST:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
            verbosity=0
        )
    return models


def needs_scaling(model_name):
    return model_name in {"LogReg", "LinearSVC", "SVC_RBF", "KNN", "GaussianNB"}


def build_model_pipe(model_name, model, preprocess):
    steps = [("preprocess", preprocess)]
    if needs_scaling(model_name):
        steps.append(("scaler", StandardScaler(with_mean=True)))
    steps.append(("model", model))
    return Pipeline(steps)


def score_to_prob(pipe, X):
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X)[:, 1]
    if hasattr(pipe, "decision_function"):
        z = np.asarray(pipe.decision_function(X), dtype=float)
        return 1.0 / (1.0 + np.exp(-z))
    return np.asarray(pipe.predict(X), dtype=float)


# =========================
# 4) 标签（A/B）
# =========================
def build_labels_log10(values: pd.Series, task: str, scheme: str, q: float):
    v = pd.to_numeric(values, errors="coerce")
    m = v.notna() & (v > 0)
    vals = np.log10(v.loc[m].astype(float).values)

    if len(vals) == 0:
        return np.array([], dtype=int), np.array([], dtype=int), {"scheme": scheme, "q": q}

    if scheme == "single_q":
        if task == "KmLow":
            thr = float(np.quantile(vals, q))
            y = (vals <= thr).astype(int)
            keep = np.ones_like(y, dtype=bool)
            meta = {"scheme": scheme, "q": q, "thr": thr}
        else:  # KcatHigh 走这里
            thr = float(np.quantile(vals, 1 - q))
            y = (vals >= thr).astype(int)
            keep = np.ones_like(y, dtype=bool)
            meta = {"scheme": scheme, "q": q, "thr": thr}

    elif scheme == "extreme_q":
        lo = float(np.quantile(vals, q))
        hi = float(np.quantile(vals, 1 - q))
        keep = (vals <= lo) | (vals >= hi)
        vals_k = vals[keep]

        if task == "KmLow":
            y = (vals_k <= lo).astype(int)
        else:  # KcatHigh 走这里
            y = (vals_k >= hi).astype(int)

        meta = {
            "scheme": scheme,
            "q": q,
            "lo": lo,
            "hi": hi,
            "dropped_mid": float(1 - 2 * q)
        }
    else:
        raise ValueError(scheme)

    idx_all = np.where(m.values)[0]
    idx_keep = idx_all[keep]
    return idx_keep, y.astype(int), meta


# =========================
# 5) 评估
# =========================
def evaluate_one_setting(task_name, X_all, groups_all, idx_keep, y, preprocess):
    X = X_all.iloc[idx_keep].copy()
    groups = groups_all[idx_keep].copy()
    y = np.asarray(y, int)

    splitter = GroupShuffleSplit(
        n_splits=N_SPLITS,
        test_size=TEST_SIZE,
        random_state=SPLIT_SEED
    )

    models = get_models()
    rows = []

    for model_name, model in models.items():
        for split_id, (tr, te) in enumerate(splitter.split(X, y, groups=groups), start=1):
            if len(te) < MIN_TEST_SAMPLES:
                continue

            ytr, yte = y[tr], y[te]

            if (np.sum(ytr == 0) < MIN_TRAIN_CLASS_COUNT) or (np.sum(ytr == 1) < MIN_TRAIN_CLASS_COUNT):
                continue
            if (np.sum(yte == 0) < MIN_TEST_CLASS_COUNT) or (np.sum(yte == 1) < MIN_TEST_CLASS_COUNT):
                continue

            try:
                pipe = build_model_pipe(model_name, model, preprocess)
                pipe.fit(X.iloc[tr], ytr)

                pte = score_to_prob(pipe, X.iloc[te])
                pred = (pte >= 0.5).astype(int)

                rows.append({
                    "model": model_name,
                    "split": split_id,
                    "auc": float(roc_auc_score(yte, pte)),
                    "kappa": float(cohen_kappa_score(yte, pred)),
                    "mcc": float(matthews_corrcoef(yte, pred)),
                    "acc": float(accuracy_score(yte, pred)),
                })
            except Exception:
                rows.append({
                    "model": model_name,
                    "split": split_id,
                    "auc": np.nan,
                    "kappa": np.nan,
                    "mcc": np.nan,
                    "acc": np.nan
                })

    res = pd.DataFrame(rows)
    res_ok = res.dropna(subset=["auc", "kappa", "mcc", "acc"]).copy()

    if res_ok.empty:
        return pd.DataFrame(), pd.DataFrame()

    summ = (
        res_ok.groupby("model", as_index=False)
        .agg(
            auc_mean=("auc", "mean"),
            auc_median=("auc", "median"),
            auc_std=("auc", "std"),
            auc_max=("auc", "max"),
            kappa_mean=("kappa", "mean"),
            mcc_mean=("mcc", "mean"),
            acc_mean=("acc", "mean"),
            n_splits=("split", "count")
        )
    )

    # 只保留 n_splits > MIN_VALID_SPLITS 的模型
    summ = summ[summ["n_splits"] > MIN_VALID_SPLITS].copy()

    if summ.empty:
        return res_ok, pd.DataFrame()

    summ = (
        summ.sort_values(
            by=["auc_mean", "auc_median", "mcc_mean", "kappa_mean", "acc_mean", "auc_max"],
            ascending=[False, False, False, False, False, False]
        )
        .reset_index(drop=True)
    )

    return res_ok, summ


# =========================
# 6) 搜索 A/B 并输出 overall best
# =========================
def empty_result_df():
    return pd.DataFrame(columns=[
        "scheme", "q", "thr", "lo", "hi", "dropped_mid",
        "n_samples", "n_groups", "best_model",
        "best_auc_mean", "best_auc_median", "best_n_splits"
    ])


def run_task(task_name, df, X_all, groups_all, preprocess, col_target):
    results_A = []
    results_B = []

    # A: single_q
    for q in QS_SINGLE:
        idx_keep, y, meta = build_labels_log10(df[col_target], task_name, "single_q", q)

        if len(y) < MIN_LABEL_SAMPLES or len(np.unique(y)) < 2:
            print(f"[{task_name}][single_q={q}] skipped | n={len(y)} | unique_y={len(np.unique(y))}")
            continue

        _, summ = evaluate_one_setting(task_name, X_all, groups_all, idx_keep, y, preprocess)

        if summ.empty:
            print(f"[{task_name}][single_q={q}] no model with n_splits > {MIN_VALID_SPLITS}")
            continue

        best = summ.iloc[0].to_dict()
        results_A.append({
            **meta,
            "n_samples": int(len(y)),
            "n_groups": int(len(np.unique(groups_all[idx_keep]))),
            "best_model": best["model"],
            "best_auc_mean": float(best["auc_mean"]),
            "best_auc_median": float(best["auc_median"]),
            "best_n_splits": int(best["n_splits"]),
        })

    # B: extreme_q
    for q in QS_EXTREME:
        idx_keep, y, meta = build_labels_log10(df[col_target], task_name, "extreme_q", q)

        if len(y) < MIN_LABEL_SAMPLES or len(np.unique(y)) < 2:
            print(f"[{task_name}][extreme_q={q}] skipped | n={len(y)} | unique_y={len(np.unique(y))}")
            continue

        _, summ = evaluate_one_setting(task_name, X_all, groups_all, idx_keep, y, preprocess)

        if summ.empty:
            print(f"[{task_name}][extreme_q={q}] no model with n_splits > {MIN_VALID_SPLITS}")
            continue

        best = summ.iloc[0].to_dict()
        results_B.append({
            **meta,
            "n_samples": int(len(y)),
            "n_groups": int(len(np.unique(groups_all[idx_keep]))),
            "best_model": best["model"],
            "best_auc_mean": float(best["auc_mean"]),
            "best_auc_median": float(best["auc_median"]),
            "best_n_splits": int(best["n_splits"]),
        })

    if results_A:
        dfA = pd.DataFrame(results_A).sort_values(
            by=["best_auc_mean", "best_auc_median", "best_n_splits"],
            ascending=[False, False, False]
        ).reset_index(drop=True)
    else:
        dfA = empty_result_df()

    if results_B:
        dfB = pd.DataFrame(results_B).sort_values(
            by=["best_auc_mean", "best_auc_median", "best_n_splits"],
            ascending=[False, False, False]
        ).reset_index(drop=True)
    else:
        dfB = empty_result_df()

    dfA.to_csv(OUT_DIR / f"{task_name}_A_singleq_grid.csv", index=False, encoding="utf-8-sig")
    dfB.to_csv(OUT_DIR / f"{task_name}_B_extremeq_grid.csv", index=False, encoding="utf-8-sig")

    cand = []
    if not dfA.empty:
        cand.append(dfA.iloc[0].to_dict())
    if not dfB.empty:
        cand.append(dfB.iloc[0].to_dict())

    if not cand:
        print(f"[{task_name}] 没有可用方案")
        return

    overall = pd.DataFrame(cand).sort_values(
        by=["best_auc_mean", "best_auc_median", "best_n_splits"],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    overall.to_csv(OUT_DIR / f"{task_name}_overall_best.csv", index=False, encoding="utf-8-sig")

    best_meta = overall.iloc[0].to_dict()
    print("\n" + "=" * 70)
    print(f"[{task_name}] OVERALL BEST (from A/B)")
    print("=" * 70)
    print(best_meta)

    # 用 overall best 重新生成标签
    scheme = best_meta["scheme"]
    q = float(best_meta["q"])
    idx_keep, y, meta = build_labels_log10(df[col_target], task_name, scheme, q)

    # 保存 best setting 的模型汇总
    _, best_summ = evaluate_one_setting(task_name, X_all, groups_all, idx_keep, y, preprocess)
    if not best_summ.empty:
        best_summ.to_csv(
            OUT_DIR / f"{task_name}_best_setting_model_summary.csv",
            index=False,
            encoding="utf-8-sig"
        )

    # 全量重训并保存
    models = get_models()
    best_model_name = best_meta["best_model"]
    pipe = build_model_pipe(best_model_name, models[best_model_name], preprocess)

    X_use = X_all.iloc[idx_keep].copy()
    pipe.fit(X_use, y)

    df_use = df.iloc[idx_keep].copy()
    df_use["y"] = y
    df_use.to_csv(
        OUT_DIR / f"{task_name}_best_setting_labeled_data.csv",
        index=False,
        encoding="utf-8-sig"
    )

    joblib.dump(
        {
            "task": task_name,
            "labeling_meta": meta,
            "best_model": best_model_name,
            "pipeline": pipe
        },
        OUT_DIR / f"{task_name}_best_setting_refit_full.joblib"
    )


def main():
    df = pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME)

    groups_all = df[COL_DOI].fillna("").astype(str).str.strip()
    miss = groups_all.eq("")
    groups_all.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df.index[miss]]
    groups_all = groups_all.values

    X_all = prepare_features(df)
    preprocess = make_preprocess()

    run_task("KcatHigh", df, X_all, groups_all, preprocess, COL_KCAT)

    print("\nSaved to:", OUT_DIR)


if __name__ == "__main__":
    main()

[KcatHigh][single_q=0.3] no model with n_splits > 10
[KcatHigh][single_q=0.7] no model with n_splits > 10
[KcatHigh][extreme_q=0.3] skipped | n=8 | unique_y=2

[KcatHigh] OVERALL BEST (from A/B)
{'scheme': 'single_q', 'q': 0.45, 'thr': 3.3198654689156797, 'n_samples': 13, 'n_groups': 8, 'best_model': 'KNN', 'best_auc_mean': 0.880952380952381, 'best_auc_median': 1.0, 'best_n_splits': 14, 'lo': nan, 'hi': nan, 'dropped_mid': nan}

Saved to: C:\Users\86158\Desktop\cat_kcat_cls_threshold_tuning_AB


In [22]:
# -*- coding: utf-8 -*-
"""
CAT KcatHigh best setting (from threshold search):
- scheme = single_q
- q = 0.45
- best_model = KNN

This script does two things:
1) Reproduce evaluation plots for the best model under the same split rules:
   - ROC mean ± std band
   - PR mean ± std band
   - Mean normalized confusion matrix
   - all valid split metrics CSV
2) Refit the best model on the full labeled subset and run SHAP analysis:
   - SHAP beeswarm
   - SHAP bar
   - mean(|SHAP|) by transformed feature
   - aggregated mean(|SHAP|) by original feature

Notes:
- Target column is CAT Kcat/s-1
- Labeling follows your best setting: KcatHigh + single_q(q=0.45)
- Split rules align with your latest Kcat script:
    TEST_SIZE = 0.2
    N_SPLITS = 50
    SPLIT_SEED = 42
    MIN_TEST_SAMPLES = 3
    MIN_TRAIN_CLASS_COUNT = 2
    MIN_TEST_CLASS_COUNT = 1
- SHAP for KNN uses KernelExplainer on transformed+scaled features
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    roc_auc_score, cohen_kappa_score, matthews_corrcoef, accuracy_score,
    roc_curve, precision_recall_curve, average_precision_score
)

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False


# =========================
# 0) Paths & config
# =========================
CAT_PATH = r"C:\Users\86158\Desktop\dataiscat.xlsx"
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\cat_kcat_best_knn_eval_shap")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = OUT_DIR / "plots_pretty"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

COL_DOI = "data reference doi"
COL_KCAT = "Kcat/s-1"

# Best setting from your search result
TASK_NAME = "KcatHigh"
BEST_SCHEME = "single_q"
BEST_Q = 0.45
EXPECTED_THR = 3.3198654689156797  # for sanity print only
BEST_MODEL_NAME = "KNN"
THR_PRED = 0.5

# Same split logic as your latest Kcat script
TEST_SIZE = 0.2
N_SPLITS = 50
SPLIT_SEED = 42
MIN_TEST_SAMPLES = 3
MIN_TRAIN_CLASS_COUNT = 2
MIN_TEST_CLASS_COUNT = 1

# Plotting
FPR_GRID = np.linspace(0, 1, 201)
REC_GRID = np.linspace(0, 1, 201)

# Kernel SHAP settings for KNN
RANDOM_SEED = 42
KNN_BACKGROUND_SIZE = 8
KNN_NSAMPLES = 100


# =========================
# 1) Feature definition (same as before)
# =========================
FEATURE_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate "
]

NUMERIC_COLS = [
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
]

CATEGORICAL_COLS = [
    "shape", "surface treatment", "dispersion medium", "Substrate "
]


# =========================
# 2) Clean / preprocess
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw: pd.DataFrame) -> pd.DataFrame:
    X = df_raw[FEATURE_COLS].copy()

    for c in NUMERIC_COLS:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in CATEGORICAL_COLS:
        X[c] = clean_text_series(X[c])

    return X


def make_preprocess():
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, NUMERIC_COLS),
            ("cat", cat_pipe, CATEGORICAL_COLS),
        ],
        remainder="drop"
    )


# =========================
# 3) Labeling (same logic)
# =========================
def build_labels_log10(values: pd.Series, task: str, scheme: str, q: float):
    v = pd.to_numeric(values, errors="coerce")
    m = v.notna() & (v > 0)
    vals = np.log10(v.loc[m].astype(float).values)

    if len(vals) == 0:
        return np.array([], dtype=int), np.array([], dtype=int), {"scheme": scheme, "q": q}

    if scheme == "single_q":
        if task == "KmLow":
            thr = float(np.quantile(vals, q))
            y = (vals <= thr).astype(int)
            keep = np.ones_like(y, dtype=bool)
            meta = {"scheme": scheme, "q": q, "thr": thr}
        else:
            thr = float(np.quantile(vals, 1 - q))
            y = (vals >= thr).astype(int)
            keep = np.ones_like(y, dtype=bool)
            meta = {"scheme": scheme, "q": q, "thr": thr}

    elif scheme == "extreme_q":
        lo = float(np.quantile(vals, q))
        hi = float(np.quantile(vals, 1 - q))
        keep = (vals <= lo) | (vals >= hi)
        vals_k = vals[keep]
        if task == "KmLow":
            y = (vals_k <= lo).astype(int)
        else:
            y = (vals_k >= hi).astype(int)
        meta = {"scheme": scheme, "q": q, "lo": lo, "hi": hi}
    else:
        raise ValueError(scheme)

    idx_all = np.where(m.values)[0]
    idx_keep = idx_all[keep]
    return idx_keep, y.astype(int), meta


# =========================
# 4) Model
# =========================
def build_knn_pipe(preprocess):
    return Pipeline([
        ("preprocess", preprocess),
        ("scaler", StandardScaler(with_mean=True)),
        ("model", KNeighborsClassifier(n_neighbors=5, weights="distance"))
    ])


def score_to_prob(pipe, X):
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X)[:, 1]
    if hasattr(pipe, "decision_function"):
        z = np.asarray(pipe.decision_function(X), dtype=float)
        return 1.0 / (1.0 + np.exp(-z))
    return np.asarray(pipe.predict(X), dtype=float)


# =========================
# 5) Split iterator (same filter rules)
# =========================
def iter_valid_splits(splitter, X, y, groups):
    for split_id, (tr, te) in enumerate(splitter.split(X, y, groups=groups), start=1):
        if len(te) < MIN_TEST_SAMPLES:
            continue
        ytr, yte = y[tr], y[te]
        if (np.sum(ytr == 0) < MIN_TRAIN_CLASS_COUNT) or (np.sum(ytr == 1) < MIN_TRAIN_CLASS_COUNT):
            continue
        if (np.sum(yte == 0) < MIN_TEST_CLASS_COUNT) or (np.sum(yte == 1) < MIN_TEST_CLASS_COUNT):
            continue
        yield split_id, tr, te


# =========================
# 6) Pretty plots
# =========================
def save_fig(fig, path: Path):
    fig.tight_layout()
    fig.savefig(path, dpi=260)
    plt.close(fig)


def set_clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)


def plot_mean_roc_band(fprs_list, tprs_list, aucs, title, out_path):
    tprs = np.array([np.interp(FPR_GRID, fpr, tpr) for fpr, tpr in zip(fprs_list, tprs_list)])
    tprs[:, 0] = 0.0
    mean_tpr = tprs.mean(axis=0)
    std_tpr = tprs.std(axis=0)

    fig = plt.figure(figsize=(6.6, 5.2))
    ax = fig.add_subplot(111)
    ax.plot(FPR_GRID, mean_tpr, linewidth=2.6)
    ax.fill_between(
        FPR_GRID,
        np.clip(mean_tpr - std_tpr, 0, 1),
        np.clip(mean_tpr + std_tpr, 0, 1),
        alpha=0.18,
    )
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)

    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(title)
    ax.text(
        0.98, 0.02,
        f"AUC_mean={np.mean(aucs):.6f}\nAUC_std ={np.std(aucs):.6f}\n(n_splits={len(aucs)})",
        ha="right", va="bottom", transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.85, edgecolor="none")
    )
    set_clean_axes(ax)
    save_fig(fig, out_path)


def plot_mean_pr_band(precs_list, recs_list, aps, title, out_path):
    prec_grid_list = []
    for prec, rec in zip(precs_list, recs_list):
        order = np.argsort(rec)
        rec_s = rec[order]
        prec_s = prec[order]
        prec_interp = np.interp(REC_GRID, rec_s, prec_s)
        prec_grid_list.append(prec_interp)

    prec_grid = np.array(prec_grid_list)
    mean_prec = prec_grid.mean(axis=0)
    std_prec = prec_grid.std(axis=0)

    fig = plt.figure(figsize=(6.6, 5.2))
    ax = fig.add_subplot(111)
    ax.plot(REC_GRID, mean_prec, linewidth=2.6)
    ax.fill_between(
        REC_GRID,
        np.clip(mean_prec - std_prec, 0, 1),
        np.clip(mean_prec + std_prec, 0, 1),
        alpha=0.18,
    )

    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(title)
    ax.text(
        0.98, 0.02,
        f"AP_mean={np.mean(aps):.6f}\nAP_std ={np.std(aps):.6f}\n(n_splits={len(aps)})",
        ha="right", va="bottom", transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", alpha=0.85, edgecolor="none")
    )
    set_clean_axes(ax)
    save_fig(fig, out_path)


def confusion_counts(y_true, y_pred):
    cm = np.zeros((2, 2), dtype=float)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1.0
    return cm


def confusion_norm_true(y_true, y_pred):
    cm = confusion_counts(y_true, y_pred)
    row_sum = cm.sum(axis=1, keepdims=True)
    return np.divide(cm, np.maximum(row_sum, 1.0))


def plot_cm_single_hue(cm, title, out_path, normalized=True):
    fig = plt.figure(figsize=(5.6, 5.0))
    ax = fig.add_subplot(111)

    vmax = 1.0 if normalized else (np.max(cm) if np.max(cm) > 0 else 1.0)
    im = ax.imshow(cm, cmap="Blues", vmin=0.0, vmax=vmax)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred 0", "Pred 1"])
    ax.set_yticklabels(["True 0", "True 1"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)

    fmt = ".2f" if normalized else "d"
    for i in range(2):
        for j in range(2):
            val = cm[i, j]
            txt = format(val, fmt)
            ax.text(
                j, i, txt,
                ha="center", va="center",
                color="white" if (val / vmax) > 0.55 else "black",
                fontsize=11, fontweight="bold"
            )

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    set_clean_axes(ax)
    save_fig(fig, out_path)


# =========================
# 7) SHAP helpers
# =========================
def normalize_shap_output(shap_values):
    if isinstance(shap_values, list):
        if len(shap_values) == 2:
            return np.asarray(shap_values[1])
        return np.asarray(shap_values[0])

    arr = np.asarray(shap_values)
    if arr.ndim == 3 and arr.shape[-1] == 2:
        return arr[:, :, 1]
    return arr


def base_feature_from_transformed(name: str):
    n = str(name)

    if "__" in n:
        n = n.split("__", 1)[1]

    n = n.replace("missingindicator_", "")

    for c in CATEGORICAL_COLS:
        if n == c or n.startswith(c + "_"):
            return c

    for c in NUMERIC_COLS:
        if n == c:
            return c

    return n.split("_")[0]


def save_shap_outputs(shap_values_2d, X_matrix, feature_names, prefix):
    mean_abs = np.mean(np.abs(shap_values_2d), axis=0)

    shap_df = pd.DataFrame({
        "feature_transformed": feature_names,
        "mean_abs_shap": mean_abs
    }).sort_values("mean_abs_shap", ascending=False)

    shap_df.to_csv(
        OUT_DIR / f"{prefix}_shap_mean_abs_per_transformed_feature.csv",
        index=False, encoding="utf-8-sig"
    )

    shap_df["feature_original"] = shap_df["feature_transformed"].apply(base_feature_from_transformed)
    agg = (
        shap_df.groupby("feature_original", as_index=False)["mean_abs_shap"]
        .sum()
        .sort_values("mean_abs_shap", ascending=False)
    )

    agg.to_csv(
        OUT_DIR / f"{prefix}_shap_mean_abs_agg_by_original_feature.csv",
        index=False, encoding="utf-8-sig"
    )

    if HAS_SHAP:
        plt.figure()
        shap.summary_plot(
            shap_values_2d,
            X_matrix,
            feature_names=feature_names,
            show=False,
            max_display=20
        )
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{prefix}_shap_summary_beeswarm.png", dpi=200)
        plt.close()

        plt.figure()
        shap.summary_plot(
            shap_values_2d,
            X_matrix,
            feature_names=feature_names,
            plot_type="bar",
            show=False,
            max_display=20
        )
        plt.tight_layout()
        plt.savefig(OUT_DIR / f"{prefix}_shap_summary_bar.png", dpi=200)
        plt.close()


# =========================
# 8) Main
# =========================
def main():
    df = pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME)

    needed = FEATURE_COLS + [COL_DOI, COL_KCAT]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # groups = DOI (missing -> pseudo DOI)
    groups_all = df[COL_DOI].fillna("").astype(str).str.strip()
    miss = groups_all.eq("")
    groups_all.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df.index[miss]]
    groups_all = groups_all.values

    # labels under best setting
    idx_keep, y, meta = build_labels_log10(df[COL_KCAT], TASK_NAME, BEST_SCHEME, BEST_Q)
    df_use = df.iloc[idx_keep].copy()
    X_use = prepare_features(df_use)
    groups_use = groups_all[idx_keep]
    y = np.asarray(y, dtype=int)

    print("=" * 72)
    print(f"[{TASK_NAME}] Best setting reproduction")
    print("=" * 72)
    print(meta)
    print(f"Expected thr from search = {EXPECTED_THR:.12f}")
    print(f"n_samples={len(y)} | pos(1)={(y==1).sum()} | neg(0)={(y==0).sum()} | groups={len(np.unique(groups_use))}")

    preprocess = make_preprocess()
    splitter = GroupShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=SPLIT_SEED)

    # ----- evaluation for best KNN model -----
    rows = []
    fprs_list, tprs_list, aucs_list = [], [], []
    precs_list, recs_list, aps_list = [], [], []
    cm_norm_list = []
    cm_count_list = []

    for split_id, tr, te in iter_valid_splits(splitter, X_use, y, groups_use):
        Xtr, Xte = X_use.iloc[tr].copy(), X_use.iloc[te].copy()
        ytr, yte = y[tr], y[te]

        try:
            pipe = build_knn_pipe(preprocess)
            pipe.fit(Xtr, ytr)
            pte = score_to_prob(pipe, Xte)
            pred_te = (pte >= THR_PRED).astype(int)

            auc = float(roc_auc_score(yte, pte))
            kappa = float(cohen_kappa_score(yte, pred_te))
            mcc = float(matthews_corrcoef(yte, pred_te))
            acc = float(accuracy_score(yte, pred_te))
            ap = float(average_precision_score(yte, pte))

            rows.append({
                "model": BEST_MODEL_NAME,
                "split": split_id,
                "n_train": int(len(tr)),
                "n_test": int(len(te)),
                "pos_train": int(np.sum(ytr == 1)),
                "neg_train": int(np.sum(ytr == 0)),
                "pos_test": int(np.sum(yte == 1)),
                "neg_test": int(np.sum(yte == 0)),
                "auc": auc,
                "ap": ap,
                "kappa": kappa,
                "mcc": mcc,
                "acc": acc,
                "error": ""
            })

            fpr, tpr, _ = roc_curve(yte, pte)
            fprs_list.append(fpr)
            tprs_list.append(tpr)
            aucs_list.append(auc)

            prec, rec, _ = precision_recall_curve(yte, pte)
            precs_list.append(prec)
            recs_list.append(rec)
            aps_list.append(ap)

            cm_norm_list.append(confusion_norm_true(yte, pred_te))
            cm_count_list.append(confusion_counts(yte, pred_te))

        except Exception as e:
            rows.append({
                "model": BEST_MODEL_NAME,
                "split": split_id,
                "n_train": int(len(tr)),
                "n_test": int(len(te)),
                "pos_train": int(np.sum(ytr == 1)),
                "neg_train": int(np.sum(ytr == 0)),
                "pos_test": int(np.sum(yte == 1)),
                "neg_test": int(np.sum(yte == 0)),
                "auc": np.nan,
                "ap": np.nan,
                "kappa": np.nan,
                "mcc": np.nan,
                "acc": np.nan,
                "error": repr(e)
            })

    res = pd.DataFrame(rows)
    res.to_csv(OUT_DIR / "kcat_best_knn_all_valid_splits.csv", index=False, encoding="utf-8-sig")

    res_ok = res.dropna(subset=["auc", "ap", "kappa", "mcc", "acc"]).copy()
    if res_ok.empty:
        raise RuntimeError("All valid split evaluations failed. Check kcat_best_knn_all_valid_splits.csv")

    summary = pd.DataFrame([{
        "model": BEST_MODEL_NAME,
        "scheme": BEST_SCHEME,
        "q": BEST_Q,
        "thr": float(meta.get("thr", np.nan)),
        "n_samples": int(len(y)),
        "n_groups": int(len(np.unique(groups_use))),
        "auc_mean": float(res_ok["auc"].mean()),
        "auc_median": float(res_ok["auc"].median()),
        "auc_std": float(res_ok["auc"].std()),
        "auc_max": float(res_ok["auc"].max()),
        "ap_mean": float(res_ok["ap"].mean()),
        "kappa_mean": float(res_ok["kappa"].mean()),
        "mcc_mean": float(res_ok["mcc"].mean()),
        "acc_mean": float(res_ok["acc"].mean()),
        "n_splits": int(len(res_ok)),
    }])
    summary.to_csv(OUT_DIR / "kcat_best_knn_summary.csv", index=False, encoding="utf-8-sig")
    print(summary.iloc[0].to_dict())

    # pretty plots
    plot_mean_roc_band(
        fprs_list, tprs_list, aucs_list,
        title=f"CAT KcatHigh ROC (mean ± std) | {BEST_MODEL_NAME} | single_q={BEST_Q}",
        out_path=PLOT_DIR / "ROC_pretty.png"
    )
    plot_mean_pr_band(
        precs_list, recs_list, aps_list,
        title=f"CAT KcatHigh PR (mean ± std) | {BEST_MODEL_NAME} | single_q={BEST_Q}",
        out_path=PLOT_DIR / "PR_pretty.png"
    )
    cm_mean_norm = np.mean(np.stack(cm_norm_list, axis=0), axis=0)
    plot_cm_single_hue(
        cm_mean_norm,
        title=f"CAT KcatHigh Mean Normalized CM over splits (thr={THR_PRED}) | {BEST_MODEL_NAME}",
        out_path=PLOT_DIR / "CM_mean_norm_pretty.png",
        normalized=True
    )

    cm_sum_count = np.sum(np.stack(cm_count_list, axis=0), axis=0).astype(int)
    pd.DataFrame(
        cm_sum_count,
        index=["True 0", "True 1"],
        columns=["Pred 0", "Pred 1"]
    ).to_csv(PLOT_DIR / "CM_sum_counts.csv", encoding="utf-8-sig")

    plot_cm_single_hue(
        cm_sum_count,
        title=f"CAT KcatHigh Summed CM over valid splits (thr={THR_PRED}) | {BEST_MODEL_NAME}",
        out_path=PLOT_DIR / "CM_sum_counts_pretty.png",
        normalized=False
    )

    # ----- full refit + save labeled subset -----
    df_use = df_use.copy()
    df_use["y"] = y
    df_use.to_csv(OUT_DIR / "kcat_best_setting_labeled_data.csv", index=False, encoding="utf-8-sig")

    full_pipe = build_knn_pipe(preprocess)
    full_pipe.fit(X_use, y)
    joblib.dump(
        {
            "task": TASK_NAME,
            "labeling_meta": meta,
            "best_model": BEST_MODEL_NAME,
            "pipeline": full_pipe,
        },
        OUT_DIR / "kcat_best_knn_refit_full.joblib"
    )

    # ----- SHAP on full refit -----
    if not HAS_SHAP:
        print("[Warn] shap not installed. SHAP outputs were skipped. Install with: pip install shap")
        print("Saved to:", OUT_DIR)
        return

    pre = full_pipe.named_steps["preprocess"]
    scaler = full_pipe.named_steps["scaler"]
    model = full_pipe.named_steps["model"]

    X_t = pre.transform(X_use)
    X_s = scaler.transform(X_t)
    feature_names = [str(x) for x in pre.get_feature_names_out()]

    bg_size = min(KNN_BACKGROUND_SIZE, len(X_s))
    background = shap.sample(X_s, bg_size, random_state=RANDOM_SEED)

    explainer = shap.KernelExplainer(model.predict_proba, background)
    shap_values = explainer.shap_values(X_s, nsamples=KNN_NSAMPLES)
    shap_values_2d = normalize_shap_output(shap_values)

    save_shap_outputs(shap_values_2d, X_s, feature_names, prefix="kcat_knn")

    print("Saved to:", OUT_DIR)


if __name__ == "__main__":
    main()


[KcatHigh] Best setting reproduction
{'scheme': 'single_q', 'q': 0.45, 'thr': 3.3198654689156797}
Expected thr from search = 3.319865468916
n_samples=13 | pos(1)=6 | neg(0)=7 | groups=8
{'model': 'KNN', 'scheme': 'single_q', 'q': 0.45, 'thr': 3.3198654689156797, 'n_samples': 13, 'n_groups': 8, 'auc_mean': 0.9166666666666666, 'auc_median': 1.0, 'auc_std': 0.23878346647045964, 'auc_max': 1.0, 'ap_mean': 0.9608333333333332, 'kappa_mean': 0.22000000000000003, 'mcc_mean': 0.2333333333333333, 'acc_mean': 0.6191666666666668, 'n_splits': 20}


  0%|          | 0/13 [00:00<?, ?it/s]

Saved to: C:\Users\86158\Desktop\cat_kcat_best_knn_eval_shap


In [2]:
# -*- coding: utf-8 -*-
"""
CAT KcatHigh：标签方案 A+B 同时搜索（只改标签，不改特征）
A: 单阈值 (single_q)  q ∈ [0.30, 0.70] step=0.05
B: 两端切分 (extreme_q) 丢中间  q ∈ {0.30, 0.35, 0.40}

统一改为：
- 严格 GroupKFold（按 DOI 分组）
- OOF 概率 / OOF 预测
- 以 OOF AUC 为主指标排序
- 仍保留 12 模型（若安装 xgboost 则加入）

额外约束：
- 标签样本数至少 >= MIN_LABEL_SAMPLES
- 只有 n_valid_folds >= MIN_VALID_FOLDS 的模型才参与该设置下的排名

输出：
- <task>_A_singleq_grid.csv
- <task>_B_extremeq_grid.csv
- <task>_overall_best.csv
- <task>_best_setting_model_summary.csv
- <task>_best_setting_labeled_data.csv
- <task>_best_setting_oof_predictions_groupkfold.csv
- <task>_best_setting_refit_full.joblib
"""

import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.model_selection import GroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.metrics import (
    roc_auc_score,
    cohen_kappa_score,
    matthews_corrcoef,
    accuracy_score
)

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False


# =========================
# 0) 路径与配置
# =========================
CAT_PATH = r"C:\Users\86158\Desktop\dataiscat.xlsx"
SHEET_NAME = 0

OUT_DIR = Path(r"C:\Users\86158\Desktop\cat_kcat_cls_threshold_tuning_AB_groupkfold_oof")
OUT_DIR.mkdir(parents=True, exist_ok=True)

COL_DOI = "data reference doi"
COL_KCAT = "Kcat/s-1"

# GroupKFold 配置
MAX_FOLDS = 5

# 小样本门槛
MIN_LABEL_SAMPLES = 10
MIN_TEST_SAMPLES = 4
MIN_TRAIN_CLASS_COUNT = 2

# 只有 n_valid_folds >= 2 的模型才参与排名
MIN_VALID_FOLDS = 2

# 搜索参数
QS_SINGLE = np.round(np.arange(0.30, 0.70 + 1e-9, 0.05), 2).tolist()
QS_EXTREME = [0.30, 0.35, 0.40]


# =========================
# 1) 列名/特征定义
# =========================
def normalize_name_list(cols):
    return (
        pd.Index(cols)
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .tolist()
    )


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )
    return df


FEATURE_COLS = normalize_name_list([
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "shape", "size (nm)", "surface treatment", "dispersion medium",
    "pH", "temperature/oC", "Substrate "
])

NUMERIC_COLS = normalize_name_list([
    "contain O", "contain N", "contain P", "contain S", "contain Si",
    "contain Se", "contain B", "contain F", "contain Cl", "contain Br", "contain I",
    "main  metal proportion", "main metal number", "main metal valence",
    "minor metal percentage", "minor metal number", "minor metal valence",
    "3rd metal ratio", "3rd metal type", "3rd metal valence",
    "size (nm)", "pH", "temperature/oC"
])

CATEGORICAL_COLS = normalize_name_list([
    "shape", "surface treatment", "dispersion medium", "Substrate "
])


# =========================
# 2) 预处理（基本保持不变）
# =========================
def clean_text_series(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    s = s.str.replace(r"\s+", " ", regex=True)
    s = s.str.lower()
    s = s.replace({"": np.nan, "nan": np.nan, "none": np.nan})
    return s


def prepare_features(df_raw: pd.DataFrame, feature_cols, numeric_cols, categorical_cols) -> pd.DataFrame:
    X = df_raw[feature_cols].copy()

    for c in numeric_cols:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce")

    for c in ["3rd metal ratio", "3rd metal type", "3rd metal valence"]:
        if c in X.columns:
            X[c] = X[c].fillna(0.0)

    for c in categorical_cols:
        if c in X.columns:
            X[c] = clean_text_series(X[c])

    return X


def make_preprocess(numeric_cols, categorical_cols):
    try:
        oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        oh = OneHotEncoder(handle_unknown="ignore", sparse=False)

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True))
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="not_reported")),
        ("onehot", oh),
    ])

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, numeric_cols),
            ("cat", cat_pipe, categorical_cols),
        ],
        remainder="drop"
    )


# =========================
# 3) 模型
# =========================
def get_models():
    models = {
        "LogReg": LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            solver="liblinear",
            random_state=42
        ),
        "LinearSVC": LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=42
        ),
        "SVC_RBF": SVC(
            C=1.0,
            gamma="scale",
            class_weight="balanced",
            probability=True,
            random_state=42
        ),
        "KNN": KNeighborsClassifier(
            n_neighbors=5,
            weights="distance"
        ),
        "GaussianNB": GaussianNB(),
        "DecisionTree": DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42
        ),
        "RF": RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "ExtraTrees": ExtraTreesClassifier(
            n_estimators=800,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),
        "GradientBoosting": GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ),
        "HistGB": HistGradientBoostingClassifier(
            max_iter=300,
            learning_rate=0.05,
            max_depth=4,
            random_state=42
        ),
        "AdaBoost": AdaBoostClassifier(
            n_estimators=300,
            learning_rate=0.05,
            random_state=42
        ),
    }

    if HAS_XGBOOST:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=42,
            n_jobs=-1,
            verbosity=0
        )
    return models


def needs_scaling(model_name):
    return model_name in {"LogReg", "LinearSVC", "SVC_RBF", "KNN", "GaussianNB"}


def build_model_pipe(model_name, model, preprocess):
    steps = [("preprocess", preprocess)]
    if needs_scaling(model_name):
        steps.append(("scaler", StandardScaler(with_mean=True)))
    steps.append(("model", model))
    return Pipeline(steps)


def score_to_prob(pipe, X):
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X)[:, 1]
    if hasattr(pipe, "decision_function"):
        z = np.asarray(pipe.decision_function(X), dtype=float)
        return 1.0 / (1.0 + np.exp(-z))
    return np.asarray(pipe.predict(X), dtype=float)


# =========================
# 4) 标签（A/B）
# =========================
def build_labels_log10(values: pd.Series, task: str, scheme: str, q: float):
    v = pd.to_numeric(values, errors="coerce")
    m = v.notna() & (v > 0)
    vals = np.log10(v.loc[m].astype(float).values)

    if len(vals) == 0:
        return np.array([], dtype=int), np.array([], dtype=int), {"scheme": scheme, "q": q}

    if scheme == "single_q":
        if task == "KmLow":
            thr = float(np.quantile(vals, q))
            y = (vals <= thr).astype(int)
            keep = np.ones_like(y, dtype=bool)
            meta = {"scheme": scheme, "q": q, "thr": thr}
        else:  # KcatHigh
            thr = float(np.quantile(vals, 1 - q))
            y = (vals >= thr).astype(int)
            keep = np.ones_like(y, dtype=bool)
            meta = {"scheme": scheme, "q": q, "thr": thr}

    elif scheme == "extreme_q":
        lo = float(np.quantile(vals, q))
        hi = float(np.quantile(vals, 1 - q))
        keep = (vals <= lo) | (vals >= hi)
        vals_k = vals[keep]

        if task == "KmLow":
            y = (vals_k <= lo).astype(int)
        else:  # KcatHigh
            y = (vals_k >= hi).astype(int)

        meta = {
            "scheme": scheme,
            "q": q,
            "lo": lo,
            "hi": hi,
            "dropped_mid": float(1 - 2 * q)
        }
    else:
        raise ValueError(scheme)

    idx_all = np.where(m.values)[0]
    idx_keep = idx_all[keep]
    return idx_keep, y.astype(int), meta


# =========================
# 5) 分组 / OOF
# =========================
def make_groups(df_unit, col_doi):
    groups = df_unit[col_doi].fillna("").astype(str).str.strip()
    miss = groups.eq("")
    groups.loc[miss] = [f"__NO_DOI_ROW_{i}__" for i in df_unit.index[miss]]
    return groups.values


def safe_auc(y_true, prob):
    y_true = np.asarray(y_true)
    prob = np.asarray(prob, dtype=float)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, prob))


def groupkfold_oof_classification(X, y, groups, model_name, model, preprocess):
    uniq_groups = np.unique(groups)
    n_folds = min(MAX_FOLDS, len(uniq_groups))
    if n_folds < 3:
        return None

    gkf = GroupKFold(n_splits=n_folds)

    oof_prob = np.full(len(y), np.nan, dtype=float)
    oof_pred = np.full(len(y), -1, dtype=int)
    fold_rows = []

    n_valid_folds = 0
    n_auc_folds = 0

    for fold_id, (tr, te) in enumerate(gkf.split(X, y, groups=groups), start=1):
        if len(te) < MIN_TEST_SAMPLES:
            continue

        ytr, yte = y[tr], y[te]

        if (np.sum(ytr == 0) < MIN_TRAIN_CLASS_COUNT) or (np.sum(ytr == 1) < MIN_TRAIN_CLASS_COUNT):
            continue

        try:
            pipe = build_model_pipe(model_name, model, preprocess)
            pipe.fit(X.iloc[tr], ytr)

            pte = score_to_prob(pipe, X.iloc[te])
            pred = (pte >= 0.5).astype(int)

            oof_prob[te] = pte
            oof_pred[te] = pred
            n_valid_folds += 1

            auc_fold = safe_auc(yte, pte)
            if pd.notna(auc_fold):
                n_auc_folds += 1

            fold_rows.append({
                "model": model_name,
                "fold": fold_id,
                "n_train": int(len(tr)),
                "n_test": int(len(te)),
                "test_pos": int(np.sum(yte == 1)),
                "test_neg": int(np.sum(yte == 0)),
                "auc_fold": auc_fold,
                "kappa_fold": float(cohen_kappa_score(yte, pred)),
                "mcc_fold": float(matthews_corrcoef(yte, pred)),
                "acc_fold": float(accuracy_score(yte, pred)),
            })

        except Exception:
            fold_rows.append({
                "model": model_name,
                "fold": fold_id,
                "n_train": int(len(tr)),
                "n_test": int(len(te)),
                "test_pos": int(np.sum(yte == 1)),
                "test_neg": int(np.sum(yte == 0)),
                "auc_fold": np.nan,
                "kappa_fold": np.nan,
                "mcc_fold": np.nan,
                "acc_fold": np.nan,
            })

    ok = np.isfinite(oof_prob)
    if ok.sum() == 0:
        return {
            "model": model_name,
            "fold_rows": fold_rows,
            "oof_prob": oof_prob,
            "oof_pred": oof_pred,
            "n_total_folds": int(n_folds),
            "n_valid_folds": int(n_valid_folds),
            "n_auc_folds": int(n_auc_folds),
            "oof_coverage": 0.0,
            "auc_oof": np.nan,
            "kappa_oof": np.nan,
            "mcc_oof": np.nan,
            "acc_oof": np.nan,
            "auc_fold_mean": np.nan,
            "auc_fold_median": np.nan,
            "auc_fold_std": np.nan,
            "auc_fold_max": np.nan,
        }

    y_ok = y[ok]
    prob_ok = oof_prob[ok]
    pred_ok = oof_pred[ok]

    auc_oof = safe_auc(y_ok, prob_ok)
    kappa_oof = float(cohen_kappa_score(y_ok, pred_ok))
    mcc_oof = float(matthews_corrcoef(y_ok, pred_ok))
    acc_oof = float(accuracy_score(y_ok, pred_ok))

    auc_folds = [r["auc_fold"] for r in fold_rows if pd.notna(r["auc_fold"])]

    return {
        "model": model_name,
        "fold_rows": fold_rows,
        "oof_prob": oof_prob,
        "oof_pred": oof_pred,
        "n_total_folds": int(n_folds),
        "n_valid_folds": int(n_valid_folds),
        "n_auc_folds": int(n_auc_folds),
        "oof_coverage": float(ok.mean()),
        "auc_oof": float(auc_oof) if pd.notna(auc_oof) else np.nan,
        "kappa_oof": float(kappa_oof),
        "mcc_oof": float(mcc_oof),
        "acc_oof": float(acc_oof),
        "auc_fold_mean": float(np.mean(auc_folds)) if len(auc_folds) > 0 else np.nan,
        "auc_fold_median": float(np.median(auc_folds)) if len(auc_folds) > 0 else np.nan,
        "auc_fold_std": float(np.std(auc_folds, ddof=1)) if len(auc_folds) > 1 else np.nan,
        "auc_fold_max": float(np.max(auc_folds)) if len(auc_folds) > 0 else np.nan,
    }


def evaluate_one_setting(task_name, X_all, groups_all, idx_keep, y, preprocess):
    X = X_all.iloc[idx_keep].copy()
    groups = groups_all[idx_keep].copy()
    y = np.asarray(y, int)

    models = get_models()
    fold_rows_all = []
    summ_rows = []

    for model_name, model in models.items():
        out = groupkfold_oof_classification(X, y, groups, model_name, model, preprocess)
        if out is None:
            continue

        fold_rows_all.extend(out["fold_rows"])

        if out["n_valid_folds"] < MIN_VALID_FOLDS:
            continue
        if pd.isna(out["auc_oof"]):
            continue

        summ_rows.append({
            "model": model_name,
            "auc_oof": out["auc_oof"],
            "kappa_oof": out["kappa_oof"],
            "mcc_oof": out["mcc_oof"],
            "acc_oof": out["acc_oof"],
            "auc_fold_mean": out["auc_fold_mean"],
            "auc_fold_median": out["auc_fold_median"],
            "auc_fold_std": out["auc_fold_std"],
            "auc_fold_max": out["auc_fold_max"],
            "n_total_folds": out["n_total_folds"],
            "n_valid_folds": out["n_valid_folds"],
            "n_auc_folds": out["n_auc_folds"],
            "oof_coverage": out["oof_coverage"],
        })

    fold_df = pd.DataFrame(fold_rows_all)

    if len(summ_rows) == 0:
        return fold_df, pd.DataFrame()

    summ = pd.DataFrame(summ_rows).sort_values(
        by=["auc_oof", "mcc_oof", "kappa_oof", "acc_oof", "auc_fold_mean", "n_valid_folds", "oof_coverage"],
        ascending=[False, False, False, False, False, False, False]
    ).reset_index(drop=True)

    return fold_df, summ


def get_best_model_oof(task_name, X_all, groups_all, idx_keep, y, preprocess, model_name):
    X = X_all.iloc[idx_keep].copy()
    groups = groups_all[idx_keep].copy()
    y = np.asarray(y, int)
    models = get_models()
    return groupkfold_oof_classification(X, y, groups, model_name, models[model_name], preprocess)


# =========================
# 6) 搜索 A/B 并输出 overall best
# =========================
def empty_result_df():
    return pd.DataFrame(columns=[
        "scheme", "q", "thr", "lo", "hi", "dropped_mid",
        "n_samples", "n_groups", "best_model",
        "best_auc_oof", "best_mcc_oof", "best_kappa_oof", "best_acc_oof",
        "best_auc_fold_mean", "best_n_valid_folds"
    ])


def run_task(task_name, df, X_all, groups_all, preprocess, col_target):
    results_A = []
    results_B = []

    # A: single_q
    for q in QS_SINGLE:
        idx_keep, y, meta = build_labels_log10(df[col_target], task_name, "single_q", q)

        if len(y) < MIN_LABEL_SAMPLES or len(np.unique(y)) < 2:
            print(f"[{task_name}][single_q={q}] skipped | n={len(y)} | unique_y={len(np.unique(y))}")
            continue

        _, summ = evaluate_one_setting(task_name, X_all, groups_all, idx_keep, y, preprocess)

        if summ.empty:
            print(f"[{task_name}][single_q={q}] no model with n_valid_folds >= {MIN_VALID_FOLDS}")
            continue

        best = summ.iloc[0].to_dict()
        results_A.append({
            **meta,
            "n_samples": int(len(y)),
            "n_groups": int(len(np.unique(groups_all[idx_keep]))),
            "best_model": best["model"],
            "best_auc_oof": float(best["auc_oof"]),
            "best_mcc_oof": float(best["mcc_oof"]),
            "best_kappa_oof": float(best["kappa_oof"]),
            "best_acc_oof": float(best["acc_oof"]),
            "best_auc_fold_mean": float(best["auc_fold_mean"]) if pd.notna(best["auc_fold_mean"]) else np.nan,
            "best_n_valid_folds": int(best["n_valid_folds"]),
        })

    # B: extreme_q
    for q in QS_EXTREME:
        idx_keep, y, meta = build_labels_log10(df[col_target], task_name, "extreme_q", q)

        if len(y) < MIN_LABEL_SAMPLES or len(np.unique(y)) < 2:
            print(f"[{task_name}][extreme_q={q}] skipped | n={len(y)} | unique_y={len(np.unique(y))}")
            continue

        _, summ = evaluate_one_setting(task_name, X_all, groups_all, idx_keep, y, preprocess)

        if summ.empty:
            print(f"[{task_name}][extreme_q={q}] no model with n_valid_folds >= {MIN_VALID_FOLDS}")
            continue

        best = summ.iloc[0].to_dict()
        results_B.append({
            **meta,
            "n_samples": int(len(y)),
            "n_groups": int(len(np.unique(groups_all[idx_keep]))),
            "best_model": best["model"],
            "best_auc_oof": float(best["auc_oof"]),
            "best_mcc_oof": float(best["mcc_oof"]),
            "best_kappa_oof": float(best["kappa_oof"]),
            "best_acc_oof": float(best["acc_oof"]),
            "best_auc_fold_mean": float(best["auc_fold_mean"]) if pd.notna(best["auc_fold_mean"]) else np.nan,
            "best_n_valid_folds": int(best["n_valid_folds"]),
        })

    if results_A:
        dfA = pd.DataFrame(results_A).sort_values(
            by=["best_auc_oof", "best_mcc_oof", "best_kappa_oof", "best_acc_oof", "best_n_valid_folds"],
            ascending=[False, False, False, False, False]
        ).reset_index(drop=True)
    else:
        dfA = empty_result_df()

    if results_B:
        dfB = pd.DataFrame(results_B).sort_values(
            by=["best_auc_oof", "best_mcc_oof", "best_kappa_oof", "best_acc_oof", "best_n_valid_folds"],
            ascending=[False, False, False, False, False]
        ).reset_index(drop=True)
    else:
        dfB = empty_result_df()

    dfA.to_csv(OUT_DIR / f"{task_name}_A_singleq_grid.csv", index=False, encoding="utf-8-sig")
    dfB.to_csv(OUT_DIR / f"{task_name}_B_extremeq_grid.csv", index=False, encoding="utf-8-sig")

    cand = []
    if not dfA.empty:
        cand.append(dfA.iloc[0].to_dict())
    if not dfB.empty:
        cand.append(dfB.iloc[0].to_dict())

    if not cand:
        print(f"[{task_name}] 没有可用方案")
        return

    overall = pd.DataFrame(cand).sort_values(
        by=["best_auc_oof", "best_mcc_oof", "best_kappa_oof", "best_acc_oof", "best_n_valid_folds"],
        ascending=[False, False, False, False, False]
    ).reset_index(drop=True)

    overall.to_csv(OUT_DIR / f"{task_name}_overall_best.csv", index=False, encoding="utf-8-sig")

    best_meta = overall.iloc[0].to_dict()
    print("\n" + "=" * 70)
    print(f"[{task_name}] OVERALL BEST (from A/B)")
    print("=" * 70)
    print(best_meta)

    # 用 overall best 重新生成标签
    scheme = best_meta["scheme"]
    q = float(best_meta["q"])
    idx_keep, y, meta = build_labels_log10(df[col_target], task_name, scheme, q)

    # 保存 best setting 的模型汇总
    fold_df, best_summ = evaluate_one_setting(task_name, X_all, groups_all, idx_keep, y, preprocess)
    if not best_summ.empty:
        best_summ.to_csv(
            OUT_DIR / f"{task_name}_best_setting_model_summary.csv",
            index=False,
            encoding="utf-8-sig"
        )

    # 保存 best model 的 OOF 预测
    best_model_name = best_meta["best_model"]
    oof_out = get_best_model_oof(task_name, X_all, groups_all, idx_keep, y, preprocess, best_model_name)

    if oof_out is not None:
        pd.DataFrame({
            "row_index": df.index[idx_keep],
            "doi_group": groups_all[idx_keep],
            "y_true": y,
            "y_oof_prob": oof_out["oof_prob"],
            "y_oof_pred": oof_out["oof_pred"],
        }).to_csv(
            OUT_DIR / f"{task_name}_best_setting_oof_predictions_groupkfold.csv",
            index=False,
            encoding="utf-8-sig"
        )

        if not fold_df.empty:
            fold_df.to_csv(
                OUT_DIR / f"{task_name}_best_setting_fold_metrics.csv",
                index=False,
                encoding="utf-8-sig"
            )

    # 全量重训并保存
    models = get_models()
    pipe = build_model_pipe(best_model_name, models[best_model_name], preprocess)

    X_use = X_all.iloc[idx_keep].copy()
    pipe.fit(X_use, y)

    df_use = df.iloc[idx_keep].copy()
    df_use["y"] = y
    df_use.to_csv(
        OUT_DIR / f"{task_name}_best_setting_labeled_data.csv",
        index=False,
        encoding="utf-8-sig"
    )

    joblib.dump(
        {
            "task": task_name,
            "labeling_meta": meta,
            "best_model": best_model_name,
            "pipeline": pipe,
            "evaluation_type": "GroupKFold OOF by DOI"
        },
        OUT_DIR / f"{task_name}_best_setting_refit_full.joblib"
    )


def main():
    print("=" * 70)
    print("Input file :", CAT_PATH)
    print("Output dir :", OUT_DIR)
    print("MAX_FOLDS  :", MAX_FOLDS)
    print("MIN_VALID_FOLDS :", MIN_VALID_FOLDS)
    print("HAS_XGBOOST :", HAS_XGBOOST)
    print("=" * 70)

    df = pd.read_excel(CAT_PATH, sheet_name=SHEET_NAME)
    df = normalize_columns(df)

    if COL_DOI not in df.columns:
        raise KeyError(f"DOI column not found: {COL_DOI}")
    if COL_KCAT not in df.columns:
        raise KeyError(f"Kcat column not found: {COL_KCAT}")

    feature_cols = [c for c in FEATURE_COLS if c in df.columns]
    numeric_cols = [c for c in NUMERIC_COLS if c in df.columns]
    categorical_cols = [c for c in CATEGORICAL_COLS if c in df.columns]

    if len(feature_cols) == 0:
        raise ValueError("No usable feature columns found after column normalization.")

    groups_all = make_groups(df, COL_DOI)
    X_all = prepare_features(df, feature_cols, numeric_cols, categorical_cols)
    preprocess = make_preprocess(numeric_cols, categorical_cols)

    run_task("KcatHigh", df, X_all, groups_all, preprocess, COL_KCAT)

    print("\nSaved to:", OUT_DIR)


if __name__ == "__main__":
    main()

Input file : C:\Users\86158\Desktop\dataiscat.xlsx
Output dir : C:\Users\86158\Desktop\cat_kcat_cls_threshold_tuning_AB_groupkfold_oof
MAX_FOLDS  : 5
MIN_VALID_FOLDS : 2
HAS_XGBOOST : True
[KcatHigh][single_q=0.3] no model with n_valid_folds >= 2
[KcatHigh][single_q=0.35] no model with n_valid_folds >= 2
[KcatHigh][single_q=0.4] no model with n_valid_folds >= 2
[KcatHigh][single_q=0.45] no model with n_valid_folds >= 2
[KcatHigh][single_q=0.5] no model with n_valid_folds >= 2
[KcatHigh][single_q=0.55] no model with n_valid_folds >= 2
[KcatHigh][single_q=0.6] no model with n_valid_folds >= 2
[KcatHigh][single_q=0.65] no model with n_valid_folds >= 2
[KcatHigh][single_q=0.7] no model with n_valid_folds >= 2
[KcatHigh][extreme_q=0.3] skipped | n=8 | unique_y=2
[KcatHigh][extreme_q=0.35] no model with n_valid_folds >= 2
[KcatHigh][extreme_q=0.4] no model with n_valid_folds >= 2
[KcatHigh] 没有可用方案

Saved to: C:\Users\86158\Desktop\cat_kcat_cls_threshold_tuning_AB_groupkfold_oof
